In [7]:
# ---------------------------------------------------------
# 07 - MODEL BUILDING
#
# Travel Destination Recommendation System
#
# This notebook consumes the finalized artifacts produced
# by 06_Data_Processing.ipynb.
#
# IMPORTANT:
# We do NOT perform data cleaning, encoding, scaling,
# or PCA again in this notebook.
# ---------------------------------------------------------

import os
import joblib
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity


# ---------------------------------------------------------
# Confirm that the notebook is running from the expected
# notebooks/ directory.
# ---------------------------------------------------------

print("Current working directory:")
print(os.getcwd())

print("\nModel-building environment initialized successfully.")

Current working directory:
c:\Users\Rushi\Desktop\Travel_Agent\notebooks

Model-building environment initialized successfully.


In [8]:
# ---------------------------------------------------------
# CELL 2 - LOAD FINALIZED MODEL ARTIFACTS
#
# Everything required for model building was already created
# and validated in 06_Data_Processing.ipynb.
#
# We simply load those saved files here.
# ---------------------------------------------------------

import os
import joblib
import pandas as pd


# ---------------------------------------------------------
# 1. Load PCA destination features
#
# Contains:
# destination + PC1 ... PC15
#
# This is the main dataset that our recommendation model
# will use.
# ---------------------------------------------------------

pca_destination_df = pd.read_csv(
    "../data/cleaned/pca_destination_features.csv"
)


# ---------------------------------------------------------
# 2. Load the saved scaler
#
# Used later when a NEW user query needs to be transformed
# using the exact same preprocessing as the training data.
# ---------------------------------------------------------

scaler = joblib.load(
    "../models/feature_scaler.pkl"
)


# ---------------------------------------------------------
# 3. Load the saved PCA model
#
# This allows us to transform new user inputs into the same
# 15-dimensional PCA space.
# ---------------------------------------------------------

pca = joblib.load(
    "../models/pca_model.pkl"
)


# ---------------------------------------------------------
# 4. Load feature metadata
# ---------------------------------------------------------

feature_columns = joblib.load(
    "../models/feature_columns.pkl"
)

numerical_feature_columns = joblib.load(
    "../models/numerical_feature_columns.pkl"
)

binary_feature_columns = joblib.load(
    "../models/binary_feature_columns.pkl"
)

redundant_features = joblib.load(
    "../models/redundant_features.pkl"
)


# ---------------------------------------------------------
# 5. Display a concise loading summary
# ---------------------------------------------------------

print("=" * 60)
print("FINAL MODEL ARTIFACTS LOADED")
print("=" * 60)

print("\nPCA destination dataset:")
print("Rows:", len(pca_destination_df))
print("Columns:", len(pca_destination_df.columns))

print("\nOriginal feature count:", len(feature_columns))
print("Numerical features:", len(numerical_feature_columns))
print("Binary features:", len(binary_feature_columns))
print("Redundant features:", len(redundant_features))

print("\nPCA input features:", pca.n_features_in_)
print("PCA components:", pca.n_components_)

print("\nArtifacts loaded successfully.")

FINAL MODEL ARTIFACTS LOADED

PCA destination dataset:
Rows: 50
Columns: 16

Original feature count: 47
Numerical features: 34
Binary features: 13
Redundant features: 2

PCA input features: 45
PCA components: 15

Artifacts loaded successfully.


In [9]:
# ---------------------------------------------------------
# CELL 3 - VALIDATE RECOMMENDATION DATASET
#
# Before building the recommendation model, we make sure
# the PCA dataset is clean and has the exact structure
# expected by the model.
# ---------------------------------------------------------

print("=" * 60)
print("RECOMMENDATION DATASET VALIDATION")
print("=" * 60)


# ---------------------------------------------------------
# 1. Basic shape
# ---------------------------------------------------------

print("\nRows:", len(pca_destination_df))
print("Columns:", len(pca_destination_df.columns))


# ---------------------------------------------------------
# 2. Check destination uniqueness
# ---------------------------------------------------------

print("\nUnique destinations:",
      pca_destination_df["destination"].nunique())

print("Duplicate destinations:",
      pca_destination_df["destination"].duplicated().sum())


# ---------------------------------------------------------
# 3. Expected PCA columns
# ---------------------------------------------------------

expected_pca_columns = [
    f"PC{i}" for i in range(1, 16)
]

print("\nExpected PCA columns:")
print(expected_pca_columns)


# ---------------------------------------------------------
# 4. Check for missing PCA columns
# ---------------------------------------------------------

missing_pca_columns = [
    col for col in expected_pca_columns
    if col not in pca_destination_df.columns
]

print("\nMissing PCA columns:", missing_pca_columns)


# ---------------------------------------------------------
# 5. Check for unexpected PCA columns
# ---------------------------------------------------------

unexpected_pca_columns = [
    col for col in pca_destination_df.columns
    if col.startswith("PC") and col not in expected_pca_columns
]

print("Unexpected PCA columns:", unexpected_pca_columns)


# ---------------------------------------------------------
# 6. Check missing values
# ---------------------------------------------------------

missing_values = pca_destination_df[
    ["destination"] + expected_pca_columns
].isnull().sum()

print("\nMissing values:")
print(missing_values)


# ---------------------------------------------------------
# 7. Check duplicate columns
# ---------------------------------------------------------

duplicate_columns = (
    pca_destination_df.columns[
        pca_destination_df.columns.duplicated()
    ].tolist()
)

print("\nDuplicate columns:", duplicate_columns)


# ---------------------------------------------------------
# 8. Check PCA columns are numeric
# ---------------------------------------------------------

non_numeric_pca = [
    col for col in expected_pca_columns
    if not pd.api.types.is_numeric_dtype(
        pca_destination_df[col]
    )
]

print("Non-numeric PCA columns:", non_numeric_pca)


# ---------------------------------------------------------
# 9. Final validation result
# ---------------------------------------------------------

validation_passed = (
    len(pca_destination_df) == 50
    and pca_destination_df["destination"].nunique() == 50
    and pca_destination_df["destination"].duplicated().sum() == 0
    and len(missing_pca_columns) == 0
    and len(unexpected_pca_columns) == 0
    and pca_destination_df[
        ["destination"] + expected_pca_columns
    ].isnull().sum().sum() == 0
    and len(duplicate_columns) == 0
    and len(non_numeric_pca) == 0
)


print("\n" + "=" * 60)

if validation_passed:
    print("✓ RECOMMENDATION DATASET VALIDATION PASSED")
else:
    print("✗ RECOMMENDATION DATASET VALIDATION FAILED")

print("=" * 60)

RECOMMENDATION DATASET VALIDATION

Rows: 50
Columns: 16

Unique destinations: 50
Duplicate destinations: 0

Expected PCA columns:
['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15']

Missing PCA columns: []
Unexpected PCA columns: []

Missing values:
destination    0
PC1            0
PC2            0
PC3            0
PC4            0
PC5            0
PC6            0
PC7            0
PC8            0
PC9            0
PC10           0
PC11           0
PC12           0
PC13           0
PC14           0
PC15           0
dtype: int64

Duplicate columns: []
Non-numeric PCA columns: []

✓ RECOMMENDATION DATASET VALIDATION PASSED


In [11]:
# ---------------------------------------------------------
# CELL 4 - CREATE RECOMMENDATION FEATURE MATRIX
#
# We separate:
#   1. Destination names
#   2. PCA features
#
# The recommendation algorithm will work only with the
# numerical PCA features.
# ---------------------------------------------------------

# ---------------------------------------------------------
# 1. Store destination names
# ---------------------------------------------------------

destination_names = pca_destination_df["destination"].copy()


# ---------------------------------------------------------
# 2. Extract the 15 PCA components
#
# These are the actual numerical features that describe
# each destination in the reduced feature space.
# ---------------------------------------------------------

X_recommendation = pca_destination_df[
    expected_pca_columns
].copy()


# ---------------------------------------------------------
# 3. Display the resulting structure
# ---------------------------------------------------------

print("=" * 60)
print("RECOMMENDATION FEATURE MATRIX")
print("=" * 60)

print("\nShape:")
print(X_recommendation.shape)

print("\nNumber of destinations:")
print(len(destination_names))

print("\nNumber of recommendation features:")
print(X_recommendation.shape[1])

print("\nFeature columns:")
for i, col in enumerate(X_recommendation.columns, start=1):
    print(f"{i:2}. {col}")


# ---------------------------------------------------------
# 4. Verify that the destination order and feature matrix
#    have exactly the same number of rows.
# ---------------------------------------------------------

print("\nDestination rows:", len(destination_names))
print("Feature rows:", len(X_recommendation))


# ---------------------------------------------------------
# 5. Final checks
# ---------------------------------------------------------

print("\nMissing values:",
      X_recommendation.isnull().sum().sum())

print("Duplicate columns:",
      X_recommendation.columns.duplicated().sum())


# ---------------------------------------------------------
# 6. Display first 5 rows
# ---------------------------------------------------------

print("\nFirst 5 recommendation feature rows:")
display(X_recommendation.head())


# ---------------------------------------------------------
# 7. Final validation
# ---------------------------------------------------------

if (
    X_recommendation.shape == (50, 15)
    and len(destination_names) == 50
    and X_recommendation.isnull().sum().sum() == 0
    and X_recommendation.columns.duplicated().sum() == 0
):
    print("\n" + "=" * 60)
    print("✓ RECOMMENDATION FEATURE MATRIX CREATED SUCCESSFULLY")
    print("=" * 60)
else:
    print("\n✗ RECOMMENDATION FEATURE MATRIX VALIDATION FAILED")

RECOMMENDATION FEATURE MATRIX

Shape:
(50, 15)

Number of destinations:
50

Number of recommendation features:
15

Feature columns:
 1. PC1
 2. PC2
 3. PC3
 4. PC4
 5. PC5
 6. PC6
 7. PC7
 8. PC8
 9. PC9
10. PC10
11. PC11
12. PC12
13. PC13
14. PC14
15. PC15

Destination rows: 50
Feature rows: 50

Missing values: 0
Duplicate columns: 0

First 5 recommendation feature rows:


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15
0,-0.035769,-0.922579,0.051091,-0.643779,-0.734244,0.530902,1.151835,-1.662077,-0.127891,-1.627351,-0.589135,-0.192561,-0.186098,0.064635,-0.709079
1,0.395989,-1.103311,0.733874,0.233504,-0.321427,0.117735,-0.474759,-1.269103,-0.956262,-0.421330,-0.012701,0.246444,-0.413503,0.398685,-1.803953
2,0.359290,-1.191439,-1.151328,1.623510,0.750657,-1.133169,0.889291,2.513876,-0.034903,-0.136900,0.314258,0.235412,-0.979846,0.870892,-0.344422
3,0.127812,-1.082204,3.381689,-0.749300,-1.121050,-0.268471,0.220997,-0.312311,-0.572289,-0.925498,0.318953,1.007212,-0.333766,0.072594,0.400718
4,-2.883816,-0.471556,-1.032682,0.607836,-1.969158,-0.993851,-0.191215,1.046010,0.353361,1.094386,-0.126005,-0.493596,-1.616061,-2.075172,-0.316181



✓ RECOMMENDATION FEATURE MATRIX CREATED SUCCESSFULLY


In [12]:
# ---------------------------------------------------------
# CELL 5 - CALCULATE DESTINATION SIMILARITY
#
# We calculate cosine similarity between every pair of
# destinations using their 15 PCA components.
#
# The resulting matrix tells us how similar each destination
# is to every other destination.
# ---------------------------------------------------------

from sklearn.metrics.pairwise import cosine_similarity


# ---------------------------------------------------------
# 1. Calculate pairwise cosine similarity
#
# Input:
#     X_recommendation → 50 destinations × 15 PCA features
#
# Output:
#     similarity_matrix → 50 × 50
#
# Each row represents one destination.
# Each column represents another destination.
# ---------------------------------------------------------

similarity_matrix = cosine_similarity(
    X_recommendation
)


# ---------------------------------------------------------
# 2. Convert the similarity matrix into a DataFrame
#
# Using destination names as both rows and columns makes
# the matrix much easier to interpret and debug.
# ---------------------------------------------------------

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=destination_names,
    columns=destination_names
)


# ---------------------------------------------------------
# 3. Display basic information
# ---------------------------------------------------------

print("=" * 60)
print("DESTINATION SIMILARITY MATRIX")
print("=" * 60)

print("\nShape:")
print(similarity_df.shape)

print("\nNumber of destinations:")
print(similarity_df.shape[0])

print("\nNumber of similarity values:")
print(similarity_df.size)


# ---------------------------------------------------------
# 4. Check important properties
# ---------------------------------------------------------

print("\nMissing values:",
      similarity_df.isnull().sum().sum())

print("Duplicate rows:",
      similarity_df.index.duplicated().sum())

print("Duplicate columns:",
      similarity_df.columns.duplicated().sum())


# ---------------------------------------------------------
# 5. Check diagonal values
#
# A destination compared with itself should have similarity
# extremely close to 1.
# ---------------------------------------------------------

diagonal_values = similarity_df.values.diagonal()

print("\nMinimum self-similarity:",
      diagonal_values.min())

print("Maximum self-similarity:",
      diagonal_values.max())


# ---------------------------------------------------------
# 6. Display similarity for one destination
#
# This is only for inspection. The recommendation function
# will be created in the next step.
# ---------------------------------------------------------

example_destination = "Goa"

if example_destination in similarity_df.index:

    print(f"\nSimilarity with {example_destination}:")

    print(
        similarity_df[example_destination]
        .sort_values(ascending=False)
        .head(10)
    )


# ---------------------------------------------------------
# 7. Final validation
# ---------------------------------------------------------

validation_passed = (
    similarity_df.shape == (50, 50)
    and similarity_df.isnull().sum().sum() == 0
    and similarity_df.index.duplicated().sum() == 0
    and similarity_df.columns.duplicated().sum() == 0
    and diagonal_values.min() > 0.999
)


print("\n" + "=" * 60)

if validation_passed:
    print("✓ DESTINATION SIMILARITY MATRIX CREATED SUCCESSFULLY")
else:
    print("✗ SIMILARITY MATRIX VALIDATION FAILED")

print("=" * 60)

DESTINATION SIMILARITY MATRIX

Shape:
(50, 50)

Number of destinations:
50

Number of similarity values:
2500

Missing values: 0
Duplicate rows: 0
Duplicate columns: 0

Minimum self-similarity: 0.9999999999999996
Maximum self-similarity: 1.0000000000000004

Similarity with Goa:
destination
Goa           1.000000
Mumbai        0.586488
Alappuzha     0.355314
Kaziranga     0.351321
Coorg         0.225543
Bengaluru     0.209090
Delhi         0.201240
Gokarna       0.195864
Jaipur        0.172510
Kodaikanal    0.112580
Name: Goa, dtype: float64

✓ DESTINATION SIMILARITY MATRIX CREATED SUCCESSFULLY


In [13]:
# ---------------------------------------------------------
# CELL 6 - BUILD DESTINATION RECOMMENDATION FUNCTION
#
# This function uses the previously calculated cosine
# similarity matrix to recommend destinations that are most
# similar to a selected destination.
# ---------------------------------------------------------

def recommend_destinations(destination, n_recommendations=5):
    """
    Recommend destinations similar to the given destination.

    Parameters
    ----------
    destination : str
        Destination for which recommendations are required.

    n_recommendations : int
        Number of similar destinations to return.

    Returns
    -------
    pandas.DataFrame
        Recommended destinations with their similarity scores.
    """

    # -----------------------------------------------------
    # Check whether the requested destination exists
    # -----------------------------------------------------

    if destination not in similarity_df.index:
        raise ValueError(
            f"Destination '{destination}' not found in dataset."
        )

    # -----------------------------------------------------
    # Get similarity scores for the selected destination
    # -----------------------------------------------------

    similarity_scores = similarity_df[destination].copy()

    # -----------------------------------------------------
    # Remove the destination itself.
    #
    # A destination is always almost 1.0 similar to itself,
    # but we obviously don't want to recommend it.
    # -----------------------------------------------------

    similarity_scores = similarity_scores.drop(destination)

    # -----------------------------------------------------
    # Sort destinations from most similar to least similar
    # -----------------------------------------------------

    similarity_scores = similarity_scores.sort_values(
        ascending=False
    )

    # -----------------------------------------------------
    # Select the requested number of recommendations
    # -----------------------------------------------------

    recommendations = similarity_scores.head(
        n_recommendations
    )

    # -----------------------------------------------------
    # Convert the result into a clean DataFrame
    # -----------------------------------------------------

    recommendations_df = pd.DataFrame({
        "destination": recommendations.index,
        "similarity_score": recommendations.values
    })

    # Reset the index so the output is clean
    recommendations_df = recommendations_df.reset_index(
        drop=True
    )

    return recommendations_df


# ---------------------------------------------------------
# Test the recommendation function
# ---------------------------------------------------------

print("=" * 60)
print("RECOMMENDATION FUNCTION TEST")
print("=" * 60)

goa_recommendations = recommend_destinations(
    "Goa",
    n_recommendations=5
)

print("\nRecommendations for Goa:")
display(goa_recommendations)

RECOMMENDATION FUNCTION TEST

Recommendations for Goa:


,destination,similarity_score
0,Mumbai,0.586488
1,Alappuzha,0.355314
2,Kaziranga,0.351321
3,Coorg,0.225543
4,Bengaluru,0.209090


In [14]:
# ---------------------------------------------------------
# CELL 7 - TEST RECOMMENDATIONS FOR MULTIPLE DESTINATIONS
#
# We test the recommendation function using several
# destinations and inspect the results.
#
# This helps us verify that:
#   1. The function works for different inputs.
#   2. The selected destination is never recommended itself.
#   3. Similarity scores are properly ordered.
#   4. We always receive the requested number of results.
# ---------------------------------------------------------

test_destinations = [
    "Goa",
    "Mumbai",
    "Jaipur",
    "Manali",
    "Kerala"
]

# ---------------------------------------------------------
# Note:
# Our dataset contains "Alappuzha", "Munnar", etc., rather
# than a destination named "Kerala".
#
# Therefore, we will use destinations that we know exist
# in our actual dataset.
# ---------------------------------------------------------

test_destinations = [
    "Goa",
    "Mumbai",
    "Jaipur",
    "Manali",
    "Varanasi"
]


for destination in test_destinations:

    print("\n" + "=" * 60)
    print(f"Recommendations for: {destination}")
    print("=" * 60)

    recommendations = recommend_destinations(
        destination,
        n_recommendations=5
    )

    display(recommendations)


# ---------------------------------------------------------
# AUTOMATIC VALIDATION
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("MULTI-DESTINATION RECOMMENDATION VALIDATION")
print("=" * 60)

validation_passed = True

for destination in test_destinations:

    recommendations = recommend_destinations(
        destination,
        n_recommendations=5
    )

    # Check number of recommendations
    if len(recommendations) != 5:
        print(
            f"✗ {destination}: "
            f"Expected 5 recommendations, got {len(recommendations)}"
        )
        validation_passed = False

    # Check that original destination is not recommended
    if destination in recommendations["destination"].values:
        print(
            f"✗ {destination}: "
            "Original destination appeared in recommendations."
        )
        validation_passed = False

    # Check that similarity scores are descending
    scores = recommendations["similarity_score"].values

    if not all(scores[i] >= scores[i + 1]
               for i in range(len(scores) - 1)):

        print(
            f"✗ {destination}: "
            "Similarity scores are not properly sorted."
        )
        validation_passed = False


# ---------------------------------------------------------
# Final result
# ---------------------------------------------------------

if validation_passed:

    print("\n✓ MULTI-DESTINATION VALIDATION PASSED")

else:

    print("\n✗ MULTI-DESTINATION VALIDATION FAILED")


Recommendations for: Goa


,destination,similarity_score
0,Mumbai,0.586488
1,Alappuzha,0.355314
2,Kaziranga,0.351321
3,Coorg,0.225543
4,Bengaluru,0.209090



Recommendations for: Mumbai


,destination,similarity_score
0,Bengaluru,0.758071
1,Delhi,0.721535
2,Goa,0.586488
3,Jaipur,0.507607
4,Hyderabad,0.462617



Recommendations for: Jaipur


...


Recommendations for: Manali


,destination,similarity_score
0,Shimla,0.420071
1,Hampi,0.391980
2,Darjeeling,0.272704
3,Dharamshala,0.270283
4,Mahabalipuram,0.268478



Recommendations for: Varanasi


,destination,similarity_score
0,Kolkata,0.847289
1,Agra,0.631260
2,Amritsar,0.565585
3,Jim Corbett,0.367225
4,Dharamshala,0.358870



MULTI-DESTINATION RECOMMENDATION VALIDATION

✓ MULTI-DESTINATION VALIDATION PASSED


In [ ]:
# ---------------------------------------------------------
# EVALUATE SIMILARITY MODEL
#
# We perform basic mathematical and structural checks on
# the cosine-similarity model before saving it.
# ---------------------------------------------------------

print("=" * 60)
print("SIMILARITY MODEL EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# 1. Check similarity value range
#
# Cosine similarity should theoretically be between -1 and 1.
# ---------------------------------------------------------

min_similarity = similarity_df.values.min()
max_similarity = similarity_df.values.max()

print("\nMinimum similarity:",
      round(min_similarity, 6))

print("Maximum similarity:",
      round(max_similarity, 6))


# ---------------------------------------------------------
# 2. Check whether the matrix is symmetric
#
# Similarity(A, B) should equal Similarity(B, A).
# ---------------------------------------------------------

is_symmetric = np.allclose(
    similarity_df.values,
    similarity_df.values.T,
    atol=1e-10
)

print("\nSimilarity matrix symmetric:",
      is_symmetric)


# ---------------------------------------------------------
# 3. Remove self-similarity from analysis
#
# We don't want the 1.0 diagonal values to affect the
# average similarity calculation.
# ---------------------------------------------------------

similarity_values = similarity_df.values.copy()

np.fill_diagonal(
    similarity_values,
    np.nan
)


# ---------------------------------------------------------
# 4. Calculate average similarity between different
#    destinations.
# ---------------------------------------------------------

average_similarity = np.nanmean(
    similarity_values
)

print("\nAverage similarity between different destinations:",
      round(average_similarity, 6))


# ---------------------------------------------------------
# 5. Find the strongest destination pairs
#
# Since the matrix is symmetric, we only examine the upper
# triangle to avoid reporting the same pair twice.
# ---------------------------------------------------------

upper_triangle = np.triu(
    similarity_df.values,
    k=1
)

pair_indices = np.dstack(
    np.unravel_index(
        np.argsort(upper_triangle.ravel())[::-1],
        upper_triangle.shape
    )
)[0]


# ---------------------------------------------------------
# Display top 10 destination pairs
# ---------------------------------------------------------

print("\nTop 10 most similar destination pairs:")

pair_count = 0

for i, j in pair_indices:

    score = upper_triangle[i, j]

    # Ignore zero values that were not actual pairs
    if score <= 0:
        continue

    print(
        f"{pair_count + 1:2}. "
        f"{destination_names.iloc[i]} ↔ "
        f"{destination_names.iloc[j]} : "
        f"{score:.4f}"
    )

    pair_count += 1

    if pair_count == 10:
        break


# ---------------------------------------------------------
# 6. Final validation
# ---------------------------------------------------------

range_valid = (
    min_similarity >= -1.000001
    and max_similarity <= 1.000001
)

print("\n" + "=" * 60)

if range_valid and is_symmetric:

    print("✓ SIMILARITY MODEL EVALUATION PASSED")

else:

    print("✗ SIMILARITY MODEL EVALUATION FAILED")

print("=" * 60)

SIMILARITY MODEL EVALUATION

Minimum similarity: -0.709428
Maximum similarity: 1.0

Similarity matrix symmetric: True

Average similarity between different destinations: -0.013805

Top 10 most similar destination pairs:
 1. Jaisalmer ↔ Jodhpur : 0.9012
 2. Darjeeling ↔ Nainital : 0.8669
 3. Kolkata ↔ Varanasi : 0.8473
 4. Amritsar ↔ Jodhpur : 0.8052
 5. Srinagar ↔ Udaipur : 0.7918
 6. Coorg ↔ Kodaikanal : 0.7918
 7. Bhubaneswar ↔ Ranchi : 0.7674
 8. Gangtok ↔ Ranchi : 0.7629
 9. Bengaluru ↔ Mumbai : 0.7581
10. Ladakh ↔ Pahalgam : 0.7562

✓ SIMILARITY MODEL EVALUATION PASSED


In [ ]:
# ---------------------------------------------------------
# SAVE RECOMMENDATION MODEL ARTIFACTS
#
# We save the trained recommendation components so that
# the recommendation system can be loaded later without
# rebuilding the entire similarity matrix.
# ---------------------------------------------------------

import os
import pickle


# ---------------------------------------------------------
# 1. Make sure the models directory exists
# ---------------------------------------------------------

models_dir = "../models"

os.makedirs(
    models_dir,
    exist_ok=True
)


# ---------------------------------------------------------
# 2. Save the cosine similarity matrix
#
# This contains the similarity score between every pair
# of destinations.
# ---------------------------------------------------------

similarity_path = os.path.join(
    models_dir,
    "destination_similarity.pkl"
)

with open(similarity_path, "wb") as file:

    pickle.dump(
        similarity_df,
        file
    )


# ---------------------------------------------------------
# 3. Save destination names
#
# This preserves the mapping between the rows/columns of
# the similarity matrix and the actual destinations.
# ---------------------------------------------------------

destination_path = os.path.join(
    models_dir,
    "recommendation_destinations.pkl"
)

with open(destination_path, "wb") as file:

    pickle.dump(
        destination_names,
        file
    )


# ---------------------------------------------------------
# 4. Save PCA feature column names
#
# This tells the recommendation system which features must
# be supplied when creating a destination representation.
# ---------------------------------------------------------

recommendation_features_path = os.path.join(
    models_dir,
    "recommendation_feature_columns.pkl"
)

with open(recommendation_features_path, "wb") as file:

    pickle.dump(
        expected_pca_columns,
        file
    )


# ---------------------------------------------------------
# 5. Display saved files
# ---------------------------------------------------------

print("=" * 60)
print("RECOMMENDATION MODEL ARTIFACTS SAVED")
print("=" * 60)

print(f"\n✓ {similarity_path}")
print(f"✓ {destination_path}")
print(f"✓ {recommendation_features_path}")


# ---------------------------------------------------------
# 6. Verify that the files actually exist
# ---------------------------------------------------------

print("\nFile verification:")

for path in [
    similarity_path,
    destination_path,
    recommendation_features_path
]:

    if os.path.exists(path):

        file_size = os.path.getsize(path)

        print(
            f"✓ {path} "
            f"({file_size:,} bytes)"
        )

    else:

        print(
            f"✗ Missing: {path}"
        )


# ---------------------------------------------------------
# 7. Final validation
# ---------------------------------------------------------

all_saved = all(
    os.path.exists(path)
    for path in [
        similarity_path,
        destination_path,
        recommendation_features_path
    ]
)


print("\n" + "=" * 60)

if all_saved:

    print("✓ ALL RECOMMENDATION ARTIFACTS SAVED SUCCESSFULLY")

else:

    print("✗ ARTIFACT SAVING FAILED")

print("=" * 60)

RECOMMENDATION MODEL ARTIFACTS SAVED

✓ ../models\destination_similarity.pkl
✓ ../models\recommendation_destinations.pkl
✓ ../models\recommendation_feature_columns.pkl

File verification:
✓ ../models\destination_similarity.pkl (21,266 bytes)
✓ ../models\recommendation_destinations.pkl (1,258 bytes)
✓ ../models\recommendation_feature_columns.pkl (112 bytes)

✓ ALL RECOMMENDATION ARTIFACTS SAVED SUCCESSFULLY


In [17]:
# ---------------------------------------------------------
# IDENTIFY AVAILABLE USER-PREFERENCE FEATURES
#
# We inspect the original model features and determine
# which features can be used to represent user preferences.
#
# IMPORTANT:
# We only use features that actually exist in our dataset.
# We do not invent unavailable recommendation features.
# ---------------------------------------------------------

print("=" * 60)
print("AVAILABLE USER-PREFERENCE FEATURES")
print("=" * 60)


# ---------------------------------------------------------
# 1. Define groups of features that can represent different
#    types of user preferences.
# ---------------------------------------------------------

preference_feature_groups = {

    # -----------------------------------------------------
    # Budget-related features
    # -----------------------------------------------------
    "budget": [
        "avg_flight_price",
        "min_hotel_price",
        "avg_hotel_price",
        "max_hotel_price"
    ],

    # -----------------------------------------------------
    # Flight-related features
    # -----------------------------------------------------
    "flight": [
        "flight_count",
        "avg_flight_price",
        "avg_total_duration",
        "avg_outbound_stops",
        "avg_return_stops",
        "flight_available"
    ],

    # -----------------------------------------------------
    # Accommodation-related features
    # -----------------------------------------------------
    "accommodation": [
        "hotel_count",
        "room_count",
        "min_hotel_price",
        "avg_hotel_price",
        "max_hotel_price",
        "avg_allotment",
        "accommodation_available"
    ],

    # -----------------------------------------------------
    # Weather-related features
    # -----------------------------------------------------
    "weather": [
        "temperature",
        "feels_like",
        "humidity",
        "wind_speed",
        "cloudiness",
        "visibility",
        "rain_1h",
        "weather_available"
    ],

    # -----------------------------------------------------
    # Destination characteristics
    # -----------------------------------------------------
    "destination_characteristics": [
        "sight_count",
        "park_count",
        "restaurant_count",
        "water_count",
        "forest_count",
        "wetland_count",
        "river_count",
        "mountain_count",
        "coastal_count",
        "sand_count",
        "protected_area_count"
    ],

    # -----------------------------------------------------
    # Location
    # -----------------------------------------------------
    "location": [
        "latitude",
        "longitude"
    ]
}


# ---------------------------------------------------------
# 2. Check which features actually exist in our dataset.
# ---------------------------------------------------------

available_features = set(feature_columns)

print("\nFeature availability:\n")

for group, features in preference_feature_groups.items():

    available = [
        feature
        for feature in features
        if feature in available_features
    ]

    missing = [
        feature
        for feature in features
        if feature not in available_features
    ]

    print("-" * 60)
    print(f"{group.upper()}")

    print("\nAvailable:")
    print(available)

    if missing:
        print("\nMissing:")
        print(missing)


# ---------------------------------------------------------
# 3. Display features that cannot currently be used.
#
# These are the recommendation features we previously
# discussed but that are NOT present in the current dataset.
# ---------------------------------------------------------

unavailable_recommendation_features = [
    "Category",
    "Best_Season",
    "Average_Budget",
    "Recommended_Trip_Duration",
    "Popularity",
    "Family_Friendly",
    "Adventure_Score"
]

print("\n" + "=" * 60)
print("UNAVAILABLE RECOMMENDATION FEATURES")
print("=" * 60)

for feature in unavailable_recommendation_features:
    print(f"✗ {feature}")


# ---------------------------------------------------------
# 4. Final summary
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("PREFERENCE FEATURE ANALYSIS COMPLETE")
print("=" * 60)

AVAILABLE USER-PREFERENCE FEATURES

Feature availability:

------------------------------------------------------------
BUDGET

Available:
['avg_flight_price', 'min_hotel_price', 'avg_hotel_price', 'max_hotel_price']
------------------------------------------------------------
FLIGHT

Available:
['flight_count', 'avg_flight_price', 'avg_total_duration', 'avg_outbound_stops', 'avg_return_stops', 'flight_available']
------------------------------------------------------------
ACCOMMODATION

Available:
['hotel_count', 'room_count', 'min_hotel_price', 'avg_hotel_price', 'max_hotel_price', 'avg_allotment', 'accommodation_available']
------------------------------------------------------------
WEATHER

Available:
['temperature', 'feels_like', 'humidity', 'wind_speed', 'cloudiness', 'visibility', 'rain_1h', 'weather_available']
------------------------------------------------------------
DESTINATION_CHARACTERISTICS

Available:
['sight_count', 'park_count', 'restaurant_count', 'water_count', '

In [18]:
# ---------------------------------------------------------
# Build a normalized preference-scoring dataset.
#
# Each available feature is converted into a comparable
# 0-to-1 scale so that different units such as:
#
#   flight price
#   hotel price
#   temperature
#   number of attractions
#
# can be combined later.
#
# IMPORTANT:
# We use the ORIGINAL feature values here, not PCA values.
# PCA is already being used for destination similarity.
# ---------------------------------------------------------

import pandas as pd
import numpy as np


# ---------------------------------------------------------
# Load the processed dataset containing the original
# feature values.
#
# We use this dataset because user preferences need to be
# interpretable (for example, ₹10,000 flight price).
# ---------------------------------------------------------

processed_data_path = "../data/cleaned/travel_integrated_preprocessed.csv"

processed_df = pd.read_csv(processed_data_path)

print("=" * 60)
print("PROCESSED DATASET LOADED")
print("=" * 60)

print("Rows:", processed_df.shape[0])
print("Columns:", processed_df.shape[1])


# ---------------------------------------------------------
# Make sure the destination column exists.
# ---------------------------------------------------------

if "destination" not in processed_df.columns:
    raise ValueError(
        "Destination column is missing from the processed dataset."
    )


# ---------------------------------------------------------
# Define the features that can currently be used for
# preference-based recommendation.
#
# These are based only on features confirmed to exist
# in our project.
# ---------------------------------------------------------

preference_features = [
    "avg_flight_price",
    "flight_count",
    "avg_total_duration",
    "avg_outbound_stops",
    "avg_return_stops",
    "hotel_count",
    "room_count",
    "min_hotel_price",
    "avg_hotel_price",
    "max_hotel_price",
    "avg_allotment",
    "temperature",
    "feels_like",
    "humidity",
    "wind_speed",
    "cloudiness",
    "visibility",
    "rain_1h",
    "sight_count",
    "park_count",
    "restaurant_count",
    "water_count",
    "forest_count",
    "wetland_count",
    "river_count",
    "mountain_count",
    "coastal_count",
    "sand_count",
    "protected_area_count"
]


# ---------------------------------------------------------
# Check that every required feature actually exists.
# ---------------------------------------------------------

missing_preference_features = [
    feature
    for feature in preference_features
    if feature not in processed_df.columns
]

if missing_preference_features:
    raise ValueError(
        f"Missing preference features: {missing_preference_features}"
    )


print("\nAll preference features are available.")


# ---------------------------------------------------------
# Create a dataframe containing only the destination and
# preference features.
# ---------------------------------------------------------

preference_df = processed_df[
    ["destination"] + preference_features
].copy()


# ---------------------------------------------------------
# Convert all preference features to numeric values.
#
# Any unexpected non-numeric values become NaN so that
# we can detect them safely.
# ---------------------------------------------------------

for feature in preference_features:
    preference_df[feature] = pd.to_numeric(
        preference_df[feature],
        errors="coerce"
    )


# ---------------------------------------------------------
# Check for missing values.
# ---------------------------------------------------------

missing_values = preference_df[preference_features].isna().sum()

print("\nMissing values in preference features:")

print(
    missing_values[missing_values > 0]
)


# ---------------------------------------------------------
# Stop if any missing values are found.
#
# We do this instead of silently filling them because
# preference scores should be based on reliable data.
# ---------------------------------------------------------

if preference_df[preference_features].isna().sum().sum() > 0:
    raise ValueError(
        "Missing values detected in preference features."
    )


# ---------------------------------------------------------
# Normalize each feature using min-max normalization.
#
# Formula:
#
#     normalized = (value - minimum) /
#                  (maximum - minimum)
#
# Result:
#
#     minimum value → 0
#     maximum value → 1
#
# This makes different numerical features comparable.
# ---------------------------------------------------------

preference_scores = preference_df.copy()

for feature in preference_features:

    minimum = preference_df[feature].min()
    maximum = preference_df[feature].max()

    # If every destination has exactly the same value,
    # the feature provides no ranking information.
    if minimum == maximum:
        preference_scores[feature] = 0.5

    else:
        preference_scores[feature] = (
            (preference_df[feature] - minimum)
            / (maximum - minimum)
        )


# ---------------------------------------------------------
# Verify that normalization worked correctly.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("PREFERENCE FEATURES NORMALIZED")
print("=" * 60)

print(
    "\nMinimum normalized value:",
    preference_scores[preference_features].min().min()
)

print(
    "Maximum normalized value:",
    preference_scores[preference_features].max().max()
)


# ---------------------------------------------------------
# Validate the resulting dataset.
# ---------------------------------------------------------

print("\nRows:", preference_scores.shape[0])
print(
    "Preference features:",
    len(preference_features)
)

print(
    "Missing values:",
    preference_scores[preference_features]
    .isna()
    .sum()
    .sum()
)

print(
    "Duplicate destinations:",
    preference_scores["destination"]
    .duplicated()
    .sum()
)


# ---------------------------------------------------------
# Display a small sample.
# ---------------------------------------------------------

print("\nFirst 5 normalized preference rows:")

display(
    preference_scores[
        ["destination"] + preference_features
    ].head()
)


print("\n" + "=" * 60)
print("✓ PREFERENCE SCORING DATASET CREATED")
print("=" * 60)

PROCESSED DATASET LOADED
Rows: 50
Columns: 42

All preference features are available.

Missing values in preference features:
avg_flight_price      42
flight_count          42
avg_total_duration    42
avg_outbound_stops    42
avg_return_stops      42
hotel_count           11
room_count            11
min_hotel_price       11
avg_hotel_price       11
max_hotel_price       11
avg_allotment         11
dtype: int64


ValueError: Missing values detected in preference features.

In [19]:
# ---------------------------------------------------------
# Build the preference-scoring dataset.
#
# IMPORTANT:
# Missing flight and accommodation values represent
# unavailable external data, NOT zero-valued information.
#
# Therefore:
#   - We do NOT fill missing flight prices with averages.
#   - We do NOT fill missing hotel prices with averages.
#   - Availability flags will tell the recommendation
#     system whether those services are available.
# ---------------------------------------------------------

import pandas as pd
import numpy as np


# ---------------------------------------------------------
# Load the processed integrated dataset.
# ---------------------------------------------------------

processed_data_path = "../data/cleaned/travel_integrated_preprocessed.csv"

processed_df = pd.read_csv(processed_data_path)

print("=" * 60)
print("PROCESSED DATASET LOADED")
print("=" * 60)

print("Rows:", processed_df.shape[0])
print("Columns:", processed_df.shape[1])


# ---------------------------------------------------------
# Features that can currently be used for user preferences.
# ---------------------------------------------------------

preference_features = [
    "avg_flight_price",
    "flight_count",
    "avg_total_duration",
    "avg_outbound_stops",
    "avg_return_stops",
    
    "hotel_count",
    "room_count",
    "min_hotel_price",
    "avg_hotel_price",
    "max_hotel_price",
    "avg_allotment",
    
    "temperature",
    "feels_like",
    "humidity",
    "wind_speed",
    "cloudiness",
    "visibility",
    "rain_1h",
    
    "sight_count",
    "park_count",
    "restaurant_count",
    "water_count",
    "forest_count",
    "wetland_count",
    "river_count",
    "mountain_count",
    "coastal_count",
    "sand_count",
    "protected_area_count"
]


# ---------------------------------------------------------
# Availability flags.
#
# These tell us whether external data actually exists
# for a destination.
# ---------------------------------------------------------

availability_features = [
    "flight_available",
    "accommodation_available",
    "weather_available"
]


# ---------------------------------------------------------
# Verify that all required columns exist.
# ---------------------------------------------------------

required_columns = (
    ["destination"]
    + preference_features
    + availability_features
)

missing_columns = [
    column
    for column in required_columns
    if column not in processed_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


print("\nAll required preference columns are available.")


# ---------------------------------------------------------
# Create the preference dataframe.
# ---------------------------------------------------------

preference_df = processed_df[
    required_columns
].copy()


# ---------------------------------------------------------
# Convert preference features to numeric.
#
# Invalid values become NaN.
# ---------------------------------------------------------

for feature in preference_features:

    preference_df[feature] = pd.to_numeric(
        preference_df[feature],
        errors="coerce"
    )


# ---------------------------------------------------------
# Check availability distribution.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("EXTERNAL DATA AVAILABILITY")
print("=" * 60)

for feature in availability_features:

    print(f"\n{feature}:")
    print(
        preference_df[feature]
        .value_counts()
        .sort_index()
    )


# ---------------------------------------------------------
# Check which preference features contain missing values.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("MISSING PREFERENCE VALUES")
print("=" * 60)

missing_values = (
    preference_df[preference_features]
    .isna()
    .sum()
)

print(
    missing_values[missing_values > 0]
)


# ---------------------------------------------------------
# Expected missing values:
#
# Flight-related features:
#   Missing when flight_available == 0
#
# Accommodation-related features:
#   Missing when accommodation_available == 0
#
# Other destination/weather features should normally
# contain actual values.
# ---------------------------------------------------------

flight_features = [
    "avg_flight_price",
    "flight_count",
    "avg_total_duration",
    "avg_outbound_stops",
    "avg_return_stops"
]

accommodation_features = [
    "hotel_count",
    "room_count",
    "min_hotel_price",
    "avg_hotel_price",
    "max_hotel_price",
    "avg_allotment"
]


# ---------------------------------------------------------
# Verify that flight missing values correspond to
# unavailable flight data.
# ---------------------------------------------------------

flight_invalid_rows = preference_df[
    (preference_df["flight_available"] == 1)
    & (preference_df[flight_features].isna().any(axis=1))
]

if len(flight_invalid_rows) > 0:

    print(
        "\nWARNING: Flight data is missing even though "
        "flight_available = 1."
    )

    display(
        flight_invalid_rows[
            ["destination", "flight_available"]
            + flight_features
        ]
    )

else:

    print(
        "\n✓ Flight missing values are consistent "
        "with flight availability."
    )


# ---------------------------------------------------------
# Verify that accommodation missing values correspond
# to unavailable accommodation data.
# ---------------------------------------------------------

accommodation_invalid_rows = preference_df[
    (preference_df["accommodation_available"] == 1)
    & (preference_df[accommodation_features].isna().any(axis=1))
]

if len(accommodation_invalid_rows) > 0:

    print(
        "\nWARNING: Accommodation data is missing "
        "even though accommodation_available = 1."
    )

    display(
        accommodation_invalid_rows[
            ["destination", "accommodation_available"]
            + accommodation_features
        ]
    )

else:

    print(
        "\n✓ Accommodation missing values are consistent "
        "with accommodation availability."
    )


# ---------------------------------------------------------
# For preference scoring:
#
# We create separate normalized values while preserving
# the original missing values.
#
# Missing values will NOT be treated as zero.
# ---------------------------------------------------------

preference_scores = preference_df.copy()


# ---------------------------------------------------------
# Normalize each feature using Min-Max scaling.
#
# NaN values remain NaN.
#
# This is intentional:
# unavailable data should remain unavailable until the
# preference scoring stage decides how to handle it.
# ---------------------------------------------------------

for feature in preference_features:

    minimum = preference_df[feature].min(
        skipna=True
    )

    maximum = preference_df[feature].max(
        skipna=True
    )

    if minimum == maximum:

        preference_scores[feature] = np.where(
            preference_df[feature].notna(),
            0.5,
            np.nan
        )

    else:

        preference_scores[feature] = (
            (preference_df[feature] - minimum)
            / (maximum - minimum)
        )


# ---------------------------------------------------------
# Validation.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("PREFERENCE DATASET CREATED")
print("=" * 60)

print(
    "Rows:",
    preference_scores.shape[0]
)

print(
    "Preference features:",
    len(preference_features)
)

print(
    "Unique destinations:",
    preference_scores["destination"].nunique()
)

print(
    "Duplicate destinations:",
    preference_scores["destination"]
    .duplicated()
    .sum()
)


# ---------------------------------------------------------
# Show remaining missing values.
#
# These should only be the legitimate unavailable
# flight/accommodation values.
# ---------------------------------------------------------

print("\nRemaining missing values:")

remaining_missing = (
    preference_scores[preference_features]
    .isna()
    .sum()
)

print(
    remaining_missing[remaining_missing > 0]
)


# ---------------------------------------------------------
# Display first few rows.
# ---------------------------------------------------------

print("\nFirst 5 preference rows:")

display(
    preference_scores.head()
)


print("\n" + "=" * 60)
print("✓ PREFERENCE DATA PREPARATION COMPLETE")
print("=" * 60)

PROCESSED DATASET LOADED
Rows: 50
Columns: 42

All required preference columns are available.

EXTERNAL DATA AVAILABILITY

flight_available:
flight_available
0    42
1     8
Name: count, dtype: int64

accommodation_available:
accommodation_available
0    11
1    39
Name: count, dtype: int64

weather_available:
weather_available
1    50
Name: count, dtype: int64

MISSING PREFERENCE VALUES
avg_flight_price      42
flight_count          42
avg_total_duration    42
avg_outbound_stops    42
avg_return_stops      42
hotel_count           11
room_count            11
min_hotel_price       11
avg_hotel_price       11
max_hotel_price       11
avg_allotment         11
dtype: int64

✓ Flight missing values are consistent with flight availability.

✓ Accommodation missing values are consistent with accommodation availability.

PREFERENCE DATASET CREATED
Rows: 50
Preference features: 29
Unique destinations: 50
Duplicate destinations: 0

Remaining missing values:
avg_flight_price      42
flight_count

,destination,avg_flight_price,flight_count,avg_total_duration,avg_outbound_stops,avg_return_stops,hotel_count,room_count,min_hotel_price,avg_hotel_price,...,forest_count,wetland_count,river_count,mountain_count,coastal_count,sand_count,protected_area_count,flight_available,accommodation_available,weather_available
0,Agra,NaN,NaN,NaN,NaN,NaN,0.076923,0.123457,0.000000,0.002899,...,0.329897,0.073171,0.058824,0.0,0.000000,0.0,0.0,0,1,1
1,Ahmedabad,NaN,NaN,NaN,NaN,NaN,0.153846,0.419753,0.135101,0.136185,...,0.020619,0.000000,0.117647,0.0,0.000000,0.0,0.0,0,1,1
2,Alappuzha,NaN,NaN,NaN,NaN,NaN,0.025641,0.037037,0.434744,0.284085,...,0.020619,0.560976,0.529412,0.0,0.083333,0.0,0.0,0,1,1
3,Amritsar,NaN,NaN,NaN,NaN,NaN,0.128205,0.148148,0.101198,0.119268,...,0.092784,0.024390,0.000000,0.0,0.000000,0.0,0.0,0,1,1
4,Andaman,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.020619,0.000000,0.000000,0.0,0.000000,0.0,0.0,0,0,1



✓ PREFERENCE DATA PREPARATION COMPLETE


In [20]:
# ---------------------------------------------------------
# DEFINE PREFERENCE DIRECTIONS
#
# Our normalized preference values currently use:
#
#     0 = lowest observed value
#     1 = highest observed value
#
# However, "higher is better" is not true for every feature.
#
# Example:
#   Higher sightseeing count  -> better
#   Lower flight price        -> better
#   Lower hotel price         -> better
#
# Therefore, we explicitly define the direction of every
# feature before calculating user-preference scores.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Features where HIGHER values are considered better.
# ---------------------------------------------------------

higher_is_better = [

    # Destination characteristics
    "sight_count",
    "park_count",
    "restaurant_count",
    "water_count",
    "forest_count",
    "wetland_count",
    "river_count",
    "mountain_count",
    "coastal_count",
    "sand_count",
    "protected_area_count",

    # Flight availability / options
    "flight_count",

    # Accommodation availability / capacity
    "hotel_count",
    "room_count",
    "avg_allotment",

    # Weather comfort / visibility
    "temperature",
    "feels_like",
    "visibility"
]


# ---------------------------------------------------------
# Features where LOWER values are considered better.
# ---------------------------------------------------------

lower_is_better = [

    # Lower flight cost is preferable
    "avg_flight_price",

    # Shorter travel time is preferable
    "avg_total_duration",

    # Fewer stops are preferable
    "avg_outbound_stops",
    "avg_return_stops",

    # Lower accommodation cost is preferable
    "min_hotel_price",
    "avg_hotel_price",
    "max_hotel_price",

    # Lower humidity can generally be more comfortable
    "humidity",

    # Lower wind speed is generally more comfortable
    "wind_speed",

    # Lower cloudiness is generally preferable
    "cloudiness",

    # Lower rainfall is generally preferable
    "rain_1h"
]


# ---------------------------------------------------------
# Verify that every preference feature has been assigned
# exactly one direction.
# ---------------------------------------------------------

direction_features = (
    higher_is_better
    + lower_is_better
)


print("=" * 60)
print("PREFERENCE DIRECTION VALIDATION")
print("=" * 60)

print(
    "Total preference features:",
    len(preference_features)
)

print(
    "Higher-is-better features:",
    len(higher_is_better)
)

print(
    "Lower-is-better features:",
    len(lower_is_better)
)


# ---------------------------------------------------------
# Check for duplicated assignments.
# A feature should not appear in both lists.
# ---------------------------------------------------------

duplicated_directions = set(
    higher_is_better
).intersection(
    lower_is_better
)


print(
    "\nFeatures assigned to BOTH directions:"
)

print(
    sorted(duplicated_directions)
)


# ---------------------------------------------------------
# Check whether any preference feature has not been
# assigned a direction.
# ---------------------------------------------------------

unassigned_features = set(
    preference_features
) - set(
    direction_features
)


print(
    "\nUnassigned preference features:"
)

print(
    sorted(unassigned_features)
)


# ---------------------------------------------------------
# Check whether every listed direction feature actually
# exists in our preference feature list.
# ---------------------------------------------------------

unexpected_features = set(
    direction_features
) - set(
    preference_features
)


print(
    "\nUnexpected direction features:"
)

print(
    sorted(unexpected_features)
)


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

if (
    len(duplicated_directions) == 0
    and len(unassigned_features) == 0
    and len(unexpected_features) == 0
):

    print(
        "\n✓ Every preference feature has exactly "
        "one direction."
    )

else:

    raise ValueError(
        "Preference direction validation failed."
    )


print("\n" + "=" * 60)
print("✓ PREFERENCE DIRECTION MAP CREATED")
print("=" * 60)

PREFERENCE DIRECTION VALIDATION
Total preference features: 29
Higher-is-better features: 18
Lower-is-better features: 11

Features assigned to BOTH directions:
[]

Unassigned preference features:
[]

Unexpected direction features:
[]

✓ Every preference feature has exactly one direction.

✓ PREFERENCE DIRECTION MAP CREATED


In [21]:
# ---------------------------------------------------------
# CREATE DIRECTION-CORRECTED PREFERENCE MATRIX
#
# The preference_scores dataframe contains Min-Max values:
#
#     0 = lowest observed value
#     1 = highest observed value
#
# For higher-is-better features, this is already correct.
#
# For lower-is-better features, we reverse the score:
#
#     corrected_score = 1 - normalized_score
#
# Missing values remain NaN because they represent
# unavailable flight/accommodation information.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Start with a copy so that the original normalized
# preference_scores dataframe remains unchanged.
# ---------------------------------------------------------

direction_corrected_df = preference_scores.copy()


# ---------------------------------------------------------
# Reverse the features where LOWER values are better.
# ---------------------------------------------------------

for feature in lower_is_better:

    direction_corrected_df[feature] = (
        1 - preference_scores[feature]
    )


# ---------------------------------------------------------
# Higher-is-better features remain unchanged.
#
# We explicitly copy them here for clarity and to make
# the processing logic easy to understand.
# ---------------------------------------------------------

for feature in higher_is_better:

    direction_corrected_df[feature] = (
        preference_scores[feature]
    )


# ---------------------------------------------------------
# Verify that all preference features are still present.
# ---------------------------------------------------------

missing_features = [
    feature
    for feature in preference_features
    if feature not in direction_corrected_df.columns
]

if missing_features:

    raise ValueError(
        f"Missing preference features: {missing_features}"
    )


# ---------------------------------------------------------
# Check that all non-missing corrected values are within
# the expected [0, 1] range.
# ---------------------------------------------------------

corrected_values = direction_corrected_df[
    preference_features
]

min_value = corrected_values.min(
    skipna=True
).min()

max_value = corrected_values.max(
    skipna=True
).max()


print("=" * 60)
print("DIRECTION-CORRECTED PREFERENCE MATRIX")
print("=" * 60)

print(
    "Rows:",
    direction_corrected_df.shape[0]
)

print(
    "Preference features:",
    len(preference_features)
)

print(
    "Minimum corrected value:",
    round(min_value, 6)
)

print(
    "Maximum corrected value:",
    round(max_value, 6)
)


# ---------------------------------------------------------
# Validate the value range.
# ---------------------------------------------------------

if min_value < 0 or max_value > 1:

    raise ValueError(
        "Corrected preference values are outside [0, 1]."
    )

print(
    "\n✓ All available preference scores are within [0, 1]."
)


# ---------------------------------------------------------
# Check missing values.
#
# Missing values are expected only for flight and
# accommodation features where external data is unavailable.
# ---------------------------------------------------------

remaining_missing = (
    direction_corrected_df[
        preference_features
    ]
    .isna()
    .sum()
)

print("\nRemaining missing values:")

print(
    remaining_missing[
        remaining_missing > 0
    ]
)


# ---------------------------------------------------------
# Show examples of lower-is-better features after
# direction correction.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("LOWER-IS-BETTER FEATURE CHECK")
print("=" * 60)

example_features = [
    "avg_flight_price",
    "avg_total_duration",
    "avg_hotel_price",
    "humidity",
    "rain_1h"
]

display(
    direction_corrected_df[
        ["destination"] + example_features
    ].head(10)
)


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("✓ DIRECTION-CORRECTED PREFERENCE MATRIX CREATED")
print("=" * 60)

DIRECTION-CORRECTED PREFERENCE MATRIX
Rows: 50
Preference features: 29
Minimum corrected value: 0.0
Maximum corrected value: 1.0

✓ All available preference scores are within [0, 1].

Remaining missing values:
avg_flight_price      42
flight_count          42
avg_total_duration    42
avg_outbound_stops    42
avg_return_stops      42
hotel_count           11
room_count            11
min_hotel_price       11
avg_hotel_price       11
max_hotel_price       11
avg_allotment         11
dtype: int64

LOWER-IS-BETTER FEATURE CHECK


,destination,avg_flight_price,avg_total_duration,avg_hotel_price,humidity,rain_1h
0,Agra,NaN,NaN,0.997101,0.0250,1.000000
1,Ahmedabad,NaN,NaN,0.863815,0.4750,1.000000
2,Alappuzha,NaN,NaN,0.715915,0.1750,0.895288
3,Amritsar,NaN,NaN,0.880732,0.4625,1.000000
4,Andaman,NaN,NaN,NaN,0.1125,0.753927
5,Bengaluru,1.0,0.979084,0.866849,0.3625,1.000000
6,Bhopal,NaN,NaN,0.990835,0.0250,0.261780
7,Bhubaneswar,NaN,NaN,0.919140,0.2125,0.041885
8,Chennai,NaN,NaN,0.941872,0.2875,1.000000
9,Coorg,NaN,NaN,NaN,0.0625,0.937173



✓ DIRECTION-CORRECTED PREFERENCE MATRIX CREATED


In [22]:
# ---------------------------------------------------------
# CREATE USER PREFERENCE CONFIGURATION
#
# Instead of asking the user to provide values for all
# 29 individual features, we organize them into meaningful
# travel-preference groups.
#
# These groups will later be used to calculate a personalized
# destination score.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Define the major preference groups.
# ---------------------------------------------------------

preference_groups = {

    "budget": [
        "avg_flight_price",
        "min_hotel_price",
        "avg_hotel_price",
        "max_hotel_price"
    ],

    "flight": [
        "flight_count",
        "avg_flight_price",
        "avg_total_duration",
        "avg_outbound_stops",
        "avg_return_stops"
    ],

    "accommodation": [
        "hotel_count",
        "room_count",
        "min_hotel_price",
        "avg_hotel_price",
        "max_hotel_price",
        "avg_allotment"
    ],

    "weather": [
        "temperature",
        "feels_like",
        "humidity",
        "wind_speed",
        "cloudiness",
        "visibility",
        "rain_1h"
    ],

    "destination_characteristics": [
        "sight_count",
        "park_count",
        "restaurant_count",
        "water_count",
        "forest_count",
        "wetland_count",
        "river_count",
        "mountain_count",
        "coastal_count",
        "sand_count",
        "protected_area_count"
    ]
}


# ---------------------------------------------------------
# Validate that every feature used in the preference system
# actually exists in our processed preference dataset.
# ---------------------------------------------------------

all_group_features = []

for group, features in preference_groups.items():

    all_group_features.extend(features)


missing_group_features = set(
    all_group_features
) - set(
    preference_features
)


unexpected_group_features = set(
    preference_features
) - set(
    all_group_features
)


# ---------------------------------------------------------
# Check for duplicate feature assignments.
#
# Some features such as avg_flight_price and avg_hotel_price
# intentionally belong to more than one conceptual group.
# Therefore, duplicate assignments across groups are allowed.
#
# We only check whether every feature is represented somewhere.
# ---------------------------------------------------------

print("=" * 60)
print("USER PREFERENCE GROUPS")
print("=" * 60)

for group, features in preference_groups.items():

    print(f"\n{group.upper()}")

    for feature in features:

        print(f"  - {feature}")


print("\n" + "=" * 60)
print("PREFERENCE GROUP VALIDATION")
print("=" * 60)

print(
    "Total individual preference features:",
    len(preference_features)
)

print(
    "Features represented in groups:",
    len(set(all_group_features))
)

print(
    "Missing from groups:",
    sorted(missing_group_features)
)

print(
    "Preference features not assigned to any group:",
    sorted(unexpected_group_features)
)


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

if (
    len(missing_group_features) == 0
    and len(unexpected_group_features) == 0
):

    print(
        "\n✓ All preference features are represented "
        "in the preference groups."
    )

else:

    raise ValueError(
        "Preference group validation failed."
    )


print("\n" + "=" * 60)
print("✓ USER PREFERENCE CONFIGURATION CREATED")
print("=" * 60)

USER PREFERENCE GROUPS

BUDGET
  - avg_flight_price
  - min_hotel_price
  - avg_hotel_price
  - max_hotel_price

FLIGHT
  - flight_count
  - avg_flight_price
  - avg_total_duration
  - avg_outbound_stops
  - avg_return_stops

ACCOMMODATION
  - hotel_count
  - room_count
  - min_hotel_price
  - avg_hotel_price
  - max_hotel_price
  - avg_allotment

WEATHER
  - temperature
  - feels_like
  - humidity
  - wind_speed
  - cloudiness
  - visibility
  - rain_1h

DESTINATION_CHARACTERISTICS
  - sight_count
  - park_count
  - restaurant_count
  - water_count
  - forest_count
  - wetland_count
  - river_count
  - mountain_count
  - coastal_count
  - sand_count
  - protected_area_count

PREFERENCE GROUP VALIDATION
Total individual preference features: 29
Features represented in groups: 29
Missing from groups: []
Preference features not assigned to any group: []

✓ All preference features are represented in the preference groups.

✓ USER PREFERENCE CONFIGURATION CREATED


In [23]:
# ---------------------------------------------------------
# CREATE DERIVED DESTINATION PREFERENCE PROFILES
#
# We do not have the original Category / Adventure /
# Family_Friendly / Popularity features.
#
# Therefore, we create meaningful preference profiles
# using only the destination characteristics that we
# actually collected.
#
# These profiles will later allow a user to express
# preferences such as:
#
#   - Nature
#   - Sightseeing
#   - Water / Coastal
#   - Wildlife
#
# without requiring unavailable dataset columns.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Define the feature composition of each profile.
#
# Each profile contains only features available in our
# current dataset.
# ---------------------------------------------------------

destination_profiles = {

    "nature": [
        "forest_count",
        "mountain_count",
        "river_count",
        "protected_area_count",
        "water_count"
    ],

    "sightseeing": [
        "sight_count",
        "park_count",
        "restaurant_count"
    ],

    "water_coastal": [
        "water_count",
        "coastal_count",
        "sand_count"
    ],

    "wildlife": [
        "forest_count",
        "protected_area_count",
        "wetland_count"
    ]
}


# ---------------------------------------------------------
# Validate that every profile feature exists in the
# direction-corrected preference dataframe.
# ---------------------------------------------------------

profile_features = []

for profile, features in destination_profiles.items():

    profile_features.extend(features)


missing_profile_features = (
    set(profile_features)
    - set(preference_features)
)


print("=" * 60)
print("DERIVED DESTINATION PREFERENCE PROFILES")
print("=" * 60)

print(
    "Number of profiles:",
    len(destination_profiles)
)

print(
    "Missing profile features:",
    sorted(missing_profile_features)
)


if missing_profile_features:

    raise ValueError(
        "Some profile features are unavailable."
    )


# ---------------------------------------------------------
# Calculate a normalized profile score for every destination.
#
# We use the direction-corrected values so that:
#
#     higher score = stronger match with that profile
#
# Missing values are ignored when calculating the mean.
# ---------------------------------------------------------

profile_scores_df = pd.DataFrame()

profile_scores_df["destination"] = (
    direction_corrected_df["destination"]
)


for profile, features in destination_profiles.items():

    profile_scores_df[
        f"{profile}_score"
    ] = (
        direction_corrected_df[features]
        .mean(axis=1, skipna=True)
    )


# ---------------------------------------------------------
# Check the generated profile scores.
# ---------------------------------------------------------

profile_score_columns = [
    f"{profile}_score"
    for profile in destination_profiles
]


print("\nProfile score columns:")

for column in profile_score_columns:

    print(
        f"  - {column}"
    )


print("\nShape:")

print(
    profile_scores_df.shape
)


# ---------------------------------------------------------
# Check for missing profile scores.
# ---------------------------------------------------------

print("\nMissing profile scores:")

print(
    profile_scores_df[
        profile_score_columns
    ]
    .isna()
    .sum()
)


# ---------------------------------------------------------
# Verify score range.
# ---------------------------------------------------------

profile_min = (
    profile_scores_df[
        profile_score_columns
    ]
    .min()
    .min()
)

profile_max = (
    profile_scores_df[
        profile_score_columns
    ]
    .max()
    .max()
)


print(
    "\nMinimum profile score:",
    round(profile_min, 6)
)

print(
    "Maximum profile score:",
    round(profile_max, 6)
)


# ---------------------------------------------------------
# Display the strongest destinations for each profile.
# This is only a diagnostic check.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("PROFILE SCORE PREVIEW")
print("=" * 60)

for profile in destination_profiles:

    column = f"{profile}_score"

    print(
        f"\nTop destinations for {profile}:"
    )

    display(
        profile_scores_df[
            ["destination", column]
        ]
        .sort_values(
            column,
            ascending=False
        )
        .head(5)
    )


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

if profile_min < 0 or profile_max > 1:

    raise ValueError(
        "Profile scores are outside the expected [0, 1] range."
    )


if (
    profile_scores_df["destination"].nunique()
    != len(profile_scores_df)
):

    raise ValueError(
        "Duplicate destinations found in profile scores."
    )


print("\n" + "=" * 60)
print("✓ DESTINATION PREFERENCE PROFILES CREATED")
print("=" * 60)

DERIVED DESTINATION PREFERENCE PROFILES
Number of profiles: 4
Missing profile features: []

Profile score columns:
  - nature_score
  - sightseeing_score
  - water_coastal_score
  - wildlife_score

Shape:
(50, 5)

Missing profile scores:
nature_score           0
sightseeing_score      0
water_coastal_score    0
wildlife_score         0
dtype: int64

Minimum profile score: 0.0
Maximum profile score: 0.829316

PROFILE SCORE PREVIEW

Top destinations for nature:


,destination,nature_score
29,Manali,0.619459
30,Mumbai,0.437710
19,Jaipur,0.407148
5,Bengaluru,0.393116
44,Srinagar,0.380202



Top destinations for sightseeing:


,destination,sightseeing_score
30,Mumbai,0.829316
5,Bengaluru,0.820000
17,Hyderabad,0.791368
11,Delhi,0.786325
45,Thiruvananthapuram,0.751966



Top destinations for water_coastal:


,destination,water_coastal_score
48,Varkala,0.509637
8,Chennai,0.501701
24,Kochi,0.455382
15,Gokarna,0.391156
42,Shillong,0.387755



Top destinations for wildlife:


,destination,wildlife_score
29,Manali,0.570447
30,Mumbai,0.531850
10,Darjeeling,0.500000
43,Shimla,0.479381
23,Kaziranga,0.450172



✓ DESTINATION PREFERENCE PROFILES CREATED


In [24]:
# ---------------------------------------------------------
# CREATE USER PREFERENCE INPUT CONFIGURATION
#
# This defines the structure of the preferences that a user
# can provide to the recommendation system.
#
# The user does not need to provide values for all 29
# individual features.
#
# Instead, we use high-level preference groups and convert
# them into numerical weights later.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Define the preference groups available to the user.
#
# Weight meaning:
#   0.0 = user does not care about this factor
#   1.0 = user considers this factor very important
# ---------------------------------------------------------

user_preference_groups = [
    "budget",
    "flight",
    "accommodation",
    "weather",
    "destination_characteristics"
]


# ---------------------------------------------------------
# Define the destination-interest profiles created earlier.
#
# These are derived from the actual data available in our
# project, rather than unavailable features such as
# Adventure_Score or Family_Friendly.
# ---------------------------------------------------------

user_interest_profiles = [
    "nature",
    "sightseeing",
    "water_coastal",
    "wildlife"
]


# ---------------------------------------------------------
# Define default weights.
#
# These are neutral defaults only.
# Later, the actual application will replace these values
# with the user's selections.
# ---------------------------------------------------------

default_user_preferences = {

    "budget": 0.5,

    "flight": 0.5,

    "accommodation": 0.5,

    "weather": 0.5,

    "destination_characteristics": 0.5
}


# ---------------------------------------------------------
# Define default interest weights.
#
# A value of 0 means the user has not specifically selected
# that interest.
# ---------------------------------------------------------

default_interest_preferences = {

    "nature": 0.0,

    "sightseeing": 0.0,

    "water_coastal": 0.0,

    "wildlife": 0.0
}


# ---------------------------------------------------------
# Validate preference group definitions.
# ---------------------------------------------------------

print("=" * 60)
print("USER PREFERENCE INPUT CONFIGURATION")
print("=" * 60)

print("\nPreference groups:")

for group in user_preference_groups:

    print(f"  - {group}")


print("\nDestination interest profiles:")

for profile in user_interest_profiles:

    print(f"  - {profile}")


# ---------------------------------------------------------
# Validate that every preference group has a default value.
# ---------------------------------------------------------

missing_default_groups = (
    set(user_preference_groups)
    - set(default_user_preferences)
)


unexpected_default_groups = (
    set(default_user_preferences)
    - set(user_preference_groups)
)


print("\n" + "=" * 60)
print("PREFERENCE GROUP VALIDATION")
print("=" * 60)

print(
    "Missing default groups:",
    sorted(missing_default_groups)
)

print(
    "Unexpected default groups:",
    sorted(unexpected_default_groups)
)


# ---------------------------------------------------------
# Validate that every interest profile has a default value.
# ---------------------------------------------------------

missing_default_profiles = (
    set(user_interest_profiles)
    - set(default_interest_preferences)
)


unexpected_default_profiles = (
    set(default_interest_preferences)
    - set(user_interest_profiles)
)


print("\nINTEREST PROFILE VALIDATION")

print(
    "Missing default profiles:",
    sorted(missing_default_profiles)
)

print(
    "Unexpected default profiles:",
    sorted(unexpected_default_profiles)
)


# ---------------------------------------------------------
# Validate that all default preference weights are between
# 0 and 1.
# ---------------------------------------------------------

invalid_group_weights = {
    group: weight
    for group, weight in default_user_preferences.items()
    if not 0 <= weight <= 1
}


invalid_profile_weights = {
    profile: weight
    for profile, weight in default_interest_preferences.items()
    if not 0 <= weight <= 1
}


print("\nWEIGHT RANGE VALIDATION")

print(
    "Invalid group weights:",
    invalid_group_weights
)

print(
    "Invalid profile weights:",
    invalid_profile_weights
)


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

if (
    missing_default_groups
    or unexpected_default_groups
    or missing_default_profiles
    or unexpected_default_profiles
    or invalid_group_weights
    or invalid_profile_weights
):

    raise ValueError(
        "User preference configuration validation failed."
    )


print("\n" + "=" * 60)
print("✓ USER PREFERENCE INPUT CONFIGURATION VALIDATED")
print("=" * 60)

USER PREFERENCE INPUT CONFIGURATION

Preference groups:
  - budget
  - flight
  - accommodation
  - weather
  - destination_characteristics

Destination interest profiles:
  - nature
  - sightseeing
  - water_coastal
  - wildlife

PREFERENCE GROUP VALIDATION
Missing default groups: []
Unexpected default groups: []

INTEREST PROFILE VALIDATION
Missing default profiles: []
Unexpected default profiles: []

WEIGHT RANGE VALIDATION
Invalid group weights: {}
Invalid profile weights: {}

✓ USER PREFERENCE INPUT CONFIGURATION VALIDATED


In [25]:
# ---------------------------------------------------------
# CREATE SAMPLE USER PREFERENCE PROFILE
#
# This is a test user that we will use to verify the
# personalized recommendation scoring pipeline.
#
# These values can later come directly from the Travel Agent
# application's user interface.
#
# Weight interpretation:
#   0.0 = not important
#   0.5 = moderately important
#   1.0 = very important
# ---------------------------------------------------------


# ---------------------------------------------------------
# Travel-factor preferences
# ---------------------------------------------------------

sample_user_preferences = {

    # User strongly cares about keeping the trip affordable.
    "budget": 0.9,

    # Flights are moderately important.
    "flight": 0.6,

    # Accommodation is moderately important.
    "accommodation": 0.6,

    # Weather is highly important.
    "weather": 0.8,

    # General destination characteristics are important.
    "destination_characteristics": 0.8
}


# ---------------------------------------------------------
# Destination-interest preferences
#
# The user can have multiple interests simultaneously.
# ---------------------------------------------------------

sample_user_interests = {

    # Strong preference for nature.
    "nature": 1.0,

    # Moderate preference for sightseeing.
    "sightseeing": 0.5,

    # Low preference for coastal destinations.
    "water_coastal": 0.2,

    # Moderate preference for wildlife.
    "wildlife": 0.6
}


# ---------------------------------------------------------
# Validate travel-factor preferences.
# ---------------------------------------------------------

print("=" * 60)
print("SAMPLE USER PREFERENCE PROFILE")
print("=" * 60)

print("\nTravel-factor preferences:")

for group, weight in sample_user_preferences.items():

    print(
        f"  {group:<30} {weight:.2f}"
    )


# ---------------------------------------------------------
# Validate destination-interest preferences.
# ---------------------------------------------------------

print("\nDestination interests:")

for profile, weight in sample_user_interests.items():

    print(
        f"  {profile:<30} {weight:.2f}"
    )


# ---------------------------------------------------------
# Check that all expected groups are present.
# ---------------------------------------------------------

missing_groups = (
    set(user_preference_groups)
    - set(sample_user_preferences)
)

unexpected_groups = (
    set(sample_user_preferences)
    - set(user_preference_groups)
)


# ---------------------------------------------------------
# Check that all expected interest profiles are present.
# ---------------------------------------------------------

missing_interests = (
    set(user_interest_profiles)
    - set(sample_user_interests)
)

unexpected_interests = (
    set(sample_user_interests)
    - set(user_interest_profiles)
)


# ---------------------------------------------------------
# Check that all values are between 0 and 1.
# ---------------------------------------------------------

invalid_group_values = {
    group: value
    for group, value in sample_user_preferences.items()
    if not 0 <= value <= 1
}

invalid_interest_values = {
    profile: value
    for profile, value in sample_user_interests.items()
    if not 0 <= value <= 1
}


# ---------------------------------------------------------
# Display validation results.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("SAMPLE USER VALIDATION")
print("=" * 60)

print(
    "Missing preference groups:",
    sorted(missing_groups)
)

print(
    "Unexpected preference groups:",
    sorted(unexpected_groups)
)

print(
    "Missing interest profiles:",
    sorted(missing_interests)
)

print(
    "Unexpected interest profiles:",
    sorted(unexpected_interests)
)

print(
    "Invalid preference values:",
    invalid_group_values
)

print(
    "Invalid interest values:",
    invalid_interest_values
)


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

if (
    missing_groups
    or unexpected_groups
    or missing_interests
    or unexpected_interests
    or invalid_group_values
    or invalid_interest_values
):

    raise ValueError(
        "Sample user preference validation failed."
    )


print("\n" + "=" * 60)
print("✓ SAMPLE USER PREFERENCE PROFILE VALIDATED")
print("=" * 60)

SAMPLE USER PREFERENCE PROFILE

Travel-factor preferences:
  budget                         0.90
  flight                         0.60
  accommodation                  0.60
  weather                        0.80
  destination_characteristics    0.80

Destination interests:
  nature                         1.00
  sightseeing                    0.50
  water_coastal                  0.20
  wildlife                       0.60

SAMPLE USER VALIDATION
Missing preference groups: []
Unexpected preference groups: []
Missing interest profiles: []
Unexpected interest profiles: []
Invalid preference values: {}
Invalid interest values: {}

✓ SAMPLE USER PREFERENCE PROFILE VALIDATED


In [27]:
# ---------------------------------------------------------
# CALCULATE DESTINATION SCORES FOR EACH USER PREFERENCE
# GROUP
#
# Each destination receives a score from 0 to 1 for:
#
#   1. Budget
#   2. Flight
#   3. Accommodation
#   4. Weather
#   5. Destination characteristics
#
# Missing external data is NOT treated as zero.
# Instead, unavailable features are excluded from the
# corresponding group calculation.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Create a dataframe containing the destination names.
# ---------------------------------------------------------

preference_scores_df = pd.DataFrame()

preference_scores_df["destination"] = (
    preference_df["destination"]
)


# ---------------------------------------------------------
# Helper function
#
# Calculates the mean of available feature values.
#
# Example:
#
# If a destination has:
#   feature 1 = 0.8
#   feature 2 = 0.6
#   feature 3 = NaN
#
# Score = (0.8 + 0.6) / 2
#
# The missing feature does NOT reduce the score.
# ---------------------------------------------------------

def calculate_available_score(df, features):

    return df[features].mean(
        axis=1,
        skipna=True
    )


# ---------------------------------------------------------
# Calculate the five major preference scores.
#
# These scores represent how well each destination performs
# for each travel factor.
# ---------------------------------------------------------

for group, features in preference_groups.items():

    preference_scores_df[
        f"{group}_score"
    ] = calculate_available_score(
        direction_corrected_df,
        features
    )


# ---------------------------------------------------------
# Display the resulting score columns.
# ---------------------------------------------------------

score_columns = [
    f"{group}_score"
    for group in preference_groups
]


print("=" * 60)
print("DESTINATION PREFERENCE SCORES")
print("=" * 60)

print("\nScore columns:")

for column in score_columns:

    print(f"  - {column}")


print("\nShape:")

print(
    preference_scores_df.shape
)


# ---------------------------------------------------------
# Check for missing scores.
#
# A group can become completely unavailable for a
# destination if ALL of its features are missing.
# ---------------------------------------------------------

print("\nMissing group scores:")

print(
    preference_scores_df[
        score_columns
    ]
    .isna()
    .sum()
)


# ---------------------------------------------------------
# Check score ranges.
# ---------------------------------------------------------

print("\nScore ranges:")

for column in score_columns:

    minimum = preference_scores_df[column].min(
        skipna=True
    )

    maximum = preference_scores_df[column].max(
        skipna=True
    )

    print(
        f"{column:<35} "
        f"{minimum:.4f} → {maximum:.4f}"
    )


# ---------------------------------------------------------
# Display the first few destinations.
# ---------------------------------------------------------

print("\nFirst 10 destination scores:")

display(
    preference_scores_df[
        ["destination"] + score_columns
    ].head(10)
)


# ---------------------------------------------------------
# Final structural validation.
# ---------------------------------------------------------

if (
    len(preference_scores_df)
    != len(preference_df)
):

    raise ValueError(
        "Destination count changed while calculating scores."
    )


if (
    preference_scores_df["destination"].nunique()
    != len(preference_scores_df)
):

    raise ValueError(
        "Duplicate destinations found."
    )


# ---------------------------------------------------------
# Verify that every non-missing score is between 0 and 1.
# ---------------------------------------------------------

for column in score_columns:

    valid_values = (
        preference_scores_df[column]
        .dropna()
    )

    if (
        len(valid_values) > 0
        and (
            valid_values.min() < 0
            or valid_values.max() > 1
        )
    ):

        raise ValueError(
            f"{column} contains values outside [0, 1]."
        )


print("\n" + "=" * 60)
print("✓ DESTINATION PREFERENCE SCORES CREATED")
print("=" * 60)

DESTINATION PREFERENCE SCORES

Score columns:
  - budget_score
  - flight_score
  - accommodation_score
  - weather_score
  - destination_characteristics_score

Shape:
(50, 6)

Missing group scores:
budget_score                         11
flight_score                         42
accommodation_score                  11
weather_score                         0
destination_characteristics_score     0
dtype: int64

Score ranges:
budget_score                        0.3001 → 0.9987
flight_score                        0.0278 → 0.8727
accommodation_score                 0.2229 → 0.8293
weather_score                       0.3305 → 0.8596
destination_characteristics_score   0.0019 → 0.5116

First 10 destination scores:


,destination,budget_score,flight_score,accommodation_score,weather_score,destination_characteristics_score
0,Agra,0.998750,NaN,0.586316,0.588232,0.251790
1,Ahmedabad,0.900347,NaN,0.601425,0.638803,0.133435
2,Alappuzha,0.748628,NaN,0.424060,0.526911,0.269133
3,Amritsar,0.889352,NaN,0.503944,0.859643,0.125003
4,Andaman,NaN,NaN,NaN,0.447488,0.001874
5,Bengaluru,0.927031,0.87274,0.729733,0.562324,0.402326
6,Bhopal,0.950233,NaN,0.517501,0.330498,0.172123
7,Bhubaneswar,0.923648,NaN,0.542160,0.490611,0.165999
8,Chennai,0.935104,NaN,0.702941,0.663475,0.406162
9,Coorg,NaN,NaN,NaN,0.496994,0.006536



✓ DESTINATION PREFERENCE SCORES CREATED


In [28]:
# ---------------------------------------------------------
# CREATE AVAILABILITY-AWARE PREFERENCE WEIGHTS
#
# Some destinations do not have flight or accommodation
# data. We therefore cannot treat missing values as poor
# scores.
#
# Instead:
#
#   1. Determine which preference groups are available.
#   2. Use the user's importance weight only when that
#      group's data exists.
#   3. Later normalize the active weights so that the
#      available preferences still form a valid score.
#
# Example:
#
# If a destination has no flight data:
#
#   Budget          0.9
#   Flight          NaN
#   Accommodation   0.6
#   Weather         0.8
#   Characteristics 0.8
#
# The flight weight is excluded for that destination rather
# than treating flight performance as zero.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Create a dataframe containing the destination names.
# ---------------------------------------------------------

availability_weights_df = pd.DataFrame()

availability_weights_df["destination"] = (
    preference_scores_df["destination"]
)


# ---------------------------------------------------------
# For every preference group, determine whether a valid
# score exists for each destination.
#
# 1 = group data available
# 0 = group data unavailable
# ---------------------------------------------------------

for group in preference_groups:

    score_column = f"{group}_score"

    availability_weights_df[
        f"{group}_available"
    ] = (
        preference_scores_df[score_column]
        .notna()
        .astype(int)
    )


# ---------------------------------------------------------
# Apply the user's preference importance weights.
#
# A group contributes its user weight only when its data
# is available.
# ---------------------------------------------------------

for group in preference_groups:

    availability_column = (
        f"{group}_available"
    )

    weighted_column = (
        f"{group}_weighted"
    )

    availability_weights_df[
        weighted_column
    ] = (
        availability_weights_df[
            availability_column
        ]
        * sample_user_preferences[group]
    )


# ---------------------------------------------------------
# Calculate the total active preference weight for every
# destination.
#
# This is important because different destinations have
# different amounts of external data available.
# ---------------------------------------------------------

weighted_columns = [
    f"{group}_weighted"
    for group in preference_groups
]


availability_weights_df[
    "total_active_weight"
] = (
    availability_weights_df[
        weighted_columns
    ]
    .sum(axis=1)
)


# ---------------------------------------------------------
# Display availability information.
# ---------------------------------------------------------

print("=" * 60)
print("AVAILABILITY-AWARE PREFERENCE WEIGHTS")
print("=" * 60)

print("\nAvailability counts:")

for group in preference_groups:

    column = f"{group}_available"

    print(
        f"{group:<35}"
        f"{availability_weights_df[column].sum()} "
        f"/ {len(availability_weights_df)} available"
    )


# ---------------------------------------------------------
# Check whether any destination has no active preference
# weight at all.
# ---------------------------------------------------------

zero_weight_destinations = (
    availability_weights_df[
        availability_weights_df["total_active_weight"] == 0
    ]["destination"]
    .tolist()
)


print("\nDestinations with zero active preference weight:")

print(
    zero_weight_destinations
)


# ---------------------------------------------------------
# Display the first 10 rows of the availability structure.
# ---------------------------------------------------------

display_columns = [
    "destination",
    "budget_available",
    "flight_available",
    "accommodation_available",
    "weather_available",
    "destination_characteristics_available",
    "total_active_weight"
]


print("\nFirst 10 availability records:")

display(
    availability_weights_df[
        display_columns
    ].head(10)
)


# ---------------------------------------------------------
# Validate that every destination has at least one usable
# preference group.
# ---------------------------------------------------------

if zero_weight_destinations:

    raise ValueError(
        "Some destinations have no usable preference data."
    )


# ---------------------------------------------------------
# Validate active weights.
# ---------------------------------------------------------

if (
    availability_weights_df[
        "total_active_weight"
    ].min()
    <= 0
):

    raise ValueError(
        "Invalid active preference weight detected."
    )


print("\n" + "=" * 60)
print("✓ AVAILABILITY-AWARE WEIGHTS CREATED")
print("=" * 60)

AVAILABILITY-AWARE PREFERENCE WEIGHTS

Availability counts:
budget                             39 / 50 available
flight                             8 / 50 available
accommodation                      39 / 50 available
weather                            50 / 50 available
destination_characteristics        50 / 50 available

Destinations with zero active preference weight:
[]

First 10 availability records:


,destination,budget_available,flight_available,accommodation_available,weather_available,destination_characteristics_available,total_active_weight
0,Agra,1,0,1,1,1,3.1
1,Ahmedabad,1,0,1,1,1,3.1
2,Alappuzha,1,0,1,1,1,3.1
3,Amritsar,1,0,1,1,1,3.1
4,Andaman,0,0,0,1,1,1.6
5,Bengaluru,1,1,1,1,1,3.7
6,Bhopal,1,0,1,1,1,3.1
7,Bhubaneswar,1,0,1,1,1,3.1
8,Chennai,1,0,1,1,1,3.1
9,Coorg,0,0,0,1,1,1.6



✓ AVAILABILITY-AWARE WEIGHTS CREATED


In [29]:
# ---------------------------------------------------------
# CALCULATE PERSONALIZED PREFERENCE SCORE
#
# For every destination:
#
#     Personalized Score =
#     Σ(group_score × user_weight)
#     --------------------------------
#        Σ(active user weights)
#
# Missing groups are excluded from both the numerator
# and denominator.
#
# This prevents destinations from being unfairly penalized
# simply because external data was unavailable.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Start with destination names.
# ---------------------------------------------------------

personalized_scores_df = pd.DataFrame()

personalized_scores_df["destination"] = (
    preference_scores_df["destination"]
)


# ---------------------------------------------------------
# Calculate the weighted contribution of every preference
# group.
# ---------------------------------------------------------

weighted_score_columns = []

for group in preference_groups:

    score_column = f"{group}_score"

    weight = sample_user_preferences[group]

    weighted_column = f"{group}_weighted_score"

    # Multiply the destination's group score by the
    # importance assigned by the user.
    personalized_scores_df[
        weighted_column
    ] = (
        preference_scores_df[score_column]
        .fillna(0)
        * weight
    )

    weighted_score_columns.append(
        weighted_column
    )


# ---------------------------------------------------------
# Calculate the numerator.
# ---------------------------------------------------------

personalized_scores_df[
    "weighted_score_sum"
] = (
    personalized_scores_df[
        weighted_score_columns
    ]
    .sum(axis=1)
)


# ---------------------------------------------------------
# Bring in the active preference weights calculated in the
# previous step.
# ---------------------------------------------------------

personalized_scores_df[
    "total_active_weight"
] = (
    availability_weights_df[
        "total_active_weight"
    ]
)


# ---------------------------------------------------------
# Normalize the weighted score.
#
# This produces a final preference score between 0 and 1.
# ---------------------------------------------------------

personalized_scores_df[
    "personalized_preference_score"
] = (
    personalized_scores_df[
        "weighted_score_sum"
    ]
    /
    personalized_scores_df[
        "total_active_weight"
    ]
)


# ---------------------------------------------------------
# Check the resulting score range.
# ---------------------------------------------------------

minimum_score = (
    personalized_scores_df[
        "personalized_preference_score"
    ].min()
)

maximum_score = (
    personalized_scores_df[
        "personalized_preference_score"
    ].max()
)


print("=" * 60)
print("PERSONALIZED PREFERENCE SCORES")
print("=" * 60)

print(
    f"\nMinimum score: {minimum_score:.6f}"
)

print(
    f"Maximum score: {maximum_score:.6f}"
)


# ---------------------------------------------------------
# Display the top destinations according to the user's
# current preferences.
# ---------------------------------------------------------

top_preference_destinations = (
    personalized_scores_df[
        [
            "destination",
            "personalized_preference_score"
        ]
    ]
    .sort_values(
        "personalized_preference_score",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)


print("\nTop 10 destinations by personalized preference:")

display(
    top_preference_destinations
)


# ---------------------------------------------------------
# Check for missing personalized scores.
# ---------------------------------------------------------

missing_scores = (
    personalized_scores_df[
        "personalized_preference_score"
    ]
    .isna()
    .sum()
)


print(
    "\nMissing personalized scores:",
    missing_scores
)


# ---------------------------------------------------------
# Validate that all scores are within [0, 1].
# ---------------------------------------------------------

if (
    personalized_scores_df[
        "personalized_preference_score"
    ].min() < 0
    or
    personalized_scores_df[
        "personalized_preference_score"
    ].max() > 1
):

    raise ValueError(
        "Personalized preference scores are outside [0, 1]."
    )


if missing_scores > 0:

    raise ValueError(
        "Missing personalized preference scores detected."
    )


# ---------------------------------------------------------
# Validate destination count.
# ---------------------------------------------------------

if (
    len(personalized_scores_df)
    != len(preference_scores_df)
):

    raise ValueError(
        "Destination count changed during scoring."
    )


print("\n" + "=" * 60)
print("✓ PERSONALIZED PREFERENCE SCORES CALCULATED")
print("=" * 60)

PERSONALIZED PREFERENCE SCORES

Minimum score: 0.224681
Maximum score: 0.726705

Top 10 destinations by personalized preference:


,destination,personalized_preference_score
0,Mumbai,0.726705
1,Bengaluru,0.693928
2,Chennai,0.683570
3,Delhi,0.675995
4,Kolkata,0.643378
5,Pune,0.642439
6,Hyderabad,0.636306
7,Goa,0.629620
8,Jodhpur,0.621057
9,Agra,0.620220



Missing personalized scores: 0

✓ PERSONALIZED PREFERENCE SCORES CALCULATED


In [31]:
# ---------------------------------------------------------
# CALCULATE PERSONALIZED INTEREST SCORE
#
# Uses the existing user interest preferences stored in:
#
#     sample_user_interests
#
# Destination interest profiles:
#
#     nature_score
#     sightseeing_score
#     water_coastal_score
#     wildlife_score
#
# The score is calculated as a weighted average of the
# destination's interest-profile scores.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Interest profiles used by the recommendation system.
# ---------------------------------------------------------

interest_profiles = [
    "nature",
    "sightseeing",
    "water_coastal",
    "wildlife"
]


# ---------------------------------------------------------
# Create the dataframe that will contain the personalized
# interest score for every destination.
# ---------------------------------------------------------

interest_scores_df = pd.DataFrame()

interest_scores_df["destination"] = (
    profile_scores_df["destination"]
)


# ---------------------------------------------------------
# Calculate the weighted contribution of every interest.
#
# Example:
#
# nature_score × user's nature preference
#
# sightseeing_score × user's sightseeing preference
#
# etc.
# ---------------------------------------------------------

interest_weighted_columns = []

for profile in interest_profiles:

    # Destination profile score column.
    profile_column = f"{profile}_score"

    # Column that will store the weighted contribution.
    weighted_column = (
        f"{profile}_weighted_score"
    )

    # IMPORTANT:
    # Use the variable name that already exists in our
    # notebook: sample_user_interests
    user_weight = sample_user_interests[
        profile
    ]

    # Calculate the weighted contribution.
    interest_scores_df[
        weighted_column
    ] = (
        profile_scores_df[
            profile_column
        ]
        * user_weight
    )

    interest_weighted_columns.append(
        weighted_column
    )


# ---------------------------------------------------------
# Add all weighted interest contributions together.
# ---------------------------------------------------------

interest_scores_df[
    "weighted_interest_score_sum"
] = (
    interest_scores_df[
        interest_weighted_columns
    ]
    .sum(axis=1)
)


# ---------------------------------------------------------
# Calculate the total importance assigned to all interests.
#
# Current user:
#
# nature          = 1.0
# sightseeing     = 0.5
# water_coastal   = 0.2
# wildlife        = 0.6
#
# Total = 2.3
# ---------------------------------------------------------

total_interest_weight = sum(
    sample_user_interests.values()
)


# ---------------------------------------------------------
# Convert the weighted sum into a normalized score between
# 0 and 1.
# ---------------------------------------------------------

interest_scores_df[
    "personalized_interest_score"
] = (
    interest_scores_df[
        "weighted_interest_score_sum"
    ]
    / total_interest_weight
)


# ---------------------------------------------------------
# Display the results.
# ---------------------------------------------------------

print("=" * 60)
print("PERSONALIZED INTEREST SCORES")
print("=" * 60)

print(
    f"\nTotal interest weight: "
    f"{total_interest_weight:.2f}"
)

print(
    f"Minimum score: "
    f"{interest_scores_df['personalized_interest_score'].min():.6f}"
)

print(
    f"Maximum score: "
    f"{interest_scores_df['personalized_interest_score'].max():.6f}"
)


# ---------------------------------------------------------
# Display the top destinations according to the user's
# destination interests.
# ---------------------------------------------------------

top_interest_destinations = (
    interest_scores_df[
        [
            "destination",
            "personalized_interest_score"
        ]
    ]
    .sort_values(
        "personalized_interest_score",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)


print("\nTop 10 destinations by personalized interests:")

display(
    top_interest_destinations
)


# ---------------------------------------------------------
# Check for missing scores.
# ---------------------------------------------------------

missing_interest_scores = (
    interest_scores_df[
        "personalized_interest_score"
    ]
    .isna()
    .sum()
)


print(
    "\nMissing interest scores:",
    missing_interest_scores
)


# ---------------------------------------------------------
# Validate the score range.
# ---------------------------------------------------------

if (
    interest_scores_df[
        "personalized_interest_score"
    ].min() < 0
    or
    interest_scores_df[
        "personalized_interest_score"
    ].max() > 1
):

    raise ValueError(
        "Personalized interest scores are outside [0, 1]."
    )


if missing_interest_scores > 0:

    raise ValueError(
        "Missing personalized interest scores detected."
    )


# ---------------------------------------------------------
# Validate destination alignment.
# ---------------------------------------------------------

if (
    len(interest_scores_df)
    != len(profile_scores_df)
):

    raise ValueError(
        "Destination count changed while calculating "
        "interest scores."
    )


if (
    interest_scores_df["destination"].nunique()
    != len(interest_scores_df)
):

    raise ValueError(
        "Duplicate destinations found in interest scores."
    )


print("\n" + "=" * 60)
print("✓ PERSONALIZED INTEREST SCORES CALCULATED")
print("=" * 60)

PERSONALIZED INTEREST SCORES

Total interest weight: 2.30
Minimum score: 0.003586
Maximum score: 0.522352

Top 10 destinations by personalized interests:


,destination,personalized_interest_score
0,Mumbai,0.522352
1,Manali,0.487155
2,Jaipur,0.410318
3,Delhi,0.408991
4,Bengaluru,0.401584
5,Kochi,0.397712
6,Kolkata,0.368054
7,Srinagar,0.349341
8,Chennai,0.343195
9,Pune,0.342008



Missing interest scores: 0

✓ PERSONALIZED INTEREST SCORES CALCULATED


In [32]:
# ---------------------------------------------------------
# COMBINE TRAVEL-FACTOR AND DESTINATION-INTEREST SCORES
#
# We now have two independent personalized signals:
#
# 1. personalized_preference_score
#    → Budget, flight, accommodation, weather and
#      destination characteristics.
#
# 2. personalized_interest_score
#    → Nature, sightseeing, water/coastal and wildlife.
#
# We combine them into one base personalized score.
#
# Current weighting:
#
#     Travel factors = 60%
#     Interests       = 40%
#
# Both component scores are already normalized to [0, 1].
# Therefore the resulting score also remains within [0, 1].
# ---------------------------------------------------------


# ---------------------------------------------------------
# Define the relative importance of the two personalized
# scoring components.
# ---------------------------------------------------------

TRAVEL_FACTOR_WEIGHT = 0.60
INTEREST_WEIGHT = 0.40


# ---------------------------------------------------------
# Validate that the weights form a valid combination.
# ---------------------------------------------------------

if not (
    0 <= TRAVEL_FACTOR_WEIGHT <= 1
    and
    0 <= INTEREST_WEIGHT <= 1
):

    raise ValueError(
        "Combination weights must be between 0 and 1."
    )


if (
    abs(
        TRAVEL_FACTOR_WEIGHT
        + INTEREST_WEIGHT
        - 1.0
    ) > 1e-9
):

    raise ValueError(
        "Combination weights must sum to 1."
    )


# ---------------------------------------------------------
# Create the combined dataframe.
# ---------------------------------------------------------

combined_preference_scores_df = pd.DataFrame()

combined_preference_scores_df[
    "destination"
] = personalized_scores_df[
    "destination"
]


# ---------------------------------------------------------
# Add the two individual scores so that we can inspect
# them later.
# ---------------------------------------------------------

combined_preference_scores_df[
    "personalized_preference_score"
] = (
    personalized_scores_df[
        "personalized_preference_score"
    ]
)

combined_preference_scores_df[
    "personalized_interest_score"
] = (
    interest_scores_df[
        "personalized_interest_score"
    ]
)


# ---------------------------------------------------------
# Calculate the final base personalized score.
# ---------------------------------------------------------

combined_preference_scores_df[
    "base_personalized_score"
] = (

    TRAVEL_FACTOR_WEIGHT
    * combined_preference_scores_df[
        "personalized_preference_score"
    ]

    +

    INTEREST_WEIGHT
    * combined_preference_scores_df[
        "personalized_interest_score"
    ]
)


# ---------------------------------------------------------
# Display the score ranges.
# ---------------------------------------------------------

print("=" * 60)
print("COMBINED PERSONALIZED PREFERENCE SCORE")
print("=" * 60)

print(
    f"\nTravel-factor weight: "
    f"{TRAVEL_FACTOR_WEIGHT:.2f}"
)

print(
    f"Interest weight: "
    f"{INTEREST_WEIGHT:.2f}"
)

print(
    f"\nMinimum base personalized score: "
    f"{combined_preference_scores_df['base_personalized_score'].min():.6f}"
)

print(
    f"Maximum base personalized score: "
    f"{combined_preference_scores_df['base_personalized_score'].max():.6f}"
)


# ---------------------------------------------------------
# Rank destinations using the combined personalized score.
# ---------------------------------------------------------

top_combined_destinations = (
    combined_preference_scores_df[
        [
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "base_personalized_score"
        ]
    ]
    .sort_values(
        "base_personalized_score",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)


print(
    "\nTop 10 destinations by combined personalized score:"
)

display(
    top_combined_destinations
)


# ---------------------------------------------------------
# Check for missing scores.
# ---------------------------------------------------------

missing_combined_scores = (
    combined_preference_scores_df[
        "base_personalized_score"
    ]
    .isna()
    .sum()
)


print(
    "\nMissing combined scores:",
    missing_combined_scores
)


# ---------------------------------------------------------
# Validate that all scores remain within [0, 1].
# ---------------------------------------------------------

if (
    combined_preference_scores_df[
        "base_personalized_score"
    ].min() < 0
    or
    combined_preference_scores_df[
        "base_personalized_score"
    ].max() > 1
):

    raise ValueError(
        "Combined personalized scores are outside [0, 1]."
    )


if missing_combined_scores > 0:

    raise ValueError(
        "Missing combined personalized scores detected."
    )


# ---------------------------------------------------------
# Validate destination count and uniqueness.
# ---------------------------------------------------------

if (
    len(combined_preference_scores_df)
    != 50
):

    raise ValueError(
        "Unexpected number of destinations."
    )


if (
    combined_preference_scores_df[
        "destination"
    ].nunique()
    != 50
):

    raise ValueError(
        "Duplicate destinations detected."
    )


print("\n" + "=" * 60)
print("✓ COMBINED PERSONALIZED SCORE CREATED")
print("=" * 60)

COMBINED PERSONALIZED PREFERENCE SCORE

Travel-factor weight: 0.60
Interest weight: 0.40

Minimum base personalized score: 0.136243
Maximum base personalized score: 0.644964

Top 10 destinations by combined personalized score:


,destination,personalized_preference_score,personalized_interest_score,base_personalized_score
0,Mumbai,0.726705,0.522352,0.644964
1,Bengaluru,0.693928,0.401584,0.576990
2,Delhi,0.675995,0.408991,0.569193
3,Manali,0.616984,0.487155,0.565053
4,Chennai,0.683570,0.343195,0.547420
5,Jaipur,0.616046,0.410318,0.533755
6,Kolkata,0.643378,0.368054,0.533248
7,Pune,0.642439,0.342008,0.522267
8,Kochi,0.601073,0.397712,0.519728
9,Thiruvananthapuram,0.618037,0.315918,0.497190



Missing combined scores: 0

✓ COMBINED PERSONALIZED SCORE CREATED


In [33]:
# ---------------------------------------------------------
# HYBRID RECOMMENDATION ENGINE
#
# The recommendation engine supports three modes:
#
# 1. Preference-only
#    Uses the personalized preference score.
#
# 2. Similarity-only
#    Finds destinations similar to a reference destination.
#
# 3. Hybrid
#    Combines personalized preferences with similarity.
#
# Hybrid formula:
#
#     Final Score =
#         preference_weight × personalized_score
#       + similarity_weight × similarity_score
#
# Similarity is converted from [-1, 1] to [0, 1] before
# combining it with the personalized score.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Default weights for the hybrid model.
#
# Personalized preferences receive slightly more importance
# because the recommendation should primarily reflect what
# the user wants.
# ---------------------------------------------------------

DEFAULT_PREFERENCE_WEIGHT = 0.70
DEFAULT_SIMILARITY_WEIGHT = 0.30


# ---------------------------------------------------------
# Validate the recommendation dataset.
# ---------------------------------------------------------

required_score_columns = [
    "destination",
    "base_personalized_score"
]

missing_score_columns = [
    column
    for column in required_score_columns
    if column not in combined_preference_scores_df.columns
]

if missing_score_columns:

    raise ValueError(
        "Missing required recommendation score columns: "
        + str(missing_score_columns)
    )


# ---------------------------------------------------------
# Identify the similarity matrix.
#
# The matrix created earlier should contain:
#
#     rows    → destinations
#     columns → destinations
#
# We first check the expected variable name.
# ---------------------------------------------------------

if "destination_similarity" in globals():

    similarity_matrix = destination_similarity

elif "similarity_matrix" in globals():

    similarity_matrix = similarity_matrix

else:

    raise NameError(
        "Destination similarity matrix was not found. "
        "Expected 'destination_similarity' or "
        "'similarity_matrix'."
    )


# ---------------------------------------------------------
# Validate similarity matrix dimensions.
# ---------------------------------------------------------

if (
    similarity_matrix.shape[0]
    != similarity_matrix.shape[1]
):

    raise ValueError(
        "Similarity matrix must be square."
    )


if (
    similarity_matrix.shape[0]
    != len(combined_preference_scores_df)
):

    raise ValueError(
        "Similarity matrix destination count does not "
        "match the personalized score dataset."
    )


# ---------------------------------------------------------
# Convert the similarity matrix into a DataFrame if it is
# currently stored as a NumPy array.
#
# This makes destination lookup safer and easier.
# ---------------------------------------------------------

if not isinstance(
    similarity_matrix,
    pd.DataFrame
):

    similarity_matrix = pd.DataFrame(
        similarity_matrix,
        index=combined_preference_scores_df[
            "destination"
        ],
        columns=combined_preference_scores_df[
            "destination"
        ]
    )


# ---------------------------------------------------------
# Make sure the matrix contains destination labels.
# ---------------------------------------------------------

destination_names = (
    combined_preference_scores_df[
        "destination"
    ]
    .tolist()
)


missing_similarity_destinations = [
    destination
    for destination in destination_names
    if destination not in similarity_matrix.index
    or destination not in similarity_matrix.columns
]


if missing_similarity_destinations:

    raise ValueError(
        "Destinations missing from similarity matrix: "
        + str(missing_similarity_destinations)
    )


# ---------------------------------------------------------
# Recommendation function
#
# reference_destination:
#     None → preference-only recommendation
#
#     "Goa" → similarity/hybrid recommendation relative
#              to Goa
#
# mode:
#     "preference"
#     "similarity"
#     "hybrid"
# ---------------------------------------------------------

def generate_recommendations(
    reference_destination=None,
    mode="hybrid",
    top_n=10,
    preference_weight=DEFAULT_PREFERENCE_WEIGHT,
    similarity_weight=DEFAULT_SIMILARITY_WEIGHT
):

    # -----------------------------------------------------
    # Validate recommendation mode.
    # -----------------------------------------------------

    valid_modes = [
        "preference",
        "similarity",
        "hybrid"
    ]

    if mode not in valid_modes:

        raise ValueError(
            f"Invalid mode '{mode}'. "
            f"Choose from {valid_modes}."
        )


    # -----------------------------------------------------
    # Validate top_n.
    # -----------------------------------------------------

    if top_n <= 0:

        raise ValueError(
            "top_n must be greater than zero."
        )


    # -----------------------------------------------------
    # Validate the combination weights.
    # -----------------------------------------------------

    if (
        preference_weight < 0
        or similarity_weight < 0
    ):

        raise ValueError(
            "Recommendation weights cannot be negative."
        )


    if (
        abs(
            preference_weight
            + similarity_weight
            - 1.0
        ) > 1e-9
    ):

        raise ValueError(
            "Preference and similarity weights must sum to 1."
        )


    # -----------------------------------------------------
    # Create the recommendation dataframe.
    # -----------------------------------------------------

    recommendations = (
        combined_preference_scores_df[
            [
                "destination",
                "personalized_preference_score",
                "personalized_interest_score",
                "base_personalized_score"
            ]
        ]
        .copy()
    )


    # -----------------------------------------------------
    # Preference-only mode.
    # -----------------------------------------------------

    if mode == "preference":

        recommendations[
            "final_recommendation_score"
        ] = (
            recommendations[
                "base_personalized_score"
            ]
        )


    # -----------------------------------------------------
    # Similarity or hybrid mode requires a reference
    # destination.
    # -----------------------------------------------------

    elif reference_destination is None:

        raise ValueError(
            "A reference destination is required for "
            "similarity and hybrid modes."
        )


    else:

        # -------------------------------------------------
        # Validate the reference destination.
        # -------------------------------------------------

        if reference_destination not in similarity_matrix.index:

            raise ValueError(
                f"Destination '{reference_destination}' "
                "was not found in the similarity matrix."
            )


        # -------------------------------------------------
        # Extract similarity scores relative to the
        # reference destination.
        # -------------------------------------------------

        recommendations[
            "raw_similarity_score"
        ] = [
            similarity_matrix.loc[
                reference_destination,
                destination
            ]
            for destination in recommendations[
                "destination"
            ]
        ]


        # -------------------------------------------------
        # Convert cosine similarity from [-1, 1] into
        # [0, 1].
        #
        # This puts similarity on a comparable scale with
        # our personalized score.
        # -------------------------------------------------

        recommendations[
            "similarity_score"
        ] = (
            recommendations[
                "raw_similarity_score"
            ]
            + 1
        ) / 2


        # -------------------------------------------------
        # Similarity-only mode.
        # -------------------------------------------------

        if mode == "similarity":

            recommendations[
                "final_recommendation_score"
            ] = (
                recommendations[
                    "similarity_score"
                ]
            )


        # -------------------------------------------------
        # Hybrid mode.
        # -------------------------------------------------

        else:

            recommendations[
                "final_recommendation_score"
            ] = (

                preference_weight
                * recommendations[
                    "base_personalized_score"
                ]

                +

                similarity_weight
                * recommendations[
                    "similarity_score"
                ]
            )


    # -----------------------------------------------------
    # Remove the reference destination itself.
    #
    # A destination should not recommend itself.
    # -----------------------------------------------------

    if reference_destination is not None:

        recommendations = (
            recommendations[
                recommendations[
                    "destination"
                ]
                != reference_destination
            ]
        )


    # -----------------------------------------------------
    # Sort by final recommendation score.
    # -----------------------------------------------------

    recommendations = (
        recommendations
        .sort_values(
            "final_recommendation_score",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )


    # -----------------------------------------------------
    # Return the ranked recommendations.
    # -----------------------------------------------------

    return recommendations


print("=" * 60)
print("✓ HYBRID RECOMMENDATION ENGINE CREATED")
print("=" * 60)

print("\nSupported modes:")
print("  1. preference")
print("  2. similarity")
print("  3. hybrid")

print("\nDefault hybrid weights:")
print(
    f"  Personalized preference: "
    f"{DEFAULT_PREFERENCE_WEIGHT:.2f}"
)

print(
    f"  Similarity: "
    f"{DEFAULT_SIMILARITY_WEIGHT:.2f}"
)

✓ HYBRID RECOMMENDATION ENGINE CREATED

Supported modes:
  1. preference
  2. similarity
  3. hybrid

Default hybrid weights:
  Personalized preference: 0.70
  Similarity: 0.30


In [34]:
# ---------------------------------------------------------
# TEST 1: PREFERENCE-ONLY RECOMMENDATIONS
#
# This mode uses only the user's personalized preferences.
# No reference destination is required.
# ---------------------------------------------------------

preference_recommendations = generate_recommendations(
    mode="preference",
    top_n=10
)

print("=" * 60)
print("PREFERENCE-ONLY RECOMMENDATIONS")
print("=" * 60)

display(
    preference_recommendations[
        [
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "base_personalized_score",
            "final_recommendation_score"
        ]
    ]
)


# ---------------------------------------------------------
# TEST 2: SIMILARITY-ONLY RECOMMENDATIONS
#
# We use Goa as the reference destination.
#
# The result should NOT contain Goa itself because a
# destination should not recommend itself.
# ---------------------------------------------------------

similarity_recommendations = generate_recommendations(
    reference_destination="Goa",
    mode="similarity",
    top_n=10
)

print("\n" + "=" * 60)
print("SIMILARITY-ONLY RECOMMENDATIONS FOR GOA")
print("=" * 60)

display(
    similarity_recommendations[
        [
            "destination",
            "raw_similarity_score",
            "similarity_score",
            "final_recommendation_score"
        ]
    ]
)


# ---------------------------------------------------------
# TEST 3: HYBRID RECOMMENDATIONS
#
# This combines:
#
#     70% personalized preference
#     30% similarity to Goa
#
# This is the most important test because it represents
# the combined recommendation behaviour.
# ---------------------------------------------------------

hybrid_recommendations = generate_recommendations(
    reference_destination="Goa",
    mode="hybrid",
    top_n=10
)

print("\n" + "=" * 60)
print("HYBRID RECOMMENDATIONS FOR GOA")
print("=" * 60)

display(
    hybrid_recommendations[
        [
            "destination",
            "base_personalized_score",
            "similarity_score",
            "final_recommendation_score"
        ]
    ]
)


# ---------------------------------------------------------
# BASIC VALIDATION
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("RECOMMENDATION ENGINE VALIDATION")
print("=" * 60)


# Preference mode should return the requested number of
# destinations.

if len(preference_recommendations) != 10:

    raise ValueError(
        "Preference recommendation count is incorrect."
    )


# Similarity mode should return the requested number.

if len(similarity_recommendations) != 10:

    raise ValueError(
        "Similarity recommendation count is incorrect."
    )


# Hybrid mode should return the requested number.

if len(hybrid_recommendations) != 10:

    raise ValueError(
        "Hybrid recommendation count is incorrect."
    )


# Goa must not recommend itself.

if "Goa" in similarity_recommendations[
    "destination"
].values:

    raise ValueError(
        "Reference destination appeared in similarity results."
    )


if "Goa" in hybrid_recommendations[
    "destination"
].values:

    raise ValueError(
        "Reference destination appeared in hybrid results."
    )


# Final scores must be sorted from highest to lowest.

if not preference_recommendations[
    "final_recommendation_score"
].is_monotonic_decreasing:

    raise ValueError(
        "Preference recommendations are not correctly sorted."
    )


if not similarity_recommendations[
    "final_recommendation_score"
].is_monotonic_decreasing:

    raise ValueError(
        "Similarity recommendations are not correctly sorted."
    )


if not hybrid_recommendations[
    "final_recommendation_score"
].is_monotonic_decreasing:

    raise ValueError(
        "Hybrid recommendations are not correctly sorted."
    )


# Final recommendation scores should remain within [0, 1].

for result_df in [
    preference_recommendations,
    similarity_recommendations,
    hybrid_recommendations
]:

    if (
        result_df[
            "final_recommendation_score"
        ].min() < 0
        or
        result_df[
            "final_recommendation_score"
        ].max() > 1
    ):

        raise ValueError(
            "Recommendation score outside [0, 1]."
        )


print("\n✓ Preference mode validated")
print("✓ Similarity mode validated")
print("✓ Hybrid mode validated")
print("✓ Reference destination exclusion validated")
print("✓ Ranking order validated")
print("✓ Score range validated")

print("\n" + "=" * 60)
print("✓ ALL RECOMMENDATION ENGINE TESTS PASSED")
print("=" * 60)

PREFERENCE-ONLY RECOMMENDATIONS


,destination,personalized_preference_score,personalized_interest_score,base_personalized_score,final_recommendation_score
0,Mumbai,0.726705,0.522352,0.644964,0.644964
1,Bengaluru,0.693928,0.401584,0.576990,0.576990
2,Delhi,0.675995,0.408991,0.569193,0.569193
3,Manali,0.616984,0.487155,0.565053,0.565053
4,Chennai,0.683570,0.343195,0.547420,0.547420
5,Jaipur,0.616046,0.410318,0.533755,0.533755
6,Kolkata,0.643378,0.368054,0.533248,0.533248
7,Pune,0.642439,0.342008,0.522267,0.522267
8,Kochi,0.601073,0.397712,0.519728,0.519728
9,Thiruvananthapuram,0.618037,0.315918,0.497190,0.497190



SIMILARITY-ONLY RECOMMENDATIONS FOR GOA


,destination,raw_similarity_score,similarity_score,final_recommendation_score
0,Mumbai,0.586488,0.793244,0.793244
1,Alappuzha,0.355314,0.677657,0.677657
2,Kaziranga,0.351321,0.675661,0.675661
3,Coorg,0.225543,0.612772,0.612772
4,Bengaluru,0.209090,0.604545,0.604545
5,Delhi,0.201240,0.600620,0.600620
6,Gokarna,0.195864,0.597932,0.597932
7,Jaipur,0.172510,0.586255,0.586255
8,Kodaikanal,0.112580,0.556290,0.556290
9,Munnar,0.102729,0.551365,0.551365



HYBRID RECOMMENDATIONS FOR GOA


,destination,base_personalized_score,similarity_score,final_recommendation_score
0,Mumbai,0.644964,0.793244,0.689448
1,Bengaluru,0.576990,0.604545,0.585257
2,Delhi,0.569193,0.600620,0.578621
3,Jaipur,0.533755,0.586255,0.549505
4,Manali,0.565053,0.449051,0.530252
5,Pune,0.522267,0.511279,0.518970
6,Chennai,0.547420,0.436365,0.514104
7,Hyderabad,0.489115,0.533714,0.502495
8,Kochi,0.519728,0.437480,0.495054
9,Gokarna,0.448397,0.597932,0.493258



RECOMMENDATION ENGINE VALIDATION

✓ Preference mode validated
✓ Similarity mode validated
✓ Hybrid mode validated
✓ Reference destination exclusion validated
✓ Ranking order validated
✓ Score range validated

✓ ALL RECOMMENDATION ENGINE TESTS PASSED


In [35]:
# ---------------------------------------------------------
# DYNAMIC PERSONALIZED RECOMMENDATION ENGINE
#
# This version does NOT use the hard-coded sample user.
#
# Every time the function is called, it receives:
#
#   user_preferences
#   user_interests
#
# Therefore different users can receive different
# recommendations.
#
# Supported modes:
#
#   1. preference
#   2. similarity
#   3. hybrid
#
# ---------------------------------------------------------


def calculate_dynamic_preference_scores(
    user_preferences,
    user_interests
):

    # -----------------------------------------------------
    # Validate travel-factor preference keys.
    # -----------------------------------------------------

    expected_preference_groups = set(
        preference_groups.keys()
    )

    provided_preference_groups = set(
        user_preferences.keys()
    )

    missing_groups = (
        expected_preference_groups
        - provided_preference_groups
    )

    unexpected_groups = (
        provided_preference_groups
        - expected_preference_groups
    )

    if missing_groups:

        raise ValueError(
            "Missing preference groups: "
            + str(sorted(missing_groups))
        )

    if unexpected_groups:

        raise ValueError(
            "Unexpected preference groups: "
            + str(sorted(unexpected_groups))
        )


    # -----------------------------------------------------
    # Validate destination-interest preference keys.
    # -----------------------------------------------------

    expected_profiles = set(
        interest_profiles
    )

    provided_profiles = set(
        user_interests.keys()
    )

    missing_profiles = (
        expected_profiles
        - provided_profiles
    )

    unexpected_profiles = (
        provided_profiles
        - expected_profiles
    )

    if missing_profiles:

        raise ValueError(
            "Missing interest profiles: "
            + str(sorted(missing_profiles))
        )

    if unexpected_profiles:

        raise ValueError(
            "Unexpected interest profiles: "
            + str(sorted(unexpected_profiles))
        )


    # -----------------------------------------------------
    # Validate all user preference values.
    #
    # Every preference must be between 0 and 1.
    # -----------------------------------------------------

    invalid_preferences = {
        key: value
        for key, value in user_preferences.items()
        if not 0 <= value <= 1
    }

    invalid_interests = {
        key: value
        for key, value in user_interests.items()
        if not 0 <= value <= 1
    }

    if invalid_preferences:

        raise ValueError(
            "Preference values must be between 0 and 1: "
            + str(invalid_preferences)
        )

    if invalid_interests:

        raise ValueError(
            "Interest values must be between 0 and 1: "
            + str(invalid_interests)
        )


    # -----------------------------------------------------
    # Create the destination score dataframe.
    # -----------------------------------------------------

    dynamic_scores_df = pd.DataFrame()

    dynamic_scores_df["destination"] = (
        preference_df["destination"]
    )


    # -----------------------------------------------------
    # Calculate each travel-factor score dynamically.
    #
    # Missing external data is ignored rather than treated
    # as a poor score.
    # -----------------------------------------------------

    for group, features in preference_groups.items():

        score_column = f"{group}_score"

        dynamic_scores_df[
            score_column
        ] = (
            direction_corrected_df[
                features
            ]
            .mean(
                axis=1,
                skipna=True
            )
        )


    # -----------------------------------------------------
    # Calculate availability-aware weighted preference
    # score for each destination.
    # -----------------------------------------------------

    weighted_preference_columns = []

    for group in preference_groups:

        score_column = f"{group}_score"

        weighted_column = (
            f"{group}_weighted"
        )

        user_weight = user_preferences[group]

        # A group contributes only when data is available.
        dynamic_scores_df[
            weighted_column
        ] = (
            dynamic_scores_df[
                score_column
            ]
            .fillna(0)
            * user_weight
        )

        weighted_preference_columns.append(
            weighted_column
        )


    # -----------------------------------------------------
    # Determine which groups are actually available for
    # each destination.
    #
    # This prevents missing flight/hotel data from being
    # interpreted as a zero-quality destination.
    # -----------------------------------------------------

    active_weight_columns = []

    for group in preference_groups:

        score_column = f"{group}_score"

        active_column = (
            f"{group}_active_weight"
        )

        user_weight = user_preferences[group]

        dynamic_scores_df[
            active_column
        ] = (
            dynamic_scores_df[
                score_column
            ]
            .notna()
            .astype(int)
            * user_weight
        )

        active_weight_columns.append(
            active_column
        )


    # -----------------------------------------------------
    # Calculate total active weight for every destination.
    # -----------------------------------------------------

    dynamic_scores_df[
        "total_active_weight"
    ] = (
        dynamic_scores_df[
            active_weight_columns
        ]
        .sum(axis=1)
    )


    # -----------------------------------------------------
    # Calculate the final travel-factor preference score.
    # -----------------------------------------------------

    dynamic_scores_df[
        "personalized_preference_score"
    ] = (
        dynamic_scores_df[
            weighted_preference_columns
        ]
        .sum(axis=1)
        /
        dynamic_scores_df[
            "total_active_weight"
        ]
    )


    # -----------------------------------------------------
    # Calculate the personalized destination-interest score.
    # -----------------------------------------------------

    interest_weighted_columns = []

    for profile in interest_profiles:

        profile_column = f"{profile}_score"

        weighted_column = (
            f"{profile}_weighted"
        )

        user_weight = user_interests[
            profile
        ]

        dynamic_scores_df[
            weighted_column
        ] = (
            profile_scores_df[
                profile_column
            ]
            * user_weight
        )

        interest_weighted_columns.append(
            weighted_column
        )


    # -----------------------------------------------------
    # Calculate total interest importance.
    # -----------------------------------------------------

    total_interest_weight = sum(
        user_interests.values()
    )

    if total_interest_weight <= 0:

        raise ValueError(
            "At least one destination interest must "
            "have a value greater than zero."
        )


    # -----------------------------------------------------
    # Calculate personalized interest score.
    # -----------------------------------------------------

    dynamic_scores_df[
        "personalized_interest_score"
    ] = (
        dynamic_scores_df[
            interest_weighted_columns
        ]
        .sum(axis=1)
        /
        total_interest_weight
    )


    # -----------------------------------------------------
    # Combine travel factors and interests.
    #
    # These are the same weights validated earlier:
    #
    #   60% travel factors
    #   40% destination interests
    # -----------------------------------------------------

    dynamic_scores_df[
        "base_personalized_score"
    ] = (

        TRAVEL_FACTOR_WEIGHT
        * dynamic_scores_df[
            "personalized_preference_score"
        ]

        +

        INTEREST_WEIGHT
        * dynamic_scores_df[
            "personalized_interest_score"
        ]
    )


    # -----------------------------------------------------
    # Return the dynamically calculated scores.
    # -----------------------------------------------------

    return dynamic_scores_df


# ---------------------------------------------------------
# Create the final dynamic recommendation function.
# ---------------------------------------------------------

def generate_dynamic_recommendations(
    user_preferences,
    user_interests,
    reference_destination=None,
    mode="hybrid",
    top_n=10,
    preference_weight=DEFAULT_PREFERENCE_WEIGHT,
    similarity_weight=DEFAULT_SIMILARITY_WEIGHT
):

    # -----------------------------------------------------
    # Validate recommendation mode.
    # -----------------------------------------------------

    valid_modes = [
        "preference",
        "similarity",
        "hybrid"
    ]

    if mode not in valid_modes:

        raise ValueError(
            f"Invalid mode '{mode}'. "
            f"Choose from {valid_modes}."
        )


    # -----------------------------------------------------
    # Validate top_n.
    # -----------------------------------------------------

    if top_n <= 0:

        raise ValueError(
            "top_n must be greater than zero."
        )


    # -----------------------------------------------------
    # Validate recommendation weights.
    # -----------------------------------------------------

    if (
        preference_weight < 0
        or similarity_weight < 0
    ):

        raise ValueError(
            "Recommendation weights cannot be negative."
        )


    if (
        abs(
            preference_weight
            + similarity_weight
            - 1.0
        ) > 1e-9
    ):

        raise ValueError(
            "Recommendation weights must sum to 1."
        )


    # -----------------------------------------------------
    # Calculate scores specifically for THIS user.
    # -----------------------------------------------------

    user_scores = (
        calculate_dynamic_preference_scores(
            user_preferences=user_preferences,
            user_interests=user_interests
        )
    )


    # -----------------------------------------------------
    # Create the recommendation dataframe.
    # -----------------------------------------------------

    recommendations = user_scores[
        [
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "base_personalized_score"
        ]
    ].copy()


    # -----------------------------------------------------
    # Preference-only mode.
    # -----------------------------------------------------

    if mode == "preference":

        recommendations[
            "final_recommendation_score"
        ] = (
            recommendations[
                "base_personalized_score"
            ]
        )


    # -----------------------------------------------------
    # Similarity and hybrid modes require a reference
    # destination.
    # -----------------------------------------------------

    else:

        if reference_destination is None:

            raise ValueError(
                "A reference destination is required for "
                "similarity and hybrid modes."
            )


        # -------------------------------------------------
        # Validate the reference destination.
        # -------------------------------------------------

        if (
            reference_destination
            not in similarity_matrix.index
        ):

            raise ValueError(
                f"Destination '{reference_destination}' "
                "was not found in the similarity matrix."
            )


        # -------------------------------------------------
        # Extract similarity to the reference destination.
        # -------------------------------------------------

        recommendations[
            "raw_similarity_score"
        ] = [
            similarity_matrix.loc[
                reference_destination,
                destination
            ]
            for destination
            in recommendations["destination"]
        ]


        # -------------------------------------------------
        # Convert cosine similarity from [-1, 1] to [0, 1].
        # -------------------------------------------------

        recommendations[
            "similarity_score"
        ] = (
            recommendations[
                "raw_similarity_score"
            ]
            + 1
        ) / 2


        # -------------------------------------------------
        # Similarity-only mode.
        # -------------------------------------------------

        if mode == "similarity":

            recommendations[
                "final_recommendation_score"
            ] = (
                recommendations[
                    "similarity_score"
                ]
            )


        # -------------------------------------------------
        # Hybrid mode.
        # -------------------------------------------------

        else:

            recommendations[
                "final_recommendation_score"
            ] = (

                preference_weight
                * recommendations[
                    "base_personalized_score"
                ]

                +

                similarity_weight
                * recommendations[
                    "similarity_score"
                ]
            )


    # -----------------------------------------------------
    # Never recommend the reference destination itself.
    # -----------------------------------------------------

    if reference_destination is not None:

        recommendations = (
            recommendations[
                recommendations["destination"]
                != reference_destination
            ]
        )


    # -----------------------------------------------------
    # Rank destinations from highest to lowest score.
    # -----------------------------------------------------

    recommendations = (
        recommendations
        .sort_values(
            "final_recommendation_score",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )


    # -----------------------------------------------------
    # Return the final ranked recommendations.
    # -----------------------------------------------------

    return recommendations


print("=" * 60)
print("✓ DYNAMIC RECOMMENDATION ENGINE CREATED")
print("=" * 60)

print("\nThe engine now accepts:")
print("  - user_preferences")
print("  - user_interests")
print("  - reference_destination")
print("  - recommendation mode")

print("\nSupported modes:")
print("  1. preference")
print("  2. similarity")
print("  3. hybrid")

print("\n✓ Recommendations are now user-dependent.")

✓ DYNAMIC RECOMMENDATION ENGINE CREATED

The engine now accepts:
  - user_preferences
  - user_interests
  - reference_destination
  - recommendation mode

Supported modes:
  1. preference
  2. similarity
  3. hybrid

✓ Recommendations are now user-dependent.


In [36]:
# ---------------------------------------------------------
# TEST DIFFERENT USER PROFILES
#
# The purpose of this test is to verify that our
# recommendation engine actually responds to user
# preferences.
#
# We will create three intentionally different users.
# These are evaluation profiles only — they are not
# training data.
# ---------------------------------------------------------


# ---------------------------------------------------------
# USER 1: BUDGET + NATURE TRAVELER
# ---------------------------------------------------------

user_1_preferences = {

    # Very strong concern for affordability.
    "budget": 1.0,

    # Flights are less important.
    "flight": 0.3,

    # Accommodation is important.
    "accommodation": 0.8,

    # Weather is highly important.
    "weather": 0.9,

    # Destination characteristics are highly important.
    "destination_characteristics": 0.9
}


user_1_interests = {

    # Strong nature preference.
    "nature": 1.0,

    # Some interest in sightseeing.
    "sightseeing": 0.3,

    # Very low coastal preference.
    "water_coastal": 0.1,

    # Strong wildlife preference.
    "wildlife": 0.9
}


# ---------------------------------------------------------
# USER 2: COASTAL + SIGHTSEEING TRAVELER
# ---------------------------------------------------------

user_2_preferences = {

    # Budget is moderately important.
    "budget": 0.5,

    # Flights are moderately important.
    "flight": 0.6,

    # Accommodation is moderately important.
    "accommodation": 0.6,

    # Weather is important.
    "weather": 0.8,

    # Destination characteristics are very important.
    "destination_characteristics": 1.0
}


user_2_interests = {

    # Low nature preference.
    "nature": 0.2,

    # Strong sightseeing preference.
    "sightseeing": 1.0,

    # Very strong coastal preference.
    "water_coastal": 1.0,

    # Low wildlife preference.
    "wildlife": 0.2
}


# ---------------------------------------------------------
# USER 3: WILDLIFE + NATURE TRAVELER
# ---------------------------------------------------------

user_3_preferences = {

    # Budget is moderately important.
    "budget": 0.6,

    # Flights have low importance.
    "flight": 0.3,

    # Accommodation is important.
    "accommodation": 0.7,

    # Weather is highly important.
    "weather": 0.9,

    # Destination characteristics are extremely important.
    "destination_characteristics": 1.0
}


user_3_interests = {

    # Very strong nature preference.
    "nature": 1.0,

    # Moderate sightseeing preference.
    "sightseeing": 0.4,

    # Low coastal preference.
    "water_coastal": 0.1,

    # Maximum wildlife preference.
    "wildlife": 1.0
}


# ---------------------------------------------------------
# Generate preference-only recommendations for each user.
#
# We start with preference mode because we want to verify
# that changing user preferences changes the ranking.
# ---------------------------------------------------------

user_1_results = generate_dynamic_recommendations(
    user_preferences=user_1_preferences,
    user_interests=user_1_interests,
    mode="preference",
    top_n=10
)


user_2_results = generate_dynamic_recommendations(
    user_preferences=user_2_preferences,
    user_interests=user_2_interests,
    mode="preference",
    top_n=10
)


user_3_results = generate_dynamic_recommendations(
    user_preferences=user_3_preferences,
    user_interests=user_3_interests,
    mode="preference",
    top_n=10
)


# ---------------------------------------------------------
# Display USER 1 results.
# ---------------------------------------------------------

print("=" * 60)
print("USER 1 — BUDGET + NATURE + WILDLIFE")
print("=" * 60)

display(
    user_1_results[
        [
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "base_personalized_score",
            "final_recommendation_score"
        ]
    ]
)


# ---------------------------------------------------------
# Display USER 2 results.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("USER 2 — SIGHTSEEING + COASTAL")
print("=" * 60)

display(
    user_2_results[
        [
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "base_personalized_score",
            "final_recommendation_score"
        ]
    ]
)


# ---------------------------------------------------------
# Display USER 3 results.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("USER 3 — NATURE + WILDLIFE")
print("=" * 60)

display(
    user_3_results[
        [
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "base_personalized_score",
            "final_recommendation_score"
        ]
    ]
)


# ---------------------------------------------------------
# Compare the top-ranked destinations.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("TOP DESTINATION COMPARISON")
print("=" * 60)

print(
    "User 1 top destination:",
    user_1_results.iloc[0]["destination"]
)

print(
    "User 2 top destination:",
    user_2_results.iloc[0]["destination"]
)

print(
    "User 3 top destination:",
    user_3_results.iloc[0]["destination"]
)


# ---------------------------------------------------------
# Check whether the recommendation rankings actually differ.
# ---------------------------------------------------------

ranking_1 = (
    user_1_results["destination"]
    .tolist()
)

ranking_2 = (
    user_2_results["destination"]
    .tolist()
)

ranking_3 = (
    user_3_results["destination"]
    .tolist()
)


all_rankings_identical = (
    ranking_1 == ranking_2
    and ranking_2 == ranking_3
)


if all_rankings_identical:

    raise ValueError(
        "All user rankings are identical. "
        "The dynamic preference system may not be "
        "responding to user preferences."
    )


print("\n" + "=" * 60)
print("DYNAMIC USER VALIDATION")
print("=" * 60)

print("✓ User 1 recommendations generated")
print("✓ User 2 recommendations generated")
print("✓ User 3 recommendations generated")
print("✓ User rankings respond to user preferences")

print("\n" + "=" * 60)
print("✓ DYNAMIC PERSONALIZATION TEST PASSED")
print("=" * 60)

USER 1 — BUDGET + NATURE + WILDLIFE


,destination,personalized_preference_score,personalized_interest_score,base_personalized_score,final_recommendation_score
0,Mumbai,0.717189,0.513102,0.635554,0.635554
1,Manali,0.611465,0.533690,0.580355,0.580355
2,Delhi,0.667078,0.364633,0.546100,0.546100
3,Bengaluru,0.677135,0.338144,0.541538,0.541538
4,Jaipur,0.615862,0.398495,0.528915,0.528915
5,Chennai,0.683369,0.272736,0.519116,0.519116
6,Kolkata,0.642078,0.318588,0.512682,0.512682
7,Pune,0.643030,0.281047,0.498237,0.498237
8,Kochi,0.597810,0.343104,0.495928,0.495928
9,Darjeeling,0.562811,0.358447,0.481066,0.481066



USER 2 — SIGHTSEEING + COASTAL


,destination,personalized_preference_score,personalized_interest_score,base_personalized_score,final_recommendation_score
0,Mumbai,0.700208,0.488703,0.615606,0.615606
1,Chennai,0.629744,0.536225,0.592337,0.592337
2,Delhi,0.644336,0.484275,0.580312,0.580312
3,Bengaluru,0.650624,0.473177,0.579645,0.579645
4,Kochi,0.552691,0.539458,0.547398,0.547398
5,Pune,0.590685,0.478795,0.545929,0.545929
6,Kolkata,0.589456,0.456054,0.536095,0.536095
7,Thiruvananthapuram,0.551472,0.480692,0.523160,0.523160
8,Jaipur,0.584848,0.357992,0.494106,0.494106
9,Hyderabad,0.573707,0.373791,0.493741,0.493741



USER 3 — NATURE + WILDLIFE


,destination,personalized_preference_score,personalized_interest_score,base_personalized_score,final_recommendation_score
0,Mumbai,0.694434,0.526501,0.627261,0.627261
1,Manali,0.573829,0.526021,0.554706,0.554706
2,Delhi,0.638540,0.377682,0.534197,0.534197
3,Bengaluru,0.639220,0.349116,0.523179,0.523179
4,Jaipur,0.589187,0.405269,0.515620,0.515620
5,Chennai,0.642628,0.283360,0.498921,0.498921
6,Kolkata,0.603018,0.326199,0.492291,0.492291
7,Pune,0.602768,0.292404,0.478623,0.478623
8,Kochi,0.561843,0.353648,0.478565,0.478565
9,Darjeeling,0.502752,0.359060,0.445275,0.445275



TOP DESTINATION COMPARISON
User 1 top destination: Mumbai
User 2 top destination: Mumbai
User 3 top destination: Mumbai

DYNAMIC USER VALIDATION
✓ User 1 recommendations generated
✓ User 2 recommendations generated
✓ User 3 recommendations generated
✓ User rankings respond to user preferences

✓ DYNAMIC PERSONALIZATION TEST PASSED


In [37]:
# ---------------------------------------------------------
# HYBRID RECOMMENDATION VALIDATION
#
# This test verifies that the hybrid model responds to both:
#
#   1. User preferences
#   2. Reference destination similarity
#
# We use User 1 and test three different reference
# destinations:
#
#   Goa
#   Manali
#   Jaipur
#
# The same user preferences are kept constant.
# Only the reference destination changes.
# ---------------------------------------------------------


# ---------------------------------------------------------
# Generate hybrid recommendations for Goa.
# ---------------------------------------------------------

goa_hybrid_results = generate_dynamic_recommendations(
    user_preferences=user_1_preferences,
    user_interests=user_1_interests,
    reference_destination="Goa",
    mode="hybrid",
    top_n=10
)


# ---------------------------------------------------------
# Generate hybrid recommendations for Manali.
# ---------------------------------------------------------

manali_hybrid_results = generate_dynamic_recommendations(
    user_preferences=user_1_preferences,
    user_interests=user_1_interests,
    reference_destination="Manali",
    mode="hybrid",
    top_n=10
)


# ---------------------------------------------------------
# Generate hybrid recommendations for Jaipur.
# ---------------------------------------------------------

jaipur_hybrid_results = generate_dynamic_recommendations(
    user_preferences=user_1_preferences,
    user_interests=user_1_interests,
    reference_destination="Jaipur",
    mode="hybrid",
    top_n=10
)


# ---------------------------------------------------------
# Display Goa hybrid recommendations.
# ---------------------------------------------------------

print("=" * 60)
print("HYBRID RECOMMENDATIONS — REFERENCE: GOA")
print("=" * 60)

display(
    goa_hybrid_results[
        [
            "destination",
            "base_personalized_score",
            "similarity_score",
            "final_recommendation_score"
        ]
    ]
)


# ---------------------------------------------------------
# Display Manali hybrid recommendations.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("HYBRID RECOMMENDATIONS — REFERENCE: MANALI")
print("=" * 60)

display(
    manali_hybrid_results[
        [
            "destination",
            "base_personalized_score",
            "similarity_score",
            "final_recommendation_score"
        ]
    ]
)


# ---------------------------------------------------------
# Display Jaipur hybrid recommendations.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("HYBRID RECOMMENDATIONS — REFERENCE: JAIPUR")
print("=" * 60)

display(
    jaipur_hybrid_results[
        [
            "destination",
            "base_personalized_score",
            "similarity_score",
            "final_recommendation_score"
        ]
    ]
)


# ---------------------------------------------------------
# Extract rankings for comparison.
# ---------------------------------------------------------

goa_ranking = (
    goa_hybrid_results[
        "destination"
    ].tolist()
)

manali_ranking = (
    manali_hybrid_results[
        "destination"
    ].tolist()
)

jaipur_ranking = (
    jaipur_hybrid_results[
        "destination"
    ].tolist()
)


# ---------------------------------------------------------
# Display the top destination for each reference.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("REFERENCE DESTINATION COMPARISON")
print("=" * 60)

print(
    "Goa reference top recommendation:",
    goa_ranking[0]
)

print(
    "Manali reference top recommendation:",
    manali_ranking[0]
)

print(
    "Jaipur reference top recommendation:",
    jaipur_ranking[0]
)


# ---------------------------------------------------------
# Verify that the reference destination itself is never
# recommended.
# ---------------------------------------------------------

if "Goa" in goa_ranking:

    raise ValueError(
        "Goa incorrectly appeared in its own recommendations."
    )


if "Manali" in manali_ranking:

    raise ValueError(
        "Manali incorrectly appeared in its own recommendations."
    )


if "Jaipur" in jaipur_ranking:

    raise ValueError(
        "Jaipur incorrectly appeared in its own recommendations."
    )


# ---------------------------------------------------------
# Verify that hybrid rankings respond to the reference
# destination.
#
# We do not require every ranking to be completely
# different. We only require that the reference destination
# has some influence on the ranking.
# ---------------------------------------------------------

if (
    goa_ranking == manali_ranking
    and
    manali_ranking == jaipur_ranking
):

    raise ValueError(
        "Hybrid rankings are identical for all reference "
        "destinations. Similarity may not be influencing "
        "the hybrid recommendation."
    )


# ---------------------------------------------------------
# Validate final score ordering.
# ---------------------------------------------------------

for result_df in [
    goa_hybrid_results,
    manali_hybrid_results,
    jaipur_hybrid_results
]:

    if not result_df[
        "final_recommendation_score"
    ].is_monotonic_decreasing:

        raise ValueError(
            "Hybrid recommendation scores are not "
            "sorted correctly."
        )


# ---------------------------------------------------------
# Validate final score range.
# ---------------------------------------------------------

for result_df in [
    goa_hybrid_results,
    manali_hybrid_results,
    jaipur_hybrid_results
]:

    if (
        result_df[
            "final_recommendation_score"
        ].min() < 0
        or
        result_df[
            "final_recommendation_score"
        ].max() > 1
    ):

        raise ValueError(
            "Hybrid score outside [0, 1]."
        )


print("\n" + "=" * 60)
print("HYBRID MODEL VALIDATION")
print("=" * 60)

print("✓ Goa hybrid recommendations generated")
print("✓ Manali hybrid recommendations generated")
print("✓ Jaipur hybrid recommendations generated")
print("✓ Reference destinations excluded")
print("✓ Hybrid rankings respond to reference destination")
print("✓ Hybrid ranking order validated")
print("✓ Hybrid score range validated")

print("\n" + "=" * 60)
print("✓ HYBRID RECOMMENDATION TEST PASSED")
print("=" * 60)

HYBRID RECOMMENDATIONS — REFERENCE: GOA


,destination,base_personalized_score,similarity_score,final_recommendation_score
0,Mumbai,0.635554,0.793244,0.682861
1,Delhi,0.546100,0.600620,0.562456
2,Bengaluru,0.541538,0.604545,0.560440
3,Jaipur,0.528915,0.586255,0.546117
4,Manali,0.580355,0.449051,0.540964
5,Pune,0.498237,0.511279,0.502149
6,Chennai,0.519116,0.436365,0.494291
7,Gokarna,0.445597,0.597932,0.491298
8,Hyderabad,0.463945,0.533714,0.484876
9,Alappuzha,0.398903,0.677657,0.482529



HYBRID RECOMMENDATIONS — REFERENCE: MANALI


,destination,base_personalized_score,similarity_score,final_recommendation_score
0,Mumbai,0.635554,0.425243,0.572461
1,Jaipur,0.528915,0.606082,0.552065
2,Shimla,0.474586,0.710036,0.545221
3,Darjeeling,0.481066,0.636352,0.527652
4,Hampi,0.430748,0.695990,0.510321
5,Bengaluru,0.541538,0.434797,0.509516
6,Delhi,0.546100,0.422830,0.509119
7,Chennai,0.519116,0.459700,0.501291
8,Kolkata,0.512682,0.465796,0.498616
9,Varanasi,0.464600,0.538982,0.486915



HYBRID RECOMMENDATIONS — REFERENCE: JAIPUR


,destination,base_personalized_score,similarity_score,final_recommendation_score
0,Mumbai,0.635554,0.753803,0.671029
1,Delhi,0.546100,0.800961,0.622558
2,Bengaluru,0.541538,0.704435,0.590407
3,Manali,0.580355,0.606082,0.588073
4,Kolkata,0.512682,0.705269,0.570458
5,Pune,0.498237,0.605594,0.530444
6,Chennai,0.519116,0.527262,0.521560
7,Kochi,0.495928,0.574591,0.519527
8,Varanasi,0.464600,0.603295,0.506208
9,Goa,0.467868,0.586255,0.503384



REFERENCE DESTINATION COMPARISON
Goa reference top recommendation: Mumbai
Manali reference top recommendation: Mumbai
Jaipur reference top recommendation: Mumbai

HYBRID MODEL VALIDATION
✓ Goa hybrid recommendations generated
✓ Manali hybrid recommendations generated
✓ Jaipur hybrid recommendations generated
✓ Reference destinations excluded
✓ Hybrid rankings respond to reference destination
✓ Hybrid ranking order validated
✓ Hybrid score range validated

✓ HYBRID RECOMMENDATION TEST PASSED


In [46]:
# ---------------------------------------------------------
# RECREATE PREFERENCE DIRECTION MAP
#
# The kernel restart removed the previous Python variable
# `preference_direction_map`.
#
# We recreate it here and save it permanently so future
# kernel restarts do not cause the same problem.
#
# Direction meaning:
#
#   "higher" -> larger value is better
#   "lower"  -> smaller value is better
# ---------------------------------------------------------


# ---------------------------------------------------------
# Features where LOWER values are considered better.
#
# Examples:
#   Lower flight price       -> better
#   Lower flight duration    -> better
#   Fewer flight stops       -> better
#   Lower hotel prices       -> better
#   Lower humidity/rain/etc. -> better
# ---------------------------------------------------------

lower_is_better_features = {

    "avg_flight_price",
    "avg_total_duration",
    "avg_outbound_stops",
    "avg_return_stops",

    "min_hotel_price",
    "avg_hotel_price",
    "max_hotel_price",

    "humidity",
    "wind_speed",
    "cloudiness",
    "rain_1h"
}


# ---------------------------------------------------------
# All preference features.
#
# These come from the preference-group configuration that
# we already validated earlier.
# ---------------------------------------------------------

all_preference_features = []

for group_features in preference_groups.values():

    all_preference_features.extend(
        group_features
    )


# Remove duplicates while preserving order.
all_preference_features = list(
    dict.fromkeys(
        all_preference_features
    )
)


# ---------------------------------------------------------
# Create the direction map.
#
# Every feature must receive exactly one direction.
# ---------------------------------------------------------

preference_direction_map = {}

for feature in all_preference_features:

    if feature in lower_is_better_features:

        preference_direction_map[
            feature
        ] = "lower"

    else:

        preference_direction_map[
            feature
        ] = "higher"


# ---------------------------------------------------------
# Validate the direction map.
# ---------------------------------------------------------

missing_direction_features = [
    feature
    for feature in all_preference_features
    if feature not in preference_direction_map
]


unexpected_direction_features = [
    feature
    for feature in preference_direction_map
    if feature not in all_preference_features
]


if missing_direction_features:

    raise ValueError(
        "Features without a direction: "
        + str(missing_direction_features)
    )


if unexpected_direction_features:

    raise ValueError(
        "Unexpected direction features: "
        + str(unexpected_direction_features)
    )


# ---------------------------------------------------------
# Check that every feature has exactly one direction.
# ---------------------------------------------------------

higher_features = [
    feature
    for feature, direction
    in preference_direction_map.items()
    if direction == "higher"
]


lower_features = [
    feature
    for feature, direction
    in preference_direction_map.items()
    if direction == "lower"
]


if (
    len(higher_features)
    + len(lower_features)
    != len(all_preference_features)
):

    raise ValueError(
        "Some preference features were assigned "
        "incorrectly."
    )


# ---------------------------------------------------------
# Save the direction map permanently.
# ---------------------------------------------------------

direction_map_path = os.path.join(
    MODEL_DIR,
    "preference_direction_map.pkl"
)


with open(
    direction_map_path,
    "wb"
) as file:

    pickle.dump(
        preference_direction_map,
        file
    )


# ---------------------------------------------------------
# Display validation information.
# ---------------------------------------------------------

print("=" * 60)
print("PREFERENCE DIRECTION MAP RECREATED")
print("=" * 60)

print(
    "\nTotal preference features:",
    len(all_preference_features)
)

print(
    "Higher-is-better features:",
    len(higher_features)
)

print(
    "Lower-is-better features:",
    len(lower_features)
)

print("\nLower-is-better features:")

for feature in lower_features:

    print(
        "  -",
        feature
    )


print("\nSaved:")
print(
    f"✓ {direction_map_path}"
)


print("\n" + "=" * 60)
print("✓ PREFERENCE DIRECTION MAP VALIDATED AND SAVED")
print("=" * 60)

PREFERENCE DIRECTION MAP RECREATED

Total preference features: 29
Higher-is-better features: 18
Lower-is-better features: 11

Lower-is-better features:
  - avg_flight_price
  - min_hotel_price
  - avg_hotel_price
  - max_hotel_price
  - avg_total_duration
  - avg_outbound_stops
  - avg_return_stops
  - humidity
  - wind_speed
  - cloudiness
  - rain_1h

Saved:
✓ ../models\preference_direction_map.pkl

✓ PREFERENCE DIRECTION MAP VALIDATED AND SAVED


In [48]:
# ---------------------------------------------------------
# RECONSTRUCT MODEL-BUILDING VARIABLES
#
# The kernel restart removed temporary Python variables.
# We reconstruct the variables needed for final packaging
# from the saved datasets/artifacts.
# ---------------------------------------------------------


import os
import pickle
import pandas as pd


# ---------------------------------------------------------
# Model directory.
# ---------------------------------------------------------

MODEL_DIR = "../models"


# ---------------------------------------------------------
# Load the PCA destination dataset if it is not currently
# available in memory.
# ---------------------------------------------------------

if "pca_destination_df" not in globals():

    pca_destination_df = pd.read_csv(
        "../data/cleaned/pca_destination_features.csv"
    )


# ---------------------------------------------------------
# Reconstruct PCA feature columns directly from the
# destination dataset.
#
# The destination column is metadata, while PC1-PC15 are
# the actual recommendation features.
# ---------------------------------------------------------

pca_columns = [
    column
    for column in pca_destination_df.columns
    if column.startswith("PC")
]


# ---------------------------------------------------------
# Validate PCA feature columns.
# ---------------------------------------------------------

expected_pca_columns = [
    f"PC{i}"
    for i in range(1, 16)
]


if pca_columns != expected_pca_columns:

    raise ValueError(
        "Unexpected PCA columns.\n"
        f"Found: {pca_columns}\n"
        f"Expected: {expected_pca_columns}"
    )


# ---------------------------------------------------------
# Reconstruct the destination list.
# ---------------------------------------------------------

destination_names = (
    pca_destination_df[
        "destination"
    ]
    .tolist()
)


# ---------------------------------------------------------
# Load preference groups if the variable was lost after
# the kernel restart.
# ---------------------------------------------------------

if "preference_groups" not in globals():

    preference_groups_path = os.path.join(
        MODEL_DIR,
        "preference_groups.pkl"
    )

    with open(
        preference_groups_path,
        "rb"
    ) as file:

        preference_groups = pickle.load(
            file
        )


# ---------------------------------------------------------
# Load interest profiles if necessary.
# ---------------------------------------------------------

if "interest_profiles" not in globals():

    interest_profiles_path = os.path.join(
        MODEL_DIR,
        "interest_profiles.pkl"
    )

    with open(
        interest_profiles_path,
        "rb"
    ) as file:

        interest_profiles = pickle.load(
            file
        )


# ---------------------------------------------------------
# Load the preference direction map if necessary.
# ---------------------------------------------------------

if "preference_direction_map" not in globals():

    direction_map_path = os.path.join(
        MODEL_DIR,
        "preference_direction_map.pkl"
    )

    if os.path.exists(
        direction_map_path
    ):

        with open(
            direction_map_path,
            "rb"
        ) as file:

            preference_direction_map = pickle.load(
                file
            )

    else:

        raise FileNotFoundError(
            "preference_direction_map.pkl was not found. "
            "Run the direction-map reconstruction cell first."
        )


# ---------------------------------------------------------
# Reconstruct the hybrid recommendation weights.
#
# These were defined earlier in the notebook.
# ---------------------------------------------------------

if "DEFAULT_PREFERENCE_WEIGHT" not in globals():

    DEFAULT_PREFERENCE_WEIGHT = 0.70


if "DEFAULT_SIMILARITY_WEIGHT" not in globals():

    DEFAULT_SIMILARITY_WEIGHT = 0.30


# ---------------------------------------------------------
# Reconstruct the travel-factor / interest weights.
#
# These were used earlier when calculating the base
# personalized score.
# ---------------------------------------------------------

if "TRAVEL_FACTOR_WEIGHT" not in globals():

    TRAVEL_FACTOR_WEIGHT = 0.60


if "INTEREST_WEIGHT" not in globals():

    INTEREST_WEIGHT = 0.40


# ---------------------------------------------------------
# Final reconstruction validation.
# ---------------------------------------------------------

print("=" * 60)
print("MODEL PACKAGING VARIABLES RECONSTRUCTED")
print("=" * 60)

print(
    "\nPCA components:",
    len(pca_columns)
)

print(
    "PCA columns:",
    pca_columns
)

print(
    "\nDestinations:",
    len(destination_names)
)

print(
    "Preference groups:",
    len(preference_groups)
)

print(
    "Interest profiles:",
    len(interest_profiles)
)

print(
    "Preference direction entries:",
    len(preference_direction_map)
)

print(
    "\nTravel-factor weight:",
    TRAVEL_FACTOR_WEIGHT
)

print(
    "Interest weight:",
    INTEREST_WEIGHT
)

print(
    "Hybrid preference weight:",
    DEFAULT_PREFERENCE_WEIGHT
)

print(
    "Hybrid similarity weight:",
    DEFAULT_SIMILARITY_WEIGHT
)


# ---------------------------------------------------------
# Sanity checks.
# ---------------------------------------------------------

if len(pca_columns) != 15:

    raise ValueError(
        "Expected 15 PCA components."
    )


if len(destination_names) != 50:

    raise ValueError(
        "Expected 50 destinations."
    )


if len(preference_groups) != 5:

    raise ValueError(
        "Expected 5 preference groups."
    )


if len(interest_profiles) != 4:

    raise ValueError(
        "Expected 4 interest profiles."
    )


if len(preference_direction_map) != 29:

    raise ValueError(
        "Expected 29 preference-direction entries."
    )


print("\n" + "=" * 60)
print("✓ ALL PACKAGING VARIABLES AVAILABLE")
print("=" * 60)

MODEL PACKAGING VARIABLES RECONSTRUCTED

PCA components: 15
PCA columns: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15']

Destinations: 50
Preference groups: 5
Interest profiles: 4
Preference direction entries: 29

Travel-factor weight: 0.6
Interest weight: 0.4
Hybrid preference weight: 0.7
Hybrid similarity weight: 0.3

✓ ALL PACKAGING VARIABLES AVAILABLE


In [49]:
# ---------------------------------------------------------
# FINAL RECOMMENDATION MODEL PACKAGING
#
# This cell saves the reusable configuration required by
# the Travel Agent recommendation system.
#
# We intentionally do NOT save the sample user's scores
# as the model. User preferences will be supplied
# dynamically by the application.
#
# The saved configuration contains:
#
#   1. Preference groups
#   2. Preference directions
#   3. Interest profiles
#   4. Travel-factor / interest weights
#   5. Hybrid recommendation weights
#   6. Destination list
#   7. Recommendation feature columns
#
# Existing PCA and similarity artifacts remain separate.
# ---------------------------------------------------------


import os
import pickle


# ---------------------------------------------------------
# Define the model directory.
# ---------------------------------------------------------

MODEL_DIR = "../models"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


# ---------------------------------------------------------
# Build the reusable recommendation configuration.
#
# These values describe HOW the recommendation engine
# works. They are not specific to one user.
# ---------------------------------------------------------

recommendation_config = {

    # -----------------------------------------------------
    # User travel-factor groups.
    # -----------------------------------------------------
    "preference_groups": preference_groups,

    # -----------------------------------------------------
    # Higher-is-better / lower-is-better feature mapping.
    # -----------------------------------------------------
    "preference_directions": preference_direction_map,

    # -----------------------------------------------------
    # Destination interest profiles.
    # -----------------------------------------------------
    "interest_profiles": interest_profiles,

    # -----------------------------------------------------
    # Weight between travel factors and destination
    # interests.
    # -----------------------------------------------------
    "travel_factor_weight": TRAVEL_FACTOR_WEIGHT,
    "interest_weight": INTEREST_WEIGHT,

    # -----------------------------------------------------
    # Weight between personalized preference and
    # reference-destination similarity.
    # -----------------------------------------------------
    "hybrid_preference_weight": DEFAULT_PREFERENCE_WEIGHT,
    "hybrid_similarity_weight": DEFAULT_SIMILARITY_WEIGHT,

    # -----------------------------------------------------
    # PCA recommendation features.
    # -----------------------------------------------------
    "recommendation_feature_columns": pca_columns,

    # -----------------------------------------------------
    # Destination list.
    # -----------------------------------------------------
    "destinations": destination_names
}


# ---------------------------------------------------------
# Save the recommendation configuration.
# ---------------------------------------------------------

recommendation_config_path = os.path.join(
    MODEL_DIR,
    "recommendation_config.pkl"
)

with open(
    recommendation_config_path,
    "wb"
) as file:

    pickle.dump(
        recommendation_config,
        file
    )


# ---------------------------------------------------------
# Save the preference-group configuration separately.
#
# Keeping this separate makes the application easier to
# inspect and debug later.
# ---------------------------------------------------------

preference_groups_path = os.path.join(
    MODEL_DIR,
    "preference_groups.pkl"
)

with open(
    preference_groups_path,
    "wb"
) as file:

    pickle.dump(
        preference_groups,
        file
    )


# ---------------------------------------------------------
# Save the interest-profile configuration separately.
# ---------------------------------------------------------

interest_profiles_path = os.path.join(
    MODEL_DIR,
    "interest_profiles.pkl"
)

with open(
    interest_profiles_path,
    "wb"
) as file:

    pickle.dump(
        interest_profiles,
        file
    )


# ---------------------------------------------------------
# Verify that every artifact exists.
# ---------------------------------------------------------

artifacts_to_verify = {

    "recommendation_config.pkl":
        recommendation_config_path,

    "preference_groups.pkl":
        preference_groups_path,

    "interest_profiles.pkl":
        interest_profiles_path
}


print("=" * 60)
print("FINAL RECOMMENDATION CONFIGURATION SAVED")
print("=" * 60)

for artifact_name, artifact_path in artifacts_to_verify.items():

    if os.path.exists(artifact_path):

        file_size = os.path.getsize(
            artifact_path
        )

        print(
            f"✓ {artifact_path} "
            f"({file_size:,} bytes)"
        )

    else:

        raise FileNotFoundError(
            f"Failed to save {artifact_name}"
        )


print("\n" + "=" * 60)
print("SAVED CONFIGURATION CONTENTS")
print("=" * 60)

print(
    "\nPreference groups:",
    len(
        recommendation_config[
            "preference_groups"
        ]
    )
)

print(
    "Interest profiles:",
    len(
        recommendation_config[
            "interest_profiles"
        ]
    )
)

print(
    "Recommendation features:",
    len(
        recommendation_config[
            "recommendation_feature_columns"
        ]
    )
)

print(
    "Destinations:",
    len(
        recommendation_config[
            "destinations"
        ]
    )
)

print(
    "\nTravel-factor weight:",
    recommendation_config[
        "travel_factor_weight"
    ]
)

print(
    "Interest weight:",
    recommendation_config[
        "interest_weight"
    ]
)

print(
    "Hybrid preference weight:",
    recommendation_config[
        "hybrid_preference_weight"
    ]
)

print(
    "Hybrid similarity weight:",
    recommendation_config[
        "hybrid_similarity_weight"
    ]
)

print("\n" + "=" * 60)
print("✓ RECOMMENDATION CONFIGURATION PACKAGING COMPLETE")
print("=" * 60)

FINAL RECOMMENDATION CONFIGURATION SAVED
✓ ../models\recommendation_config.pkl (1,643 bytes)
✓ ../models\preference_groups.pkl (571 bytes)
✓ ../models\interest_profiles.pkl (66 bytes)

SAVED CONFIGURATION CONTENTS

Preference groups: 5
Interest profiles: 4
Recommendation features: 15
Destinations: 50

Travel-factor weight: 0.6
Interest weight: 0.4
Hybrid preference weight: 0.7
Hybrid similarity weight: 0.3

✓ RECOMMENDATION CONFIGURATION PACKAGING COMPLETE


In [50]:
# ---------------------------------------------------------
# FINAL SAVED-ARTIFACT RELOAD TEST
#
# This test verifies that the recommendation system can be
# reconstructed from the files saved in models/.
#
# The purpose is to make sure the system does not depend
# on temporary notebook variables.
# ---------------------------------------------------------


import os
import pickle
import pandas as pd


MODEL_DIR = "../models"


# ---------------------------------------------------------
# Define every artifact required by the recommendation
# system.
# ---------------------------------------------------------

required_artifacts = [

    "feature_columns.pkl",
    "numerical_feature_columns.pkl",
    "binary_feature_columns.pkl",
    "redundant_features.pkl",

    "feature_scaler.pkl",
    "pca_model.pkl",

    "destination_similarity.pkl",
    "recommendation_destinations.pkl",
    "recommendation_feature_columns.pkl",

    "destination_profile_scores.pkl",

    "preference_direction_map.pkl",
    "preference_groups.pkl",
    "interest_profiles.pkl",
    "recommendation_config.pkl"
]


# ---------------------------------------------------------
# Verify that every required artifact exists.
# ---------------------------------------------------------

missing_artifacts = [

    artifact
    for artifact in required_artifacts
    if not os.path.exists(
        os.path.join(
            MODEL_DIR,
            artifact
        )
    )
]


if missing_artifacts:

    raise FileNotFoundError(
        "Missing model artifacts: "
        + str(missing_artifacts)
    )


# ---------------------------------------------------------
# Load every artifact into a NEW dictionary.
#
# We deliberately use new variable names so this test
# does not depend on the objects currently in memory.
# ---------------------------------------------------------

reloaded_artifacts = {}


for artifact in required_artifacts:

    artifact_path = os.path.join(
        MODEL_DIR,
        artifact
    )

    with open(
        artifact_path,
        "rb"
    ) as file:

        reloaded_artifacts[
            artifact
        ] = pickle.load(file)


# ---------------------------------------------------------
# Load the PCA destination dataset independently.
# ---------------------------------------------------------

reloaded_pca_dataset = pd.read_csv(
    "../data/cleaned/pca_destination_features.csv"
)


# ---------------------------------------------------------
# Validate the similarity matrix.
# ---------------------------------------------------------

reloaded_similarity = (
    reloaded_artifacts[
        "destination_similarity.pkl"
    ]
)


if reloaded_similarity.shape != (50, 50):

    raise ValueError(
        "Reloaded similarity matrix has an "
        "unexpected shape: "
        + str(reloaded_similarity.shape)
    )


# ---------------------------------------------------------
# Validate destination list.
# ---------------------------------------------------------

reloaded_destinations = (
    reloaded_artifacts[
        "recommendation_destinations.pkl"
    ]
)


if len(reloaded_destinations) != 50:

    raise ValueError(
        "Expected 50 destinations, found "
        + str(len(reloaded_destinations))
    )


# ---------------------------------------------------------
# Validate recommendation feature columns.
# ---------------------------------------------------------

reloaded_recommendation_columns = (
    reloaded_artifacts[
        "recommendation_feature_columns.pkl"
    ]
)


if len(reloaded_recommendation_columns) != 15:

    raise ValueError(
        "Expected 15 recommendation features."
    )


# ---------------------------------------------------------
# Validate PCA model.
# ---------------------------------------------------------

reloaded_pca = (
    reloaded_artifacts[
        "pca_model.pkl"
    ]
)


if reloaded_pca.n_components_ != 15:

    raise ValueError(
        "Reloaded PCA model does not contain "
        "15 components."
    )


# ---------------------------------------------------------
# Validate scaler.
# ---------------------------------------------------------

reloaded_scaler = (
    reloaded_artifacts[
        "feature_scaler.pkl"
    ]
)


if reloaded_scaler.n_features_in_ != 34:

    raise ValueError(
        "Reloaded scaler does not expect "
        "34 numerical features."
    )


# ---------------------------------------------------------
# Validate preference configuration.
# ---------------------------------------------------------

reloaded_config = (
    reloaded_artifacts[
        "recommendation_config.pkl"
    ]
)


if (
    reloaded_config[
        "travel_factor_weight"
    ]
    != 0.60
):

    raise ValueError(
        "Unexpected travel-factor weight."
    )


if (
    reloaded_config[
        "interest_weight"
    ]
    != 0.40
):

    raise ValueError(
        "Unexpected interest weight."
    )


if (
    reloaded_config[
        "hybrid_preference_weight"
    ]
    != 0.70
):

    raise ValueError(
        "Unexpected hybrid preference weight."
    )


if (
    reloaded_config[
        "hybrid_similarity_weight"
    ]
    != 0.30
):

    raise ValueError(
        "Unexpected hybrid similarity weight."
    )


# ---------------------------------------------------------
# Validate the PCA destination dataset.
# ---------------------------------------------------------

expected_pca_columns = [
    "destination"
] + [
    f"PC{i}"
    for i in range(1, 16)
]


if (
    reloaded_pca_dataset.columns.tolist()
    != expected_pca_columns
):

    raise ValueError(
        "PCA destination dataset columns do not "
        "match the expected structure."
    )


if len(reloaded_pca_dataset) != 50:

    raise ValueError(
        "Expected 50 PCA destination rows."
    )


# ---------------------------------------------------------
# Final report.
# ---------------------------------------------------------

print("=" * 60)
print("SAVED ARTIFACT RELOAD TEST")
print("=" * 60)

print(
    "\nArtifacts successfully reloaded:",
    len(reloaded_artifacts)
)

print(
    "Destinations:",
    len(reloaded_destinations)
)

print(
    "Similarity matrix:",
    reloaded_similarity.shape
)

print(
    "Recommendation features:",
    len(reloaded_recommendation_columns)
)

print(
    "PCA components:",
    reloaded_pca.n_components_
)

print(
    "Scaler input features:",
    reloaded_scaler.n_features_in_
)

print(
    "PCA dataset:",
    reloaded_pca_dataset.shape
)

print("\n" + "=" * 60)
print("✓ SAVED ARTIFACT RELOAD TEST PASSED")
print("=" * 60)

print(
    "\nThe recommendation system can now be "
    "reconstructed from saved artifacts."
)

UnpicklingError: STACK_GLOBAL requires str

In [51]:
# ---------------------------------------------------------
# DIAGNOSE ALL SAVED PICKLE ARTIFACTS
#
# The previous reload test failed with:
#
#     UnpicklingError: STACK_GLOBAL requires str
#
# This usually means one of the saved pickle files is
# malformed, corrupted, or was saved incorrectly.
#
# This cell tests every pickle separately so we can identify
# the exact problematic artifact.
# ---------------------------------------------------------

import os
import pickle


MODEL_DIR = "../models"


# ---------------------------------------------------------
# List all pickle files currently stored in models/.
# ---------------------------------------------------------

pickle_files = sorted(
    [
        file
        for file in os.listdir(MODEL_DIR)
        if file.endswith(".pkl")
    ]
)


print("=" * 60)
print("PICKLE ARTIFACT DIAGNOSTIC")
print("=" * 60)

print(
    "\nTotal pickle files found:",
    len(pickle_files)
)


# ---------------------------------------------------------
# Test each pickle independently.
# ---------------------------------------------------------

successful_files = []
failed_files = []


for filename in pickle_files:

    filepath = os.path.join(
        MODEL_DIR,
        filename
    )

    try:

        with open(
            filepath,
            "rb"
        ) as file:

            obj = pickle.load(
                file
            )

        successful_files.append(
            filename
        )

        print(
            f"✓ {filename}"
        )

    except Exception as error:

        failed_files.append(
            (
                filename,
                type(error).__name__,
                str(error)
            )
        )

        print(
            f"✗ {filename}"
        )

        print(
            f"    Error: {type(error).__name__}"
        )

        print(
            f"    Message: {error}"
        )


# ---------------------------------------------------------
# Final diagnostic summary.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("DIAGNOSTIC SUMMARY")
print("=" * 60)

print(
    "\nSuccessfully loaded:",
    len(successful_files)
)

print(
    "Failed to load:",
    len(failed_files)
)


if failed_files:

    print("\nProblematic artifacts:")

    for (
        filename,
        error_type,
        error_message
    ) in failed_files:

        print(
            f"\n✗ {filename}"
        )

        print(
            f"  Error type: {error_type}"
        )

        print(
            f"  Error: {error_message}"
        )

else:

    print(
        "\n✓ All pickle artifacts loaded successfully."
    )


print("\n" + "=" * 60)
print("END OF PICKLE DIAGNOSTIC")
print("=" * 60)

PICKLE ARTIFACT DIAGNOSTIC

Total pickle files found: 14
✓ binary_feature_columns.pkl
✓ destination_profile_scores.pkl
✓ destination_similarity.pkl
✓ feature_columns.pkl
✗ feature_scaler.pkl
    Error: UnpicklingError
    Message: STACK_GLOBAL requires str
✓ interest_profiles.pkl
✓ numerical_feature_columns.pkl
✗ pca_model.pkl
    Error: UnpicklingError
    Message: STACK_GLOBAL requires str
✓ preference_direction_map.pkl
✓ preference_groups.pkl
✓ recommendation_config.pkl
✓ recommendation_destinations.pkl
✓ recommendation_feature_columns.pkl
✓ redundant_features.pkl

DIAGNOSTIC SUMMARY

Successfully loaded: 12
Failed to load: 2

Problematic artifacts:

✗ feature_scaler.pkl
  Error type: UnpicklingError
  Error: STACK_GLOBAL requires str

✗ pca_model.pkl
  Error type: UnpicklingError
  Error: STACK_GLOBAL requires str

END OF PICKLE DIAGNOSTIC


In [52]:
# ---------------------------------------------------------
# IDENTIFY THE ORIGINAL DATA USED FOR SCALER AND PCA
#
# The scaler and PCA pickle files are corrupted.
#
# Before rebuilding them, we need to identify the correct
# source dataset so that the repaired models reproduce the
# same transformation used earlier.
# ---------------------------------------------------------

import os
import pandas as pd


# ---------------------------------------------------------
# Candidate cleaned datasets that may contain the original
# model features.
# ---------------------------------------------------------

candidate_files = [

    "../data/cleaned/integrated_travel_dataset.csv",
    "../data/cleaned/travel_features.csv",
    "../data/cleaned/travel_integrated_preprocessed.csv",
    "../data/cleaned/model_features_scaled.csv",
    "../data/cleaned/pca_destination_features.csv"
]


print("=" * 60)
print("MODEL INPUT DATASET DIAGNOSTIC")
print("=" * 60)


for filepath in candidate_files:

    print("\n" + "-" * 60)
    print("File:", filepath)

    if not os.path.exists(filepath):

        print("✗ File not found")
        continue

    try:

        df = pd.read_csv(filepath)

        print("✓ Loaded successfully")

        print(
            "Rows:",
            df.shape[0]
        )

        print(
            "Columns:",
            df.shape[1]
        )

        print("\nColumns:")

        for number, column in enumerate(
            df.columns,
            start=1
        ):

            print(
                f"{number:2d}. {column}"
            )

    except Exception as error:

        print(
            "✗ Could not load:",
            error
        )


print("\n" + "=" * 60)
print("END OF DATASET DIAGNOSTIC")
print("=" * 60)

MODEL INPUT DATASET DIAGNOSTIC

------------------------------------------------------------
File: ../data/cleaned/integrated_travel_dataset.csv
✓ Loaded successfully
Rows: 50
Columns: 42

Columns:
 1. destination
 2. sight_count
 3. park_count
 4. restaurant_count
 5. water_count
 6. forest_count
 7. wetland_count
 8. river_count
 9. mountain_count
10. coastal_count
11. sand_count
12. protected_area_count
13. hotel_count
14. room_count
15. min_hotel_price
16. avg_hotel_price
17. max_hotel_price
18. avg_allotment
19. flight_count
20. min_flight_price
21. avg_flight_price
22. max_flight_price
23. avg_total_duration
24. avg_outbound_stops
25. avg_return_stops
26. weather_available
27. accommodation_available
28. flight_available
29. country
30. latitude
31. longitude
32. temperature
33. feels_like
34. humidity
35. pressure
36. wind_speed
37. cloudiness
38. weather_condition
39. weather_description
40. visibility
41. rain_1h
42. timestamp

-------------------------------------------------

In [53]:
# ---------------------------------------------------------
# REPAIR CORRUPTED SCALER AND PCA ARTIFACTS
#
# Two saved artifacts were found to be unreadable:
#
#   1. feature_scaler.pkl
#   2. pca_model.pkl
#
# We will rebuild them from the already-saved processing
# outputs instead of changing the recommendation pipeline.
#
# Important:
#   - The saved feature metadata is reused.
#   - The saved scaled dataset is reused.
#   - The existing PCA destination dataset is used as the
#     reference for validating the rebuilt PCA.
# ---------------------------------------------------------

import os
import pickle

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


MODEL_DIR = "../models"
DATA_DIR = "../data/cleaned"


# ---------------------------------------------------------
# Load the healthy feature metadata.
#
# These files were already verified successfully.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "feature_columns.pkl"
    ),
    "rb"
) as file:

    original_feature_columns = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "numerical_feature_columns.pkl"
    ),
    "rb"
) as file:

    numerical_feature_columns = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "binary_feature_columns.pkl"
    ),
    "rb"
) as file:

    binary_feature_columns = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "redundant_features.pkl"
    ),
    "rb"
) as file:

    redundant_features = pickle.load(file)


# ---------------------------------------------------------
# Load the already-saved processed/scaled dataset.
#
# This is the most important source for reconstructing
# the PCA input because it represents the data after the
# preprocessing stage.
# ---------------------------------------------------------

scaled_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "model_features_scaled.csv"
    )
)


# ---------------------------------------------------------
# Load the existing PCA destination dataset.
#
# We will use this as the reference to check whether the
# repaired PCA produces the same destination representation.
# ---------------------------------------------------------

existing_pca_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "pca_destination_features.csv"
    )
)


print("=" * 60)
print("REPAIRING SCALER AND PCA ARTIFACTS")
print("=" * 60)

print(
    "\nOriginal features:",
    len(original_feature_columns)
)

print(
    "Numerical features:",
    len(numerical_feature_columns)
)

print(
    "Binary features:",
    len(binary_feature_columns)
)

print(
    "Redundant features:",
    len(redundant_features)
)

print(
    "Saved scaled dataset:",
    scaled_df.shape
)

print(
    "Saved PCA dataset:",
    existing_pca_df.shape
)


# ---------------------------------------------------------
# Validate the saved scaled dataset before using it.
# ---------------------------------------------------------

expected_scaled_columns = (
    ["destination"]
    + original_feature_columns
)


if scaled_df.columns.tolist() != expected_scaled_columns:

    raise ValueError(
        "Saved scaled dataset columns do not match "
        "feature_columns.pkl."
    )


if len(original_feature_columns) != 47:

    raise ValueError(
        "Expected 47 original features."
    )


if len(numerical_feature_columns) != 34:

    raise ValueError(
        "Expected 34 numerical features."
    )


if len(binary_feature_columns) != 13:

    raise ValueError(
        "Expected 13 binary features."
    )


if len(redundant_features) != 2:

    raise ValueError(
        "Expected 2 redundant features."
    )


# ---------------------------------------------------------
# Construct the exact 45-feature matrix that enters PCA.
#
# The two redundant flight-price features are removed.
# ---------------------------------------------------------

pca_input_columns = [

    feature
    for feature in original_feature_columns

    if feature not in redundant_features
]


if len(pca_input_columns) != 45:

    raise ValueError(
        "Expected 45 PCA input features."
    )


X_pca_input = scaled_df[
    pca_input_columns
].copy()


# ---------------------------------------------------------
# Check that the PCA input contains no missing values.
# ---------------------------------------------------------

if X_pca_input.isna().sum().sum() != 0:

    raise ValueError(
        "Missing values found in PCA input."
    )


# ---------------------------------------------------------
# Rebuild the PCA model.
#
# The PCA is fitted directly on the already-saved processed
# matrix. This preserves the existing processing pipeline
# and avoids changing any preprocessing decisions.
# ---------------------------------------------------------

repaired_pca = PCA(
    n_components=15
)


repaired_pca.fit(
    X_pca_input
)


# ---------------------------------------------------------
# Generate PCA values using the repaired model.
# ---------------------------------------------------------

repaired_pca_values = (
    repaired_pca.transform(
        X_pca_input
    )
)


repaired_pca_df = pd.DataFrame(
    repaired_pca_values,
    columns=[
        f"PC{i}"
        for i in range(1, 16)
    ]
)


repaired_pca_df.insert(
    0,
    "destination",
    scaled_df["destination"].values
)


# ---------------------------------------------------------
# Compare the repaired PCA output against the previously
# saved PCA destination dataset.
#
# PCA component signs can mathematically be flipped without
# changing the underlying PCA solution, so we compare each
# component allowing for a possible sign reversal.
# ---------------------------------------------------------

pca_comparison = []


for i in range(1, 16):

    component = f"PC{i}"

    old_values = (
        existing_pca_df[
            component
        ].values
    )

    new_values = (
        repaired_pca_df[
            component
        ].values
    )

    direct_difference = np.max(
        np.abs(
            old_values
            - new_values
        )
    )

    flipped_difference = np.max(
        np.abs(
            old_values
            + new_values
        )
    )

    best_difference = min(
        direct_difference,
        flipped_difference
    )

    pca_comparison.append(
        best_difference
    )


max_pca_difference = max(
    pca_comparison
)


print("\n" + "=" * 60)
print("PCA RECONSTRUCTION CHECK")
print("=" * 60)

print(
    "\nPCA input features:",
    len(pca_input_columns)
)

print(
    "PCA components:",
    repaired_pca.n_components_
)

print(
    "Maximum component difference:",
    f"{max_pca_difference:.12f}"
)


# ---------------------------------------------------------
# Save the repaired PCA model.
# ---------------------------------------------------------

pca_path = os.path.join(
    MODEL_DIR,
    "pca_model.pkl"
)


with open(
    pca_path,
    "wb"
) as file:

    pickle.dump(
        repaired_pca,
        file
    )


print(
    f"\n✓ Repaired PCA saved:"
    f" {pca_path}"
)


# ---------------------------------------------------------
# Rebuild the scaler.
#
# We reconstruct the scaler from the original processed
# dataset using the exact 34 numerical feature columns.
#
# The scaler is fitted on the numerical feature matrix
# before PCA, while binary features remain unchanged.
# ---------------------------------------------------------

processed_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "travel_integrated_preprocessed.csv"
    )
)


# ---------------------------------------------------------
# Verify that all required numerical columns exist.
# ---------------------------------------------------------

missing_numerical = [

    feature
    for feature in numerical_feature_columns
    if feature not in processed_df.columns
]


if missing_numerical:

    raise ValueError(
        "Missing numerical features in processed dataset: "
        + str(missing_numerical)
    )


# ---------------------------------------------------------
# Extract numerical data.
# ---------------------------------------------------------

X_numerical = processed_df[
    numerical_feature_columns
].copy()


# ---------------------------------------------------------
# The scaler cannot be fitted on missing values.
#
# Verify the source data first.
# ---------------------------------------------------------

missing_numerical_values = (
    X_numerical.isna().sum()
)


if missing_numerical_values.sum() > 0:

    print(
        "\nWarning: numerical source data contains "
        "missing values."
    )

    print(
        missing_numerical_values[
            missing_numerical_values > 0
        ]
    )

    raise ValueError(
        "Cannot safely rebuild scaler until numerical "
        "missing values are handled."
    )


# ---------------------------------------------------------
# Create and fit the repaired StandardScaler.
# ---------------------------------------------------------

repaired_scaler = StandardScaler()


repaired_scaler.fit(
    X_numerical
)


# ---------------------------------------------------------
# Save the repaired scaler.
# ---------------------------------------------------------

scaler_path = os.path.join(
    MODEL_DIR,
    "feature_scaler.pkl"
)


with open(
    scaler_path,
    "wb"
) as file:

    pickle.dump(
        repaired_scaler,
        file
    )


print(
    f"✓ Repaired scaler saved:"
    f" {scaler_path}"
)


# ---------------------------------------------------------
# Final artifact information.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("REPAIRED ARTIFACT SUMMARY")
print("=" * 60)

print(
    "\nScaler features:",
    repaired_scaler.n_features_in_
)

print(
    "PCA input features:",
    repaired_pca.n_features_in_
)

print(
    "PCA components:",
    repaired_pca.n_components_
)

print(
    "\nSaved artifacts:"
)

print(
    f"✓ {scaler_path}"
)

print(
    f"✓ {pca_path}"
)

print("\n" + "=" * 60)
print("✓ SCALER AND PCA REPAIR COMPLETE")
print("=" * 60)

REPAIRING SCALER AND PCA ARTIFACTS

Original features: 47
Numerical features: 34
Binary features: 13
Redundant features: 2
Saved scaled dataset: (50, 48)
Saved PCA dataset: (50, 16)

PCA RECONSTRUCTION CHECK

PCA input features: 45
PCA components: 15
Maximum component difference: 0.000000000000

✓ Repaired PCA saved: ../models\pca_model.pkl

flight_count          42
min_flight_price      42
avg_flight_price      42
max_flight_price      42
avg_total_duration    42
avg_outbound_stops    42
avg_return_stops      42
hotel_count           11
room_count            11
min_hotel_price       11
avg_hotel_price       11
max_hotel_price       11
avg_allotment         11
dtype: int64


ValueError: Cannot safely rebuild scaler until numerical missing values are handled.

In [54]:
# ---------------------------------------------------------
# SCALER RECONSTRUCTION DIAGNOSTIC
#
# We already repaired PCA successfully.
#
# The scaler is different because the source numerical
# features contain missing flight/accommodation values,
# while the saved scaled dataset contains no missing values.
#
# Before rebuilding the scaler, this cell investigates the
# exact transformation used previously.
#
# We compare:
#
#   1. Min-Max scaling
#   2. Standard scaling
#
# and several common missing-value strategies:
#
#   - fill with 0
#   - fill with median
#
# The saved model_features_scaled.csv is treated as the
# ground truth.
# ---------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler
)


# ---------------------------------------------------------
# File paths.
# ---------------------------------------------------------

DATA_DIR = "../data/cleaned"


processed_path = (
    f"{DATA_DIR}/travel_integrated_preprocessed.csv"
)

scaled_path = (
    f"{DATA_DIR}/model_features_scaled.csv"
)


# ---------------------------------------------------------
# Load the two datasets.
# ---------------------------------------------------------

processed_df = pd.read_csv(
    processed_path
)

saved_scaled_df = pd.read_csv(
    scaled_path
)


# ---------------------------------------------------------
# Load feature metadata.
# ---------------------------------------------------------

with open(
    "../models/feature_columns.pkl",
    "rb"
) as file:

    original_feature_columns = pickle.load(
        file
    )


with open(
    "../models/numerical_feature_columns.pkl",
    "rb"
) as file:

    numerical_feature_columns = pickle.load(
        file
    )


# ---------------------------------------------------------
# The saved scaled dataset contains destination plus the
# 47 original model features.
#
# We only investigate the 34 numerical features because
# those are the features handled by the scaler.
# ---------------------------------------------------------

raw_numeric = processed_df[
    numerical_feature_columns
].copy()


saved_numeric = saved_scaled_df[
    numerical_feature_columns
].copy()


print("=" * 60)
print("SCALER RECONSTRUCTION DIAGNOSTIC")
print("=" * 60)

print(
    "\nNumerical features:",
    len(numerical_feature_columns)
)

print(
    "Processed dataset:",
    raw_numeric.shape
)

print(
    "Saved scaled dataset:",
    saved_numeric.shape
)


# ---------------------------------------------------------
# Test several possible preprocessing strategies.
#
# We compare only values that were originally present.
#
# This prevents missing values from falsely affecting the
# comparison.
# ---------------------------------------------------------

strategies = {}


# ---------------------------------------------------------
# Strategy 1:
# Fill missing values with zero, then Min-Max scale.
# ---------------------------------------------------------

zero_filled = raw_numeric.fillna(0)

minmax_zero = MinMaxScaler()

minmax_zero_values = (
    minmax_zero.fit_transform(
        zero_filled
    )
)

strategies[
    "MinMax + zero fill"
] = minmax_zero_values


# ---------------------------------------------------------
# Strategy 2:
# Fill missing values with column median, then Min-Max.
# ---------------------------------------------------------

median_filled = raw_numeric.copy()

for column in numerical_feature_columns:

    median_value = (
        median_filled[column]
        .median()
    )

    median_filled[column] = (
        median_filled[column]
        .fillna(median_value)
    )


minmax_median = MinMaxScaler()

minmax_median_values = (
    minmax_median.fit_transform(
        median_filled
    )
)

strategies[
    "MinMax + median fill"
] = minmax_median_values


# ---------------------------------------------------------
# Strategy 3:
# Fill missing values with zero, then StandardScaler.
# ---------------------------------------------------------

standard_zero = StandardScaler()

standard_zero_values = (
    standard_zero.fit_transform(
        zero_filled
    )
)

strategies[
    "StandardScaler + zero fill"
] = standard_zero_values


# ---------------------------------------------------------
# Strategy 4:
# Fill missing values with median, then StandardScaler.
# ---------------------------------------------------------

standard_median = StandardScaler()

standard_median_values = (
    standard_median.fit_transform(
        median_filled
    )
)

strategies[
    "StandardScaler + median fill"
] = standard_median_values


# ---------------------------------------------------------
# Calculate reconstruction error.
#
# Only originally non-missing values are compared.
# ---------------------------------------------------------

results = []


for strategy_name, predicted_values in (
    strategies.items()
):

    feature_errors = []

    for index, column in enumerate(
        numerical_feature_columns
    ):

        # Identify values that existed in the original
        # processed dataset.
        valid_mask = (
            raw_numeric[column]
            .notna()
        )

        original_values = (
            saved_numeric.loc[
                valid_mask,
                column
            ].values
        )

        predicted_column = (
            predicted_values[
                valid_mask,
                index
            ]
        )

        if len(original_values) == 0:

            continue

        error = np.max(
            np.abs(
                original_values
                - predicted_column
            )
        )

        feature_errors.append(
            error
        )

    if feature_errors:

        results.append({

            "strategy":
                strategy_name,

            "maximum_error":
                max(feature_errors),

            "mean_feature_error":
                np.mean(feature_errors),

            "features_with_near_zero_error":
                sum(
                    error < 1e-9
                    for error in feature_errors
                )
        })


# ---------------------------------------------------------
# Display comparison.
# ---------------------------------------------------------

diagnostic_df = pd.DataFrame(
    results
).sort_values(
    "maximum_error"
)


print("\n" + "=" * 60)
print("SCALING STRATEGY COMPARISON")
print("=" * 60)

print(
    diagnostic_df.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Determine the best candidate.
# ---------------------------------------------------------

best_strategy = (
    diagnostic_df.iloc[0]
)


print("\n" + "=" * 60)
print("BEST MATCH")
print("=" * 60)

print(
    "\nStrategy:",
    best_strategy[
        "strategy"
    ]
)

print(
    "Maximum error:",
    best_strategy[
        "maximum_error"
    ]
)

print(
    "Mean feature error:",
    best_strategy[
        "mean_feature_error"
    ]
)

print(
    "Features with near-zero error:",
    int(
        best_strategy[
            "features_with_near_zero_error"
        ]
    ),
    "/",
    len(numerical_feature_columns)
)


print("\n" + "=" * 60)
print("END OF SCALER DIAGNOSTIC")
print("=" * 60)

SCALER RECONSTRUCTION DIAGNOSTIC

Numerical features: 34
Processed dataset: (50, 34)
Saved scaled dataset: (50, 34)

SCALING STRATEGY COMPARISON
                    strategy  maximum_error  mean_feature_error  features_with_near_zero_error
  StandardScaler + zero fill   8.881784e-16        3.000051e-16                             34
StandardScaler + median fill   4.837422e+00        6.003194e-01                             22
        MinMax + median fill   6.576250e+00        2.756605e+00                              0
          MinMax + zero fill   6.576250e+00        2.756605e+00                              0

BEST MATCH

Strategy: StandardScaler + zero fill
Maximum error: 8.881784197001252e-16
Mean feature error: 3.000051187865772e-16
Features with near-zero error: 34 / 34

END OF SCALER DIAGNOSTIC


In [55]:
# ---------------------------------------------------------
# REBUILD THE CORRUPTED FEATURE SCALER
#
# The diagnostic proved that the original preprocessing
# used:
#
#     Missing numerical values → 0
#     Then StandardScaler
#
# The reconstructed scaler should reproduce the existing
# model_features_scaled.csv exactly.
# ---------------------------------------------------------

import os
import pickle

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler


MODEL_DIR = "../models"
DATA_DIR = "../data/cleaned"


# ---------------------------------------------------------
# Load the healthy numerical-feature metadata.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "numerical_feature_columns.pkl"
    ),
    "rb"
) as file:

    numerical_feature_columns = pickle.load(
        file
    )


# ---------------------------------------------------------
# Load the original processed dataset.
# ---------------------------------------------------------

processed_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "travel_integrated_preprocessed.csv"
    )
)


# ---------------------------------------------------------
# Load the existing scaled dataset.
#
# This is our ground truth for validating the rebuilt
# scaler.
# ---------------------------------------------------------

saved_scaled_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "model_features_scaled.csv"
    )
)


# ---------------------------------------------------------
# Extract numerical features and apply the exact missing
# value strategy discovered by our diagnostic.
# ---------------------------------------------------------

X_numerical = (
    processed_df[
        numerical_feature_columns
    ]
    .fillna(0)
)


# ---------------------------------------------------------
# Create and fit the StandardScaler.
# ---------------------------------------------------------

repaired_scaler = StandardScaler()

repaired_scaler.fit(
    X_numerical
)


# ---------------------------------------------------------
# Transform the processed numerical data.
# ---------------------------------------------------------

reconstructed_scaled = (
    repaired_scaler.transform(
        X_numerical
    )
)


# ---------------------------------------------------------
# Compare the reconstructed values with the saved scaled
# dataset.
# ---------------------------------------------------------

saved_scaled_values = (
    saved_scaled_df[
        numerical_feature_columns
    ].values
)


maximum_difference = np.max(
    np.abs(
        reconstructed_scaled
        - saved_scaled_values
    )
)


mean_difference = np.mean(
    np.abs(
        reconstructed_scaled
        - saved_scaled_values
    )
)


# ---------------------------------------------------------
# Count features that reproduce the saved values.
# ---------------------------------------------------------

feature_errors = []

for index in range(
    len(numerical_feature_columns)
):

    error = np.max(
        np.abs(
            reconstructed_scaled[:, index]
            -
            saved_scaled_values[:, index]
        )
    )

    feature_errors.append(
        error
    )


matching_features = sum(
    error < 1e-9
    for error in feature_errors
)


# ---------------------------------------------------------
# Validate reconstruction before saving.
# ---------------------------------------------------------

print("=" * 60)
print("FEATURE SCALER RECONSTRUCTION")
print("=" * 60)

print(
    "\nNumerical features:",
    len(numerical_feature_columns)
)

print(
    "Scaler expects:",
    repaired_scaler.n_features_in_
)

print(
    "Maximum difference:",
    f"{maximum_difference:.15f}"
)

print(
    "Mean difference:",
    f"{mean_difference:.15f}"
)

print(
    "Features matching:",
    f"{matching_features} / "
    f"{len(numerical_feature_columns)}"
)


# ---------------------------------------------------------
# Safety check.
#
# We expect floating-point differences only around machine
# precision.
# ---------------------------------------------------------

if maximum_difference > 1e-9:

    raise ValueError(
        "Reconstructed scaler does not reproduce "
        "the saved scaled dataset."
    )


if matching_features != len(
    numerical_feature_columns
):

    raise ValueError(
        "Not all numerical features reproduced "
        "the saved scaling."
    )


# ---------------------------------------------------------
# Save the repaired scaler.
# ---------------------------------------------------------

scaler_path = os.path.join(
    MODEL_DIR,
    "feature_scaler.pkl"
)


with open(
    scaler_path,
    "wb"
) as file:

    pickle.dump(
        repaired_scaler,
        file
    )


print(
    f"\n✓ Repaired scaler saved:"
    f" {scaler_path}"
)


# ---------------------------------------------------------
# Confirm that the saved scaler can be immediately loaded.
# ---------------------------------------------------------

with open(
    scaler_path,
    "rb"
) as file:

    test_scaler = pickle.load(
        file
    )


print(
    "\n✓ Repaired scaler successfully reloaded."
)

print(
    "Reloaded scaler expects:",
    test_scaler.n_features_in_,
    "features"
)


print("\n" + "=" * 60)
print("✓ FEATURE SCALER REPAIR COMPLETE")
print("=" * 60)

FEATURE SCALER RECONSTRUCTION

Numerical features: 34
Scaler expects: 34
Maximum difference: 0.000000000000001
Mean difference: 0.000000000000000
Features matching: 34 / 34

✓ Repaired scaler saved: ../models\feature_scaler.pkl

✓ Repaired scaler successfully reloaded.
Reloaded scaler expects: 34 features

✓ FEATURE SCALER REPAIR COMPLETE


In [57]:
# ---------------------------------------------------------
# FINAL END-TO-END ARTIFACT CONSISTENCY TEST — CORRECTED
#
# IMPORTANT:
#
# model_features_scaled.csv already contains:
#
#   34 scaled numerical features
#   13 encoded binary features
#
# Therefore, we should NOT rebuild binary features from
# travel_integrated_preprocessed.csv.
#
# Instead, we use the saved model_features_scaled.csv as
# the authoritative 47-feature processed representation.
#
# This avoids incorrectly trying to find columns such as
# weather_condition_Clear in the original categorical data.
# ---------------------------------------------------------

import os
import pickle

import numpy as np
import pandas as pd


MODEL_DIR = "../models"
DATA_DIR = "../data/cleaned"


# ---------------------------------------------------------
# Load saved model artifacts.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "feature_columns.pkl"
    ),
    "rb"
) as file:

    feature_columns = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "numerical_feature_columns.pkl"
    ),
    "rb"
) as file:

    numerical_columns = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "binary_feature_columns.pkl"
    ),
    "rb"
) as file:

    binary_columns = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "redundant_features.pkl"
    ),
    "rb"
) as file:

    redundant_features = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "feature_scaler.pkl"
    ),
    "rb"
) as file:

    scaler = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "pca_model.pkl"
    ),
    "rb"
) as file:

    pca_model = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "destination_similarity.pkl"
    ),
    "rb"
) as file:

    similarity_matrix = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_destinations.pkl"
    ),
    "rb"
) as file:

    recommendation_destinations = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_feature_columns.pkl"
    ),
    "rb"
) as file:

    recommendation_columns = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_config.pkl"
    ),
    "rb"
) as file:

    recommendation_config = pickle.load(file)


# ---------------------------------------------------------
# Load the saved complete model-feature dataset.
#
# This contains all 47 features, including the 13 binary
# one-hot encoded features.
# ---------------------------------------------------------

saved_scaled_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "model_features_scaled.csv"
    )
)


# ---------------------------------------------------------
# Load the saved PCA destination dataset.
# ---------------------------------------------------------

saved_pca_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "pca_destination_features.csv"
    )
)


print("=" * 60)
print("FINAL END-TO-END ARTIFACT CONSISTENCY TEST")
print("=" * 60)


# ---------------------------------------------------------
# 1. Validate feature metadata.
# ---------------------------------------------------------

print("\nArtifact dimensions:")

print(
    "Original features:",
    len(feature_columns)
)

print(
    "Numerical features:",
    len(numerical_columns)
)

print(
    "Binary features:",
    len(binary_columns)
)

print(
    "Redundant features:",
    len(redundant_features)
)

print(
    "Scaler input:",
    scaler.n_features_in_
)

print(
    "PCA input:",
    pca_model.n_features_in_
)

print(
    "PCA components:",
    pca_model.n_components_
)

print(
    "Similarity matrix:",
    similarity_matrix.shape
)

print(
    "Destinations:",
    len(recommendation_destinations)
)


# ---------------------------------------------------------
# 2. Validate expected feature counts.
# ---------------------------------------------------------

if len(feature_columns) != 47:

    raise ValueError(
        "Expected 47 original features."
    )


if len(numerical_columns) != 34:

    raise ValueError(
        "Expected 34 numerical features."
    )


if len(binary_columns) != 13:

    raise ValueError(
        "Expected 13 binary features."
    )


if len(redundant_features) != 2:

    raise ValueError(
        "Expected 2 redundant features."
    )


if scaler.n_features_in_ != 34:

    raise ValueError(
        "Scaler should expect 34 numerical features."
    )


if pca_model.n_features_in_ != 45:

    raise ValueError(
        "PCA should expect 45 input features."
    )


if pca_model.n_components_ != 15:

    raise ValueError(
        "PCA should contain 15 components."
    )


if similarity_matrix.shape != (50, 50):

    raise ValueError(
        "Similarity matrix should be 50 × 50."
    )


# ---------------------------------------------------------
# 3. Validate saved scaled dataset.
# ---------------------------------------------------------

expected_scaled_columns = (
    ["destination"]
    + feature_columns
)


if (
    saved_scaled_df.columns.tolist()
    != expected_scaled_columns
):

    raise ValueError(
        "Saved scaled dataset does not match "
        "feature_columns.pkl."
    )


if saved_scaled_df.isna().sum().sum() != 0:

    raise ValueError(
        "Saved scaled dataset contains missing values."
    )


# ---------------------------------------------------------
# 4. Reconstruct the exact 45-feature PCA matrix.
#
# We start with the saved 47-feature model representation
# because it already contains:
#
#   - scaled numerical features
#   - unchanged binary features
#
# Then remove the two redundant features.
# ---------------------------------------------------------

reconstructed_pca_input = (
    saved_scaled_df[
        feature_columns
    ]
    .drop(
        columns=redundant_features
    )
    .copy()
)


# ---------------------------------------------------------
# Verify PCA input dimensions.
# ---------------------------------------------------------

if reconstructed_pca_input.shape != (50, 45):

    raise ValueError(
        "Reconstructed PCA input should be "
        "(50, 45), found "
        + str(
            reconstructed_pca_input.shape
        )
    )


# ---------------------------------------------------------
# 5. Apply the repaired PCA model.
# ---------------------------------------------------------

reconstructed_pca_values = (
    pca_model.transform(
        reconstructed_pca_input
    )
)


# ---------------------------------------------------------
# 6. Compare reconstructed PCA values with the saved
# PCA destination dataset.
#
# PCA signs can theoretically be reversed while representing
# the same component, so both orientations are considered.
# ---------------------------------------------------------

pca_errors = []


for index, column in enumerate(
    recommendation_columns
):

    saved_values = (
        saved_pca_df[
            column
        ].values
    )

    reconstructed_values = (
        reconstructed_pca_values[
            :,
            index
        ]
    )

    direct_error = np.max(
        np.abs(
            saved_values
            -
            reconstructed_values
        )
    )

    flipped_error = np.max(
        np.abs(
            saved_values
            +
            reconstructed_values
        )
    )

    pca_errors.append(
        min(
            direct_error,
            flipped_error
        )
    )


maximum_pca_error = max(
    pca_errors
)


# ---------------------------------------------------------
# 7. Validate destination ordering.
# ---------------------------------------------------------

saved_destinations = (
    saved_pca_df[
        "destination"
    ].tolist()
)


if (
    saved_destinations
    !=
    list(recommendation_destinations)
):

    raise ValueError(
        "Destination ordering does not match."
    )


# ---------------------------------------------------------
# 8. Validate similarity matrix.
# ---------------------------------------------------------

similarity_array = np.asarray(
    similarity_matrix
)


if not np.allclose(
    similarity_array,
    similarity_array.T,
    atol=1e-10
):

    raise ValueError(
        "Similarity matrix is not symmetric."
    )


# ---------------------------------------------------------
# 9. Validate recommendation configuration.
# ---------------------------------------------------------

if (
    recommendation_config[
        "travel_factor_weight"
    ]
    != 0.60
):

    raise ValueError(
        "Unexpected travel-factor weight."
    )


if (
    recommendation_config[
        "interest_weight"
    ]
    != 0.40
):

    raise ValueError(
        "Unexpected interest weight."
    )


if (
    recommendation_config[
        "hybrid_preference_weight"
    ]
    != 0.70
):

    raise ValueError(
        "Unexpected hybrid preference weight."
    )


if (
    recommendation_config[
        "hybrid_similarity_weight"
    ]
    != 0.30
):

    raise ValueError(
        "Unexpected hybrid similarity weight."
    )


# ---------------------------------------------------------
# 10. Final PCA consistency check.
# ---------------------------------------------------------

if maximum_pca_error > 1e-9:

    raise ValueError(
        "Reconstructed PCA does not match the "
        "saved PCA destination dataset."
    )


# ---------------------------------------------------------
# FINAL SUCCESS REPORT
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("END-TO-END VALIDATION RESULTS")
print("=" * 60)

print(
    "\nSaved model-feature dataset:",
    saved_scaled_df.shape
)

print(
    "Reconstructed PCA input:",
    reconstructed_pca_input.shape
)

print(
    "Reconstructed PCA output:",
    reconstructed_pca_values.shape
)

print(
    "Saved PCA dataset:",
    saved_pca_df.shape
)

print(
    "Maximum PCA reconstruction error:",
    f"{maximum_pca_error:.15f}"
)

print(
    "Similarity matrix symmetric: ✓"
)

print(
    "Destination ordering: ✓"
)

print(
    "Recommendation configuration: ✓"
)

print("\n" + "=" * 60)
print("✓ END-TO-END ARTIFACT CONSISTENCY TEST PASSED")
print("=" * 60)

print(
    "\nAll saved recommendation artifacts are "
    "internally consistent."
)

print(
    "\nThe model is ready for evaluation."
)

FINAL END-TO-END ARTIFACT CONSISTENCY TEST

Artifact dimensions:
Original features: 47
Numerical features: 34
Binary features: 13
Redundant features: 2
Scaler input: 34
PCA input: 45
PCA components: 15
Similarity matrix: (50, 50)
Destinations: 50

END-TO-END VALIDATION RESULTS

Saved model-feature dataset: (50, 48)
Reconstructed PCA input: (50, 45)
Reconstructed PCA output: (50, 15)
Saved PCA dataset: (50, 16)
Maximum PCA reconstruction error: 0.000000000000041
Similarity matrix symmetric: ✓
Destination ordering: ✓
Recommendation configuration: ✓

✓ END-TO-END ARTIFACT CONSISTENCY TEST PASSED

All saved recommendation artifacts are internally consistent.

The model is ready for evaluation.


In [58]:
# ---------------------------------------------------------
# FIRST REAL RECOMMENDATION PREDICTION
#
# We will load the saved recommendation artifacts and use
# the sample user profile to generate personalized
# recommendations.
#
# This is an actual model prediction, not just a validation.
# ---------------------------------------------------------

import os
import pickle
import pandas as pd
import numpy as np


MODEL_DIR = "../models"


# ---------------------------------------------------------
# Load the saved recommendation engine artifacts.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "destination_similarity.pkl"
    ),
    "rb"
) as file:

    destination_similarity = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_destinations.pkl"
    ),
    "rb"
) as file:

    recommendation_destinations = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_config.pkl"
    ),
    "rb"
) as file:

    recommendation_config = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "destination_profile_scores.pkl"
    ),
    "rb"
) as file:

    destination_profile_scores = pickle.load(file)


# ---------------------------------------------------------
# Load the destination preference scores that were created
# earlier in the model-building process.
# ---------------------------------------------------------

preference_scores_path = (
    "../data/cleaned/"
    "integrated_travel_dataset.csv"
)

processed_df = pd.read_csv(
    preference_scores_path
)


# ---------------------------------------------------------
# Use the preference score tables already generated in
# memory if available.
#
# If they are not available after a kernel restart, we will
# reconstruct the required scores from the saved artifacts
# in the next step.
# ---------------------------------------------------------

print("=" * 60)
print("FIRST REAL RECOMMENDATION PREDICTION")
print("=" * 60)

print(
    "\nDestinations available:",
    len(recommendation_destinations)
)

print(
    "Similarity matrix:",
    destination_similarity.shape
)

print(
    "Recommendation configuration loaded: ✓"
)


# ---------------------------------------------------------
# Define the user profile.
#
# Values range from 0 to 1:
#
# 0.0 = not important
# 1.0 = extremely important
# ---------------------------------------------------------

user_preferences = {

    # Strongly cares about affordability.
    "budget": 0.90,

    # Moderately cares about flights.
    "flight": 0.60,

    # Moderately cares about accommodation.
    "accommodation": 0.60,

    # Weather is highly important.
    "weather": 0.80,

    # General destination characteristics are important.
    "destination_characteristics": 0.80
}


# ---------------------------------------------------------
# Define destination interests.
# ---------------------------------------------------------

user_interests = {

    # Strong nature preference.
    "nature": 1.00,

    # Moderate sightseeing preference.
    "sightseeing": 0.50,

    # Low coastal/water preference.
    "water_coastal": 0.20,

    # Moderate wildlife preference.
    "wildlife": 0.60
}


print("\n" + "=" * 60)
print("USER PROFILE")
print("=" * 60)

print("\nTravel preferences:")

for key, value in user_preferences.items():

    print(
        f"  {key:30s} {value:.2f}"
    )


print("\nDestination interests:")

for key, value in user_interests.items():

    print(
        f"  {key:30s} {value:.2f}"
    )


# ---------------------------------------------------------
# Verify the saved profile-score structure.
# ---------------------------------------------------------

if isinstance(
    destination_profile_scores,
    pd.DataFrame
):

    profile_df = (
        destination_profile_scores
        .copy()
    )

else:

    profile_df = pd.DataFrame(
        destination_profile_scores
    )


print("\n" + "=" * 60)
print("PROFILE SCORE DATA")
print("=" * 60)

print(
    "\nRows:",
    len(profile_df)
)

print(
    "Columns:",
    profile_df.columns.tolist()
)


# ---------------------------------------------------------
# Calculate personalized interest score.
#
# The destination profile scores represent:
#
#   nature
#   sightseeing
#   water_coastal
#   wildlife
#
# We calculate the weighted average according to the user's
# stated interests.
# ---------------------------------------------------------

interest_score = pd.Series(
    0.0,
    index=profile_df.index
)


total_interest_weight = sum(
    user_interests.values()
)


for profile, weight in user_interests.items():

    score_column = (
        f"{profile}_score"
    )

    if score_column not in profile_df.columns:

        raise ValueError(
            f"Missing profile score column: "
            f"{score_column}"
        )

    interest_score += (
        profile_df[
            score_column
        ].fillna(0)
        * weight
    )


interest_score = (
    interest_score
    / total_interest_weight
)


# ---------------------------------------------------------
# Create the personalized interest result table.
# ---------------------------------------------------------

interest_results = pd.DataFrame({

    "destination":
        profile_df["destination"],

    "personalized_interest_score":
        interest_score

})


# ---------------------------------------------------------
# Sort destinations by personalized interest score.
# ---------------------------------------------------------

interest_results = (
    interest_results
    .sort_values(
        "personalized_interest_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Display the prediction.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("FIRST MODEL PREDICTION")
print("=" * 60)

print(
    "\nTop 10 destinations based on "
    "user interests:"
)

print(
    interest_results.head(10).to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Basic prediction validation.
# ---------------------------------------------------------

if len(
    interest_results
) != len(
    recommendation_destinations
):

    raise ValueError(
        "Prediction destination count does not "
        "match recommendation destination count."
    )


if interest_results[
    "personalized_interest_score"
].isna().any():

    raise ValueError(
        "Prediction contains missing scores."
    )


print("\n" + "=" * 60)
print("✓ FIRST RECOMMENDATION PREDICTION COMPLETED")
print("=" * 60)

FIRST REAL RECOMMENDATION PREDICTION

Destinations available: 50
Similarity matrix: (50, 50)
Recommendation configuration loaded: ✓

USER PROFILE

Travel preferences:
  budget                         0.90
  flight                         0.60
  accommodation                  0.60
  weather                        0.80
  destination_characteristics    0.80

Destination interests:
  nature                         1.00
  sightseeing                    0.50
  water_coastal                  0.20
  wildlife                       0.60

PROFILE SCORE DATA

Rows: 50
Columns: ['destination', 'nature_score', 'sightseeing_score', 'water_coastal_score', 'wildlife_score']

FIRST MODEL PREDICTION

Top 10 destinations based on user interests:
destination  personalized_interest_score
     Mumbai                     0.522352
     Manali                     0.487155
     Jaipur                     0.410318
      Delhi                     0.408991
  Bengaluru                     0.401584
      Kochi       

In [59]:
# ---------------------------------------------------------
# FULL HYBRID RECOMMENDATION PREDICTION
#
# This is the main prediction stage of the project.
#
# The final recommendation combines:
#
#   1. Personalized travel-factor preference score
#   2. Personalized destination-interest score
#   3. Destination similarity
#   4. Availability-aware weighting
#
# Final hybrid score:
#
#   70% personalized preference
#   30% destination similarity
#
# The result is a ranked list of recommended destinations.
# ---------------------------------------------------------

import os
import pickle
import numpy as np
import pandas as pd


MODEL_DIR = "../models"
DATA_DIR = "../data/cleaned"


# ---------------------------------------------------------
# Load saved recommendation artifacts.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "destination_similarity.pkl"
    ),
    "rb"
) as file:

    destination_similarity = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_destinations.pkl"
    ),
    "rb"
) as file:

    recommendation_destinations = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "destination_profile_scores.pkl"
    ),
    "rb"
) as file:

    destination_profile_scores = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "preference_groups.pkl"
    ),
    "rb"
) as file:

    preference_groups = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "preference_direction_map.pkl"
    ),
    "rb"
) as file:

    preference_direction_map = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_config.pkl"
    ),
    "rb"
) as file:

    recommendation_config = pickle.load(file)


# ---------------------------------------------------------
# Load the processed dataset.
#
# This contains the availability indicators and preference
# features required for personalized scoring.
# ---------------------------------------------------------

processed_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "integrated_travel_dataset.csv"
    )
)


# ---------------------------------------------------------
# USER PROFILE
#
# These values represent how important each travel factor
# is to this particular user.
# ---------------------------------------------------------

user_preferences = {

    "budget": 0.90,

    "flight": 0.60,

    "accommodation": 0.60,

    "weather": 0.80,

    "destination_characteristics": 0.80
}


# ---------------------------------------------------------
# USER INTERESTS
#
# These values represent the user's destination interests.
# ---------------------------------------------------------

user_interests = {

    "nature": 1.00,

    "sightseeing": 0.50,

    "water_coastal": 0.20,

    "wildlife": 0.60
}


print("=" * 60)
print("FULL HYBRID RECOMMENDATION PREDICTION")
print("=" * 60)


# ---------------------------------------------------------
# Construct destination profile DataFrame.
# ---------------------------------------------------------

profile_df = (
    destination_profile_scores
    .copy()
)


# ---------------------------------------------------------
# Make sure destination ordering is consistent.
# ---------------------------------------------------------

if (
    profile_df["destination"].tolist()
    !=
    list(recommendation_destinations)
):

    raise ValueError(
        "Destination ordering mismatch between "
        "profile scores and recommendation model."
    )


# ---------------------------------------------------------
# Calculate personalized destination-interest score.
# ---------------------------------------------------------

interest_scores = pd.Series(
    0.0,
    index=profile_df.index
)


total_interest_weight = sum(
    user_interests.values()
)


for profile, weight in user_interests.items():

    score_column = (
        f"{profile}_score"
    )

    interest_scores += (
        profile_df[
            score_column
        ].fillna(0)
        * weight
    )


interest_scores = (
    interest_scores
    / total_interest_weight
)


# ---------------------------------------------------------
# Create interest-score DataFrame.
# ---------------------------------------------------------

interest_result = pd.DataFrame({

    "destination":
        profile_df["destination"],

    "personalized_interest_score":
        interest_scores

})


# ---------------------------------------------------------
# Calculate availability-aware travel-factor scores.
#
# Preference scores are derived from the processed dataset.
#
# For each group:
#
#   available → contribute to score
#   unavailable → excluded from denominator
#
# This prevents missing flight/accommodation data from
# unfairly penalizing destinations.
# ---------------------------------------------------------

group_scores = pd.DataFrame({

    "destination":
        processed_df["destination"]

})


# ---------------------------------------------------------
# Helper function for availability-aware group scoring.
# ---------------------------------------------------------

def calculate_group_score(
    dataframe,
    features,
    user_weight,
    direction_map
):
    """
    Calculate the weighted preference score for one
    preference group.

    Missing feature values are ignored.

    Each feature is first converted so that:
        higher value = better

    This allows all features to contribute in the same
    direction.
    """

    score_values = pd.Series(
        0.0,
        index=dataframe.index
    )

    active_weights = pd.Series(
        0.0,
        index=dataframe.index
    )

    # -----------------------------------------------------
    # Process every feature in this preference group.
    # -----------------------------------------------------

    for feature in features:

        if feature not in dataframe.columns:

            continue

        values = dataframe[
            feature
        ].copy()

        # -------------------------------------------------
        # Skip features with no usable values.
        # -------------------------------------------------

        valid_mask = values.notna()

        if not valid_mask.any():

            continue

        # -------------------------------------------------
        # Convert lower-is-better features so that higher
        # always means better.
        # -------------------------------------------------

        if direction_map.get(
            feature
        ) == "lower":

            values = 1 - values

        # -------------------------------------------------
        # Add this feature's contribution.
        # -------------------------------------------------

        score_values.loc[
            valid_mask
        ] += (
            values.loc[
                valid_mask
            ]
            * user_weight
        )

        active_weights.loc[
            valid_mask
        ] += user_weight

    # -----------------------------------------------------
    # Normalize using only available features.
    # -----------------------------------------------------

    result = pd.Series(
        np.nan,
        index=dataframe.index
    )

    valid_rows = (
        active_weights > 0
    )

    result.loc[
        valid_rows
    ] = (
        score_values.loc[
            valid_rows
        ]
        /
        active_weights.loc[
            valid_rows
        ]
    )

    return result


# ---------------------------------------------------------
# Calculate all five travel-factor group scores.
# ---------------------------------------------------------

for group_name, features in (
    preference_groups.items()
):

    group_scores[
        f"{group_name}_score"
    ] = calculate_group_score(

        processed_df,

        features,

        user_preferences[
            group_name
        ],

        preference_direction_map
    )


# ---------------------------------------------------------
# Combine the five travel-factor scores.
#
# Availability-aware normalization means only groups with
# usable data contribute to each destination's score.
# ---------------------------------------------------------

travel_score_columns = [

    "budget_score",

    "flight_score",

    "accommodation_score",

    "weather_score",

    "destination_characteristics_score"
]


weighted_sum = pd.Series(
    0.0,
    index=group_scores.index
)


total_active_weight = pd.Series(
    0.0,
    index=group_scores.index
)


for group_name in user_preferences:

    score_column = (
        f"{group_name}_score"
    )

    user_weight = (
        user_preferences[
            group_name
        ]
    )

    valid_mask = (
        group_scores[
            score_column
        ].notna()
    )

    weighted_sum.loc[
        valid_mask
    ] += (
        group_scores.loc[
            valid_mask,
            score_column
        ]
        * user_weight
    )

    total_active_weight.loc[
        valid_mask
    ] += user_weight


personalized_preference_score = (
    weighted_sum
    /
    total_active_weight
)


# ---------------------------------------------------------
# Add the personalized preference score.
# ---------------------------------------------------------

group_scores[
    "personalized_preference_score"
] = personalized_preference_score


# ---------------------------------------------------------
# Add personalized interest score.
# ---------------------------------------------------------

group_scores = group_scores.merge(

    interest_result,

    on="destination",

    how="left"
)


# ---------------------------------------------------------
# Calculate the base personalized score.
#
# The saved configuration specifies:
#
#   travel factors = 60%
#   interests      = 40%
# ---------------------------------------------------------

travel_factor_weight = (
    recommendation_config[
        "travel_factor_weight"
    ]
)


interest_weight = (
    recommendation_config[
        "interest_weight"
    ]
)


group_scores[
    "base_personalized_score"
] = (

    group_scores[
        "personalized_preference_score"
    ]
    * travel_factor_weight

    +

    group_scores[
        "personalized_interest_score"
    ]
    * interest_weight
)


# ---------------------------------------------------------
# Similarity component.
#
# For a pure preference-driven prediction there is no
# reference destination, so we use the personalized score
# as the main ranking signal.
#
# We will test destination-to-destination similarity
# separately next.
#
# For this first full-user prediction, similarity is
# therefore anchored to the strongest personalized
# destinations.
# ---------------------------------------------------------

# Sort by personalized score first.
group_scores = (
    group_scores
    .sort_values(
        "base_personalized_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Generate a similarity-enhanced score.
#
# Each destination receives similarity support from the
# strongest personalized destinations.
#
# This allows the model to recommend destinations that are
# structurally similar to destinations already favored by
# the user profile.
# ---------------------------------------------------------

preference_vector = (
    group_scores[
        "base_personalized_score"
    ].values
)


destination_index = {
    destination: index
    for index, destination
    in enumerate(
        recommendation_destinations
    )
}


similarity_array = np.asarray(
    destination_similarity
)


similarity_scores = []


# ---------------------------------------------------------
# Calculate similarity support for each destination.
# ---------------------------------------------------------

for destination in (
    group_scores[
        "destination"
    ]
):

    index = destination_index[
        destination
    ]

    similarities = (
        similarity_array[
            index
        ]
    )

    # -----------------------------------------------------
    # Use the strongest personalized destinations as the
    # reference neighbourhood.
    # -----------------------------------------------------

    top_indices = np.argsort(
        preference_vector
    )[::-1][:5]

    top_weights = (
        preference_vector[
            top_indices
        ]
    )

    top_similarities = (
        similarities[
            top_indices
        ]
    )

    weight_sum = (
        top_weights.sum()
    )

    if weight_sum > 0:

        similarity_score = (
            np.dot(
                top_similarities,
                top_weights
            )
            /
            weight_sum
        )

    else:

        similarity_score = 0.0

    similarity_scores.append(
        similarity_score
    )


group_scores[
    "similarity_score"
] = similarity_scores


# ---------------------------------------------------------
# Calculate final hybrid recommendation score.
#
# Saved model configuration:
#
#   preference = 0.70
#   similarity = 0.30
# ---------------------------------------------------------

hybrid_preference_weight = (
    recommendation_config[
        "hybrid_preference_weight"
    ]
)


hybrid_similarity_weight = (
    recommendation_config[
        "hybrid_similarity_weight"
    ]
)


group_scores[
    "hybrid_score"
] = (

    group_scores[
        "base_personalized_score"
    ]
    * hybrid_preference_weight

    +

    group_scores[
        "similarity_score"
    ]
    * hybrid_similarity_weight
)


# ---------------------------------------------------------
# Final ranking.
# ---------------------------------------------------------

final_recommendations = (
    group_scores[
        [
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "base_personalized_score",
            "similarity_score",
            "hybrid_score"
        ]
    ]
    .sort_values(
        "hybrid_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Add recommendation rank.
# ---------------------------------------------------------

final_recommendations.insert(
    0,
    "rank",
    range(
        1,
        len(
            final_recommendations
        ) + 1
    )
)


# ---------------------------------------------------------
# Display the final prediction.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("TOP 10 HYBRID RECOMMENDATIONS")
print("=" * 60)

print(
    final_recommendations
    .head(10)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Final prediction checks.
# ---------------------------------------------------------

if len(
    final_recommendations
) != 50:

    raise ValueError(
        "Expected recommendations for all "
        "50 destinations."
    )


if final_recommendations[
    "hybrid_score"
].isna().any():

    raise ValueError(
        "Hybrid prediction contains missing scores."
    )


print("\n" + "=" * 60)
print("✓ FULL HYBRID PREDICTION COMPLETED")
print("=" * 60)

print(
    "\nRecommended destination:",
    final_recommendations.iloc[0][
        "destination"
    ]
)

print(
    "Final hybrid score:",
    round(
        final_recommendations.iloc[0][
            "hybrid_score"
        ],
        4
    )
)

FULL HYBRID RECOMMENDATION PREDICTION

TOP 10 HYBRID RECOMMENDATIONS
 rank   destination  personalized_preference_score  personalized_interest_score  base_personalized_score  similarity_score  hybrid_score
    1        Ranchi                     716.337532                     0.189931               429.878492          0.024175    300.922197
    2        Ladakh                     715.555065                     0.095445               429.371217          0.051435    300.575283
    3     Kaziranga                     712.391753                     0.237083               427.529885          0.118363    299.306428
    4      Pahalgam                     712.123831                     0.100956               427.314681         -0.116853    299.085221
    5       Wayanad                     711.400974                     0.221637               426.929239          0.007648    298.852762
    6 Mahabalipuram                     710.568961                     0.271323               426.449906     

In [60]:
# ---------------------------------------------------------
# REBUILD NORMALIZED PREFERENCE SCORES FOR PREDICTION
#
# IMPORTANT:
# The previous prediction accidentally used raw values from
# integrated_travel_dataset.csv.
#
# Here we rebuild the preference representation correctly:
#
#     raw feature
#          ↓
#     normalize to [0, 1]
#          ↓
#     higher/lower direction correction
#          ↓
#     availability-aware group score
#
# This creates the same type of 0–1 preference scores that
# were used earlier in the model-building process.
# ---------------------------------------------------------

import os
import pickle

import numpy as np
import pandas as pd


DATA_DIR = "../data/cleaned"
MODEL_DIR = "../models"


# ---------------------------------------------------------
# Load the original integrated dataset.
# ---------------------------------------------------------

processed_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "integrated_travel_dataset.csv"
    )
)


# ---------------------------------------------------------
# Load preference configuration artifacts.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "preference_groups.pkl"
    ),
    "rb"
) as file:

    preference_groups = pickle.load(file)


with open(
    os.path.join(
        MODEL_DIR,
        "preference_direction_map.pkl"
    ),
    "rb"
) as file:

    preference_direction_map = pickle.load(file)


print("=" * 60)
print("REBUILDING NORMALIZED PREFERENCE SCORES")
print("=" * 60)


# ---------------------------------------------------------
# User preferences.
# ---------------------------------------------------------

user_preferences = {

    "budget": 0.90,

    "flight": 0.60,

    "accommodation": 0.60,

    "weather": 0.80,

    "destination_characteristics": 0.80
}


# ---------------------------------------------------------
# Normalize a single feature to [0, 1].
#
# Missing values remain missing because missing external
# information must be handled through availability-aware
# weighting rather than pretending that the value exists.
# ---------------------------------------------------------

def normalize_feature(
    series
):
    """
    Convert a numeric feature to the [0, 1] range.

    Missing values remain NaN.

    If every available value is identical, the feature is
    assigned 0.5 so that it does not artificially become
    either maximally good or maximally bad.
    """

    values = pd.to_numeric(
        series,
        errors="coerce"
    )

    minimum = values.min()

    maximum = values.max()

    if pd.isna(minimum) or pd.isna(maximum):

        return pd.Series(
            np.nan,
            index=series.index
        )

    if maximum == minimum:

        result = pd.Series(
            np.nan,
            index=series.index
        )

        result.loc[
            values.notna()
        ] = 0.5

        return result

    return (
        values - minimum
    ) / (
        maximum - minimum
    )


# ---------------------------------------------------------
# Create normalized feature dataframe.
# ---------------------------------------------------------

normalized_features = pd.DataFrame(
    index=processed_df.index
)


# ---------------------------------------------------------
# Process every preference feature.
# ---------------------------------------------------------

all_preference_features = []

for features in preference_groups.values():

    all_preference_features.extend(
        features
    )


# Remove duplicates while preserving order.
all_preference_features = list(
    dict.fromkeys(
        all_preference_features
    )
)


for feature in all_preference_features:

    if feature not in processed_df.columns:

        raise ValueError(
            f"Preference feature missing from "
            f"dataset: {feature}"
        )

    normalized_features[
        feature
    ] = normalize_feature(
        processed_df[
            feature
        ]
    )


# ---------------------------------------------------------
# Direction correction.
#
# After this step:
#
#     higher score = better for the traveler
#
# regardless of whether the original feature was:
#
#     higher-is-better
#     lower-is-better
# ---------------------------------------------------------

for feature in all_preference_features:

    direction = (
        preference_direction_map.get(
            feature
        )
    )

    if direction == "lower":

        valid_mask = (
            normalized_features[
                feature
            ].notna()
        )

        normalized_features.loc[
            valid_mask,
            feature
        ] = (
            1
            -
            normalized_features.loc[
                valid_mask,
                feature
            ]
        )


# ---------------------------------------------------------
# Validate normalized feature values.
# ---------------------------------------------------------

available_values = (
    normalized_features[
        all_preference_features
    ]
    .stack()
)


minimum_value = (
    available_values.min()
)

maximum_value = (
    available_values.max()
)


print(
    "\nNormalized feature range:"
)

print(
    "Minimum:",
    minimum_value
)

print(
    "Maximum:",
    maximum_value
)


if minimum_value < 0 or maximum_value > 1:

    raise ValueError(
        "Normalized preference values are outside [0, 1]."
    )


# ---------------------------------------------------------
# Calculate availability-aware preference scores.
#
# A missing value does NOT contribute to the denominator.
#
# Example:
#
#     budget available     → budget contributes
#     flight unavailable   → flight excluded
#     weather available   → weather contributes
#
# This prevents missing external data from becoming an
# artificial zero score.
# ---------------------------------------------------------

preference_scores = pd.DataFrame({

    "destination":
        processed_df[
            "destination"
        ]

})


for group_name, features in (
    preference_groups.items()
):

    user_weight = (
        user_preferences[
            group_name
        ]
    )

    weighted_values = pd.Series(
        0.0,
        index=processed_df.index
    )

    active_weight = pd.Series(
        0.0,
        index=processed_df.index
    )

    for feature in features:

        values = (
            normalized_features[
                feature
            ]
        )

        valid_mask = values.notna()

        weighted_values.loc[
            valid_mask
        ] += (
            values.loc[
                valid_mask
            ]
            * user_weight
        )

        active_weight.loc[
            valid_mask
        ] += user_weight

    group_result = pd.Series(
        np.nan,
        index=processed_df.index
    )

    valid_rows = (
        active_weight > 0
    )

    group_result.loc[
        valid_rows
    ] = (
        weighted_values.loc[
            valid_rows
        ]
        /
        active_weight.loc[
            valid_rows
        ]
    )

    preference_scores[
        f"{group_name}_score"
    ] = group_result


# ---------------------------------------------------------
# Calculate overall personalized preference score.
#
# Again, unavailable groups are excluded from the
# denominator for each destination.
# ---------------------------------------------------------

group_score_columns = [

    "budget_score",

    "flight_score",

    "accommodation_score",

    "weather_score",

    "destination_characteristics_score"
]


weighted_total = pd.Series(
    0.0,
    index=processed_df.index
)


active_total = pd.Series(
    0.0,
    index=processed_df.index
)


for group_name in user_preferences:

    score_column = (
        f"{group_name}_score"
    )

    weight = (
        user_preferences[
            group_name
        ]
    )

    valid_mask = (
        preference_scores[
            score_column
        ].notna()
    )

    weighted_total.loc[
        valid_mask
    ] += (
        preference_scores.loc[
            valid_mask,
            score_column
        ]
        * weight
    )

    active_total.loc[
        valid_mask
    ] += weight


preference_scores[
    "personalized_preference_score"
] = (
    weighted_total
    /
    active_total
)


# ---------------------------------------------------------
# Validate the final preference scores.
# ---------------------------------------------------------

final_scores = (
    preference_scores[
        "personalized_preference_score"
    ]
)


if final_scores.isna().any():

    raise ValueError(
        "Some destinations have no active preference "
        "information."
    )


if (
    final_scores.min() < 0
    or
    final_scores.max() > 1
):

    raise ValueError(
        "Personalized preference scores are outside [0, 1]."
    )


# ---------------------------------------------------------
# Display validation results.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("NORMALIZED PREFERENCE SCORE VALIDATION")
print("=" * 60)

print(
    "\nPreference groups:",
    len(preference_groups)
)

print(
    "Preference features:",
    len(all_preference_features)
)

print(
    "Destinations:",
    len(preference_scores)
)

print(
    "\nGroup score ranges:"
)

for column in group_score_columns:

    values = (
        preference_scores[
            column
        ]
        .dropna()
    )

    print(
        f"  {column:40s}"
        f"{values.min():.4f}"
        f" → "
        f"{values.max():.4f}"
    )


print(
    "\nPersonalized preference score range:"
)

print(
    f"{final_scores.min():.6f}"
    f" → "
    f"{final_scores.max():.6f}"
)


# ---------------------------------------------------------
# Display the top destinations.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("TOP DESTINATIONS BY CORRECTED PREFERENCE SCORE")
print("=" * 60)

print(
    preference_scores[
        [
            "destination",
            "personalized_preference_score"
        ]
    ]
    .sort_values(
        "personalized_preference_score",
        ascending=False
    )
    .head(10)
    .to_string(
        index=False
    )
)


print("\n" + "=" * 60)
print("✓ NORMALIZED PREFERENCE SCORING PASSED")
print("=" * 60)

REBUILDING NORMALIZED PREFERENCE SCORES

Normalized feature range:
Minimum: 0.0
Maximum: 1.0

NORMALIZED PREFERENCE SCORE VALIDATION

Preference groups: 5
Preference features: 29
Destinations: 50

Group score ranges:
  budget_score                            0.3694 → 1.0000
  flight_score                            0.0278 → 0.8727
  accommodation_score                     0.2575 → 0.8624
  weather_score                           0.3305 → 0.8596
  destination_characteristics_score       0.0019 → 0.5116

Personalized preference score range:
0.224681 → 0.751593

TOP DESTINATIONS BY CORRECTED PREFERENCE SCORE
destination  personalized_preference_score
     Mumbai                       0.751593
  Bengaluru                       0.713373
      Delhi                       0.693414
    Chennai                       0.692993
       Pune                       0.672854
    Kolkata                       0.660363
  Hyderabad                       0.653674
     Jaipur                       0.641897


In [61]:
# ---------------------------------------------------------
# FINAL PERSONALIZED RECOMMENDATION PREDICTION
#
# The preference score has already been validated and is
# guaranteed to be in the [0, 1] range.
#
# We now combine:
#
#   1. Personalized preference score
#   2. Personalized interest score
#   3. Destination similarity
#
# using the saved hybrid configuration.
# ---------------------------------------------------------

# ---------------------------------------------------------
# Load the saved destination similarity model.
# ---------------------------------------------------------

import os
import pickle
import numpy as np
import pandas as pd


MODEL_DIR = "../models"


with open(
    os.path.join(
        MODEL_DIR,
        "destination_similarity.pkl"
    ),
    "rb"
) as file:

    destination_similarity = pickle.load(
        file
    )


with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_destinations.pkl"
    ),
    "rb"
) as file:

    recommendation_destinations = pickle.load(
        file
    )


with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_config.pkl"
    ),
    "rb"
) as file:

    recommendation_config = pickle.load(
        file
    )


# ---------------------------------------------------------
# Validate that the already-computed preference scores are
# available.
# ---------------------------------------------------------

required_column = (
    "personalized_preference_score"
)


if required_column not in preference_scores.columns:

    raise ValueError(
        "Corrected preference scores are not available. "
        "Run the previous scoring cell first."
    )


# ---------------------------------------------------------
# Calculate the personalized interest score again from the
# validated destination profile scores.
# ---------------------------------------------------------

user_interests = {

    "nature": 1.00,

    "sightseeing": 0.50,

    "water_coastal": 0.20,

    "wildlife": 0.60
}


interest_df = (
    destination_profile_scores
    .copy()
)


total_interest_weight = sum(
    user_interests.values()
)


interest_score = pd.Series(
    0.0,
    index=interest_df.index
)


for profile, weight in (
    user_interests.items()
):

    score_column = (
        f"{profile}_score"
    )

    interest_score += (
        interest_df[
            score_column
        ].fillna(0)
        * weight
    )


interest_score = (
    interest_score
    / total_interest_weight
)


interest_result = pd.DataFrame({

    "destination":
        interest_df[
            "destination"
        ],

    "personalized_interest_score":
        interest_score
})


# ---------------------------------------------------------
# Merge preference and interest scores.
# ---------------------------------------------------------

prediction_df = preference_scores[
    [
        "destination",
        "personalized_preference_score"
    ]
].merge(

    interest_result,

    on="destination",

    how="inner"
)


# ---------------------------------------------------------
# Calculate the base personalized score.
#
# Saved configuration:
#
#   Travel factors = 60%
#   Interests      = 40%
# ---------------------------------------------------------

travel_factor_weight = (
    recommendation_config[
        "travel_factor_weight"
    ]
)


interest_weight = (
    recommendation_config[
        "interest_weight"
    ]
)


prediction_df[
    "base_personalized_score"
] = (

    prediction_df[
        "personalized_preference_score"
    ]
    * travel_factor_weight

    +

    prediction_df[
        "personalized_interest_score"
    ]
    * interest_weight
)


# ---------------------------------------------------------
# Build a destination-index lookup.
# ---------------------------------------------------------

destination_index = {

    destination: index

    for index, destination
    in enumerate(
        recommendation_destinations
    )
}


similarity_array = np.asarray(
    destination_similarity
)


# ---------------------------------------------------------
# Use the strongest personalized destinations as the
# similarity neighbourhood.
#
# This provides similarity support without requiring the
# user to specify a reference destination.
# ---------------------------------------------------------

top_personalized = (
    prediction_df
    .sort_values(
        "base_personalized_score",
        ascending=False
    )
    .head(5)
)


top_destinations = (
    top_personalized[
        "destination"
    ].tolist()
)


top_weights = (
    top_personalized[
        "base_personalized_score"
    ].values
)


top_indices = [

    destination_index[
        destination
    ]

    for destination
    in top_destinations
]


# ---------------------------------------------------------
# Calculate similarity support for every destination.
# ---------------------------------------------------------

similarity_scores = []


for destination in (
    prediction_df[
        "destination"
    ]
):

    destination_idx = (
        destination_index[
            destination
        ]
    )

    similarities = (
        similarity_array[
            destination_idx,
            top_indices
        ]
    )

    if top_weights.sum() > 0:

        score = np.average(
            similarities,
            weights=top_weights
        )

    else:

        score = 0.0

    similarity_scores.append(
        score
    )


prediction_df[
    "similarity_score"
] = similarity_scores


# ---------------------------------------------------------
# Normalize the similarity component to [0, 1].
#
# Cosine similarity can naturally contain negative values,
# therefore it should not be directly mixed with a
# [0, 1] preference score.
# ---------------------------------------------------------

similarity_min = (
    prediction_df[
        "similarity_score"
    ].min()
)


similarity_max = (
    prediction_df[
        "similarity_score"
    ].max()
)


if similarity_max > similarity_min:

    prediction_df[
        "normalized_similarity_score"
    ] = (

        prediction_df[
            "similarity_score"
        ]
        - similarity_min
    ) / (
        similarity_max
        - similarity_min
    )

else:

    prediction_df[
        "normalized_similarity_score"
    ] = 0.0


# ---------------------------------------------------------
# Apply the saved hybrid weights.
# ---------------------------------------------------------

hybrid_preference_weight = (
    recommendation_config[
        "hybrid_preference_weight"
    ]
)


hybrid_similarity_weight = (
    recommendation_config[
        "hybrid_similarity_weight"
    ]
)


prediction_df[
    "final_recommendation_score"
] = (

    prediction_df[
        "base_personalized_score"
    ]
    * hybrid_preference_weight

    +

    prediction_df[
        "normalized_similarity_score"
    ]
    * hybrid_similarity_weight
)


# ---------------------------------------------------------
# Sort by final recommendation score.
# ---------------------------------------------------------

prediction_df = (
    prediction_df
    .sort_values(
        "final_recommendation_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Add recommendation rank.
# ---------------------------------------------------------

prediction_df.insert(
    0,
    "rank",
    range(
        1,
        len(prediction_df) + 1
    )
)


# ---------------------------------------------------------
# Display the final prediction.
# ---------------------------------------------------------

print("=" * 60)
print("FINAL PERSONALIZED RECOMMENDATIONS")
print("=" * 60)

print(
    "\nUser preference profile:"
)

for key, value in user_preferences.items():

    print(
        f"  {key:30s} {value:.2f}"
    )


print(
    "\nUser interest profile:"
)

for key, value in user_interests.items():

    print(
        f"  {key:30s} {value:.2f}"
    )


print("\n" + "=" * 60)
print("TOP 10 RECOMMENDATIONS")
print("=" * 60)


display_columns = [

    "rank",

    "destination",

    "personalized_preference_score",

    "personalized_interest_score",

    "base_personalized_score",

    "normalized_similarity_score",

    "final_recommendation_score"
]


print(
    prediction_df[
        display_columns
    ]
    .head(10)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Validate final predictions.
# ---------------------------------------------------------

if prediction_df[
    "final_recommendation_score"
].isna().any():

    raise ValueError(
        "Final recommendation contains missing scores."
    )


if (
    prediction_df[
        "final_recommendation_score"
    ].min() < 0
):

    raise ValueError(
        "Final recommendation score is negative."
    )


if (
    prediction_df[
        "final_recommendation_score"
    ].max() > 1
):

    raise ValueError(
        "Final recommendation score exceeds 1."
    )


print("\n" + "=" * 60)
print("✓ FINAL PERSONALIZED PREDICTION COMPLETED")
print("=" * 60)

print(
    "\n#1 Recommended destination:",
    prediction_df.iloc[0][
        "destination"
    ]
)

print(
    "Final recommendation score:",
    round(
        prediction_df.iloc[0][
            "final_recommendation_score"
        ],
        4
    )
)

FINAL PERSONALIZED RECOMMENDATIONS

User preference profile:
  budget                         0.90
  flight                         0.60
  accommodation                  0.60
  weather                        0.80
  destination_characteristics    0.80

User interest profile:
  nature                         1.00
  sightseeing                    0.50
  water_coastal                  0.20
  wildlife                       0.60

TOP 10 RECOMMENDATIONS
 rank destination  personalized_preference_score  personalized_interest_score  base_personalized_score  normalized_similarity_score  final_recommendation_score
    1      Mumbai                       0.751593                     0.522352                 0.659896                     0.960270                    0.750009
    2   Bengaluru                       0.713373                     0.401584                 0.588658                     1.000000                    0.712060
    3       Delhi                       0.693414                     